<div align="center">

# Trabajo de Fin de Grado
#### Grado en Ingeniería Informática — Mención en Computación y Sistemas Inteligentes (CSI)

---

**Autor:** Rafael Carrillo Arroyo  
**Email:** [rafacarrillo@correo.ugr.es](mailto:rafacarrillo@correo.ugr.es)  
**Institución:** Universidad de Granada (UGR)  
**Módulo:** Fase 1 - Codificador Visual Clínico (Extractor Espacial) | Notebook I de II

---

</div>

### *Resumen del Módulo*
Este *notebook* constituye el núcleo predictivo y de interpretabilidad visual del proyecto. Se centra expresamente en la **Fase 1**, cuyo objetivo es la construcción de unos cimientos visuales universales y robustos que codifiquen la anatomía radiológica preservando estrictamente la topología espacial. Abarca desde la ingesta de datos del *dataset* **CheXpert** y su purificación mediante *pipelines* de MONAI, hasta la extracción de características visuales y la optimización de umbrales diagnósticos (**Índice de Youden**).

Para maximizar la transferencia de conocimiento médico, el bloque extractor utiliza una arquitectura **DenseNet121** bajo la librería especializada **`torchxrayvision`**. Se implementa una técnica avanzada de **cirugía de pesos (*Weight Surgery*)** para adaptar el modelo a entradas monocanal y heredar de forma directa los tensores preentrenados del dominio radiológico. Como estrategia clave para la preservación topológica, se extirpa deliberadamente la capa de *Global Average Pooling*; en lugar de colapsar la radiografía en un vector unidimensional ciego, el modelo extrae una **cuadrícula espacial de $10 \times 10$** (100 *tokens* visuales continuos de 1024 canales) capaz de aislar regiones anatómicas concretas.

Asimismo, este módulo integra el algoritmo **Grad-CAM** como herramienta crítica de explicabilidad médica, permitiendo proyectar mapas de activación sobre las radiografías para auditar el modelo empíricamente, certificando la ausencia de atajos predictivos (*Shortcut Learning*) y demostrando una focalización anatómica exacta sobre la patología real.

> **Nota:** El puente multimodal mediante Proyector MLP (Fase 2) y la sintonización del modelo base BioGPT mediante LoRA para la redacción de informes diagnósticos (Fase 3), se encuentran desarrollados en el **Notebook II: Ciclo Completo de Alineamiento Geométrico-Generativo**.

### *Especificaciones Técnicas*

| Componente | Detalle |
| :--- | :--- |
| **Arquitectura Base** | `DenseNet121` (Pesos fundacionales del ecosistema `torchxrayvision`). **Se extirpa el *Global Average Pooling*** para la extracción de una matriz de 100 *tokens* visuales (1024 canales). |
| **Técnica de Transferencia** | **Cirugía de Pesos (*Weight Surgery*)** con adaptación a entrada monocanal y trasplante directo de gradientes inter-patologías, sumado a la inicialización inteligente de *Bias*. |
| **Framework Base** | **PyTorch** + **MONAI** (Medical Open Network for AI) para el *pipeline* de datos. |
| **Optimización y Estabilidad** | Ejecución bajo **Precisión Mixta Automática (AMP)** y acumulación de gradientes (paso de 4) para máxima eficiencia de VRAM. Aprendizaje multietiqueta en paralelo (14 patologías) combatiendo el desbalanceo extremo (*Long-Tail*) mediante pérdida asimétrica ponderada (**BCE + Pos-Weight**) y mapeo de incertidumbre clínica (**Soft Targets**). |
| **Explicabilidad** | Mapas de calor **Grad-CAM** extraídos de la última capa convolucional para la auditoría y certificación empírica de atención en regiones anómalas. |
| **Validación y Veredicto** | Análisis de curvas ROC-AUC con orquestación de *Early Stopping* (alcanzando un **Macro-AUC de Validación de 0.7517** y un **Test Inédito de 0.7414**). Calibración de sensibilidad operativa mediante el **Índice de Youden** y evaluación frente al *Dataset Shift* usando **PR AUC**. |
---

# *Bloque I : Marco conceptual y definicion del problema*


## 1.1. Contexto Clínico



El diagnóstico por imagen de tórax constituye la piedra angular de la radiología convencional. No obstante, la creciente demanda asistencial y la fatiga cognitiva del especialista pueden comprometer la detección de hallazgos críticos. Patologías con patrones visuales heterogéneos, tales como la **Neumonía** o el **Edema Pulmonar**, exigen una precisión técnica que puede verse afectada por la variabilidad inter-observador.

> **Propuesta del Proyecto:** Desarrollo de un **Sistema de Soporte a la Decisión Clínica (CDSS)**. El objetivo es actuar como un filtro de triaje inteligente que priorice casos anómalos, optimizando los tiempos de respuesta y minimizando la tasa de falsos negativos en el flujo de trabajo hospitalario.


##  1.2. Objetivos de la Fase de Clasificación

**1. Detección Multietiqueta y Blindaje Estadístico**
> Análisis automatizado de 14 patologías torácicas en paralelo mediante una arquitectura `DenseNet121` adaptada a entrada monocanal mediante *Weight Surgery*. Se combate el desbalanceo extremo (*Long-Tail*) inherente a los datos clínicos empleando pérdida asimétrica ponderada (*BCE + Pos-Weight*) y mapeo de incertidumbre clínica (*Soft Targets*).

**2. Preservación Topológica Espacial**
> Reestructuración de la red base extirpando deliberadamente la capa de *Global Average Pooling*. En lugar de colapsar la radiografía en un vector unidimensional ciego, el modelo extrae una cuadrícula espacial de $10 \times 10$ (100 tokens visuales continuos de 1024 canales) capaz de aislar regiones anatómicas concretas.

**3. Optimización Clínica y Auditoría Visual**
> Calibración de umbrales para maximizar la sensibilidad clínica frente a la exactitud puramente estadística, logrando un Macro-AUC Global de 0.741 (alcanzando un AUC $> 0.91$ en anomalías macroscópicas como Derrame Pleural y Edema). Además, la auditoría visual mediante Grad-CAM certifica empíricamente la ausencia de atajos predictivos (*Shortcut Learning*), demostrando una focalización anatómica exacta sobre la patología real.

**4. Fundamento Multimodal (Nexo Generativo)**
> Construcción de unos cimientos visuales universales y robustos que generan el vocabulario base para el modelo generativo. Este espacio latente, matemáticamente estabilizado y con alta fidelidad estructural, está listo para ser inyectado en el Proyector MLP (Fase 2), garantizando que el modelo de lenguaje reciba un mapa semántico libre de ruido instrumental.


## 1.3. Origen de Datos: CheXpert-Small

**1. Geometría de Entrada**
> Las imágenes son ingeridas, redimensionadas y empaquetadas de forma matricial homogénea a una resolución estandarizada de **320x320 píxeles**, garantizando la consistencia espacial que requiere el extractor visual.

**2. Complejidad Multietiqueta**
> El sistema analiza un espectro diagnóstico de **14 clases simultáneas**. Este espacio abarca 13 patologías torácicas específicas junto con la categoría diagnóstica de exclusión *No Finding* (ausencia de hallazgos).

**3. Gestión de la Incertidumbre Clínica**
> La arquitectura del sistema gestiona la ambigüedad inherente a los informes radiológicos originales mediante el mapeo de etiquetas de incertidumbre (*Soft Targets*). Esto asegura un entrenamiento robusto, combatiendo el ruido en las anotaciones y alineando la optimización matemática con la realidad probabilística del diagnóstico médico.

# *Bloque II : Desarrollo experimental y metodología*

## <span style="color:#003366;">2.1. Configuración del Entorno y Gestión de Recursos</span>



### <font color='darkblue'>2.1.1. Instalación de Dependencias y Carga de Librerías</span>
En esta fase se preparan las herramientas de software necesarias. Se prioriza el uso de **MONAI** para el tratamiento de tensores médicos.

In [ ]:
# ==============================================================================
# MÓDULO 1: ARQUITECTURA DE DATOS Y VISIÓN COMPUTACIONAL (DENSENET121)
# ==============================================================================
import os
import sys
import subprocess
import random
import warnings

# 1. DETECCIÓN DINÁMICA DE ENTORNO
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🤖 Entorno Colab detectado. Instalando dependencias de visión médica...")
    # Instalación silenciosa vía subprocess para mantener el log limpio
    dependencies = ["monai", "opencv-python", "torchxrayvision"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + dependencies)

# 2. IMPORTACIONES DE CORE CIENTÍFICO (Consolidadas)
import cv2
import pandas as pd
import numpy as np
from PIL import Image as PILImage
from tqdm.auto import tqdm
from IPython.display import Image as IPyImage, display, Markdown

# 3. DEEP LEARNING (PyTorch y ecosistema médico)
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchxrayvision as xrv

# 4. COMPONENTES MONAI
from monai.networks.nets import DenseNet121
import monai.transforms as mt
from monai.data import Dataset, DataLoader as MonaiDataLoader
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstD, ResizeD,
    ScaleIntensityD, CastToTyped, BorderPadD
)

# 5. EVALUACIÓN Y MÉTRICAS (Bioestadística)
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc, classification_report,
    confusion_matrix, precision_recall_curve, average_precision_score,
    accuracy_score, recall_score, precision_score, f1_score
)

# 6. VISUALIZACIÓN GRÁFICA Y ESTÉTICA (Estándar TFG)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from PIL import Image


# 7. REPRODUCIBILIDAD Y CONSISTENCIA
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Configuración visual global
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
warnings.filterwarnings('ignore')

# Configuración de dispositivo (GPU indispensable para DenseNet121)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    display(Markdown(f"### <font color='green'> ✅ ACELERADOR HARDWARE DETECTADO: {gpu_name}</font>"))
else:
    display(Markdown("### <font color='red'> ⚠️ ADVERTENCIA: PIPELINE EJECUTÁNDOSE EN CPU</font>"))
    display(Markdown("El entrenamiento de la DenseNet121 requiere aceleración por hardware. Selecciona **T4 GPU**."))

### <font color='darkblue'>2.1.2. Sincronización con Google Drive y Descompresión Local</span>
Para maximizar el ancho de banda de lectura (I/O) durante el entrenamiento, se transfieren los datasets desde el almacenamiento en la nube hacia el disco sólido local de la instancia de computación.

In [ ]:
# ==============================================================================
# MÓDULO 1: CONFIGURACIÓN GLOBAL (TFG) - ESTRUCTURA JERÁRQUICA POR MÓDULOS
# ==============================================================================
import os
import sys
import zipfile

# 1. DETECCIÓN DINÁMICA DEL ENTORNO Y DECLARACIÓN DE VARIABLES BASE
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🤖 Entorno detectado: Google Colab. Vinculando almacenamiento persistente...")
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    BASE_PERSISTENTE = '/content/drive/MyDrive/TFG_Rafa'
    BASE_LOCAL = '/content/chexpert_local'
else:
    print("💻 Entorno detectado: Local / Servidor Git. Configurando rutas relativas...")
    BASE_ROOT = os.getcwd()
    BASE_PERSISTENTE = os.path.join(BASE_ROOT, 'outputs')
    BASE_LOCAL = os.path.join(BASE_ROOT, 'data', 'chexpert_local')

# 2. DEFINICIÓN DEL DICCIONARIO CENTRAL (Única Fuente de Verdad Jerárquica)
# Definimos las raíces globales para mantener el repositorio limpio
DIR_EDA_GLOBAL = os.path.join(BASE_PERSISTENTE, 'Resultados_EDA')
DIR_EVAL_GLOBAL = os.path.join(BASE_PERSISTENTE, 'Resultados_Evaluacion')
DIR_CKPT_GLOBAL = os.path.join(BASE_PERSISTENTE, 'checkpoints')

CONFIG = {
    "dir": {
        # Subcarpetas exclusivas del MÓDULO 1
        "checkpoints": os.path.join(DIR_CKPT_GLOBAL, 'Modulo_1'),
        "eda":         os.path.join(DIR_EDA_GLOBAL, 'Modulo_1'),

        # Carpetas de Evaluación
        "evaluacion":  os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1'), # Raíz de evaluación
        "eval_valid":  os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1', 'valid'),
        "eval_test":   os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1', 'test'),

        # GradCAM anidado dentro de test
        "gradcam":     os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1', 'test', 'GradCAM'),

        # Carpetas compartidas (Transversales a todo el TFG)
        "assets":      os.path.join(BASE_PERSISTENTE, 'assets'),
        "data_drive":  os.path.join(BASE_PERSISTENTE, 'data'),
        "data_local":  BASE_LOCAL
    },
    "files": {
        # Archivos del Dataset (Compartidos)
        "zip_chexpert": os.path.join(BASE_PERSISTENTE, 'data', 'chexpert_small.zip'),
        "train_csv":    os.path.join(BASE_LOCAL, 'train.csv'),
        "valid_csv":    os.path.join(BASE_LOCAL, 'valid.csv'),

        # Artefactos generados exclusivos del MÓDULO 1
        "best_model":   os.path.join(DIR_CKPT_GLOBAL, 'Modulo_1', 'MejorModelo_M1_14.pth'),
        "tabla_viabilidad_csv": os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1', 'analisis_viabilidad_patologias.csv'),
        "tabla_viabilidad_tex": os.path.join(DIR_EVAL_GLOBAL, 'Modulo_1', 'analisis_viabilidad_patologias.tex')
    }
}

# 3. Creación automatizada de directorios
for nombre, ruta in CONFIG["dir"].items():
    if "local" not in nombre and not os.path.exists(ruta):
        os.makedirs(ruta, exist_ok=True)
        print(f" 📁 Creado directorio persistente: {ruta}")

# 4. Descompresión condicional inteligente
if not os.path.exists(CONFIG["dir"]["data_local"]):
    if os.path.exists(CONFIG["files"]["zip_chexpert"]):
        print("\n 📦 Extrayendo dataset al entorno local para optimizar I/O...")
        with zipfile.ZipFile(CONFIG["files"]["zip_chexpert"], 'r') as zip_ref:
            zip_ref.extractall(CONFIG["dir"]["data_local"])
        print(" ✅ Dataset listo para entrenamiento de alta velocidad.")
    else:
        print(f" ⚠️ Aviso: No se encontró el dataset en {CONFIG['files']['zip_chexpert']}")
else:
    print("\n ✅ Dataset ya presente en el SSD local.")

Ahora prepararemos las dependencias necesarias para el proyecto:
1. **Montaje de Drive**: Se establece conexión con Google Drive para acceder a los scripts de utilidades personalizados (`Colab_Utils`).
2. **Dependencias**: Se instala la API de Google GenAI para permitir la interacción con modelos de lenguaje.
3. **Módulo de Monitorización**: Se importa el decorador `supervisar_ia` desde el repositorio local, el cual permite enviar notificaciones de estado y progreso de los procesos de entrenamiento directamente a Telegram.

In [ ]:
from google.colab import drive
import sys
import os

# Conectar Google Drive
if not os.path.exists('/content/drive'):
    drive.mount("/content/drive")
sys.path.append("/content/drive/MyDrive/Colab_Utils")

# Instalar la librería de Gemini e importar el decorador
!pip install google-genai -q
from monitor_ia import supervisar_ia

## <span style="color:#003366;">2.2. Auditoría Científica de Datos</span>


En primer lugar, la selección de la variante optimizada `chexpert_small` como piedra angular de este sistema no constituye una concesión empírica, sino una decisión de ingeniería de datos fundamentada en la viabilidad computacional y la agilidad de iteración requeridas para un Trabajo de Fin de Grado. Mientras que el repositorio íntegro del *Stanford Machine Learning Group* comprende cientos de gigabytes con estudios radiológicos en resoluciones masivas, su ingesta directa introduce un cuello de botella de latencia de entrada/salida (I/O) inasumible para entornos de desarrollo en la nube con restricciones de memoria de video (VRAM) y tiempos de ejecución efímeros.

El uso de la distribución `small` estandariza la dimensionalidad de los tensores visuales desde el inicio del *pipeline*, mitigando drásticamente el consumo de recursos de la GPU sin sacrificar la densidad clínica de los biomarcadores patológicos. Como avalan las métricas de rendimiento obtenidas durante la auditoría del Módulo I, esta compresión espacial conserva la granularidad exacta que el codificador convolucional necesita para delinear las fronteras anatómicas de las anomalías torácicas críticas.

Para el desarrollo del codificador visual en este Trabajo de Fin de Grado (TFG), se ha determinado de manera estratégica el **uso exclusivo de radiografías de tórax en proyecciones estrictamente frontales** (Anteroposterior [AP] y Posteroanterior [PA]), aplicando un criterio de exclusión geométrico completo sobre las vistas sagitales o laterales.

Esta decisión de diseño metodológico se fundamenta en tres pilares críticos de la ingeniería de datos médicos y el aprendizaje profundo multietiqueta:

1. **Invarianza Espacial y Consistencia en el Puente Multimodal (Nexo Generativo):**
   La fusión de características visuales y textuales requiere que la matriz latente extraída por la red convolucional mantenga una coherencia anatómica constante con los reportes clínicos. Las vistas frontales concentran la mayor densidad de descriptores diagnósticos compartidos con la literatura médica estándar. Introducir proyecciones laterales —las cuales representan un universo visual y geométrico completamente distinto— inyectaría una variabilidad posicional masiva. Esto forzaría a la red a lidiar con una invarianza traslacional innecesaria, penalizando drásticamente la eficiencia del **alineamiento geométrico-generativo en el Proyector MLP acoplado a embeddings posicionales 2D (Fase 2)**, y desestabilizando la posterior **sintonización autorregresiva del razonador clínico BioGPT mediante LoRA (Fase 3)**.

2. **Optimización del Aprendizaje por Transferencia (*Transfer Learning* con *Weight Surgery*):**
   La arquitectura de extracción visual (*DenseNet121* adaptada a entrada monocanal) se nutre de inicializaciones de pesos optimizadas en el dominio de la radiología digital. Estos modelos base exhiben sus mayores cotas de especificidad y sensibilidad en el plano frontal debido a la constancia en la disposición de las estructuras (silueta cardíaca central, campos pulmonares basales y apicales). Mantener el pipeline alineado con esta distribución nativa previene la degradación de los gradientes y asegura que la **cuadrícula espacial de $10 \times 10$ (100 tokens visuales de 1024 canales)** extraída contenga la máxima resolución semántica y geométrica posible, vital para aislar regiones anatómicas concretas.

3. **Mitigación del Sesgo de Distribución (*Dataset Shift*) y Auditoría Visual:**
   Como el conjunto de prueba de consenso "patrón oro" provisto por CheXpert (`valid.csv`) opera sobre evaluaciones diagnósticas del plano frontal, purificar el conjunto de entrenamiento bajo la misma geometría anula de raíz el sesgo de distribución entre subconjuntos. Esto garantiza que las métricas finales de rendimiento (como el Macro-AUC global) reflejen la verdadera capacidad de generalización del sistema y que las pruebas de interpretabilidad (la **auditoría visual mediante Grad-CAM**) mapeen con fidelidad biomarcadores reales, certificando empíricamente la ausencia de atajos predictivos (*Shortcut Learning*).

Con el fin de cuantificar y demostrar el impacto real de este criterio de exclusión geométrico sobre el volumen de datos y la distribución del ruido semántico, la auditoría de integridad de este proyecto se estructura en dos fases analíticas complementarias:

* **Perspectiva Global de Partida (Muestra Base Colectiva):** Un barrido exploratorio inicial sobre el ecosistema nativo del dataset ($n = 223,414$) que expone las métricas brutas de prevalencia e incertidumbre resultantes del etiquetado automatizado original antes de la depuración.
* **Perspectiva Purificada en Exclusiva (Plano Estrictamente Frontal):** Un desglose analítico dirigido tras la aplicación del filtro geométrico ($n = 191,027$), el cual constituye la verdadera "fuente de verdad" cuantitativa sobre la que operará el optimizador adaptativo del codificador visual para clasificar con éxito las 14 categorías clínicas.



Cuantificación del balance entre estudios Frontales (PA/AP) y Laterales. Dada la diferente perspectiva anatómica, este balance influye en la capacidad del modelo para localizar hallazgos en el espacio tridimensional del tórax.

In [ ]:
# ==============================================================================
# ENCODER VISUAL: JUSTIFICACIÓN DEL DESCARTE GEOMÉTRICO (COHORTE FRONTAL VS LATERAL)
# ==============================================================================

plt.style.use('default')
plt.figure(figsize=(7, 7), dpi=120)

# 1. Leemos el CSV original de configuración (bruto sin filtrar) para recuperar el histórico
df_raw_original = pd.read_csv(CONFIG["files"]["train_csv"])
view_counts_original = df_raw_original['Frontal/Lateral'].value_counts()

# Mapeamos los nombres a etiquetas académicas más elegantes para la gráfica
labels_elegantes = [f"{idx} (AP/PA)" if idx == 'Frontal' else f"{idx} (LAT)" for idx in view_counts_original.index]

plt.pie(
    view_counts_original,
    labels=labels_elegantes,
    autopct='%1.1f%%',
    colors=['#1f4e79', '#8faadc'],  # Azul corporativo profundo vs Azul clínico atenuado
    startangle=140,
    pctdistance=0.75,              # Desplaza los porcentajes al centro de cada sector
    textprops={'fontweight': 'bold', 'fontsize': 11, 'color': 'black'},
    wedgeprops={'edgecolor': 'white', 'linewidth': 3, 'width': 0.4} # 'width' transforma la tarta en un gráfico de anillo limpio
)

plt.title("Composición del Corpus Original de CheXpert\ny Criterio de Exclusión Proyectiva", fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD - CONFIG)
# ----------------------------------------------------------------------
save_path_tarta = os.path.join(CONFIG["dir"]["eda"], "proporcion_vistas_radiograficas.png")

plt.savefig(save_path_tarta, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Gráfico de exclusión geométrica (Donut) exportado con éxito a Drive: {save_path_tarta}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

El análisis de la composición de las proyecciones radiográficas en el dataset bruto de partida revela que un **14.5%** del corpus original (32.387 imágenes) corresponde a proyecciones de perfil o sagitales (*Lateral*). El **85.5%** restante (191.027 imágenes) se compone de proyecciones anteroposteriores (*AP*) y posteroanteriores (*PA*), englobadas bajo la categoría de plano frontal. Desde una perspectiva de arquitectura bioinformática y optimización de redes neuronales convolucionales, la inclusión simultánea de planos frontales y laterales en un único vector de entrada para el codificador visual (*DenseNet121*) constituye una **inconsistencia de modelado estructural** por las siguientes razones técnicas:

1. **Ruptura de la Invarianza Espacial y Degradación de Características Locales:**
   Las redes convolucionales (*CNN*) se basan en la extracción jerárquica de características espaciales fundamentadas en la vecindad de píxeles (bordes, texturas, gradientes densitométricos). Una proyección lateral altera por completo la topología y disposición anatómica de los órganos: la silueta cardíaca se proyecta sobre la columna vertebral, los campos pulmonares izquierdo y derecho se fusionan ópticamente en una superposición cónica y los senos costofrénicos cambian sus vectores de curvatura. Forzar al optimizador adaptativo a aproximar una función de mapeo común para dos distribuciones espaciales e histogramas de radiancia tan dispares degrada críticamente la convergencia de los pesos en las capas profundas de la red, limitando su capacidad para extraer con precisión la **cuadrícula espacial de $10 \times 10$ (100 tokens visuales continuos)** necesaria para aislar regiones anatómicas y patrones patológicos finos.

2. **Ambigüedad Posicional en el Anclaje Semántico Multimodal:**
   El objetivo del pipeline multimodal es alinear la matriz visual con representaciones de texto libre para transferirlos al razonador clínico autorregresivo (**BioGPT**). La narrativa radiológica estandarizada y los patrones de redacción clínica describen los hallazgos tomando como referencia primaria la placa frontal (ej. *"cardiomegalia"* basada en el índice cardiotorácico frontal o *"infiltrados bilaterales"*). Introducir imágenes laterales sin un mecanismo explícito de condicionado de proyección inyectaría una severa ambigüedad posicional que distorsionaría el mapeo de los **Embeddings Posicionales 2D** y los mecanismos de atención (*Attention Mechanisms*) del decodificador durante su **sintonización mediante LoRA (Fase 3)**, induciendo alucinaciones semánticas e inconsistencias clínico-textuales.

**Decisión de Ingeniería de Datos:**
Por consiguiente, se establece como **filtro de entrada irrevocable la exclusión sistemática del 14.5% de vistas laterales**. Aunque esta acción de depuración reduce el tamaño bruto del dataset de entrenamiento a 191.027 muestras, la purificación del espacio de hipótesis compensa con creces la reducción cuantitativa. Al garantizar que el 100% de los tensores de entrada compartan una geometría espacial homogénea, se optimiza la resolución de los mapas de características latentes, se facilita el proceso de **alineamiento geométrico-generativo en el Proyector MLP (Fase 2)** y se blinda la interpretabilidad y fidelidad de la **auditoría visual empírica mediante Grad-CAM** que certifica la ausencia de atajos predictivos.


### <font color='darkblue'>2.2.1. Marco Analítico y Estructuración de Datos</font>
### 1. Auditoría del Dataset y Selección del Vector Objetivo (Target)

Antes de conectar el codificador visual con el razonador clínico autorregresivo (**BioGPT**) a través del puente multimodal (Proyector MLP), es fundamental definir qué patologías intentará predecir la red y cómo se estructurará su aprendizaje. Para tomar una decisión de diseño basada en datos empíricos, implementamos un motor de auditoría (`CheXpertMasterAnalyzer`).

**Objetivo del análisis:** Escanear las 14 categorías diagnósticas originales del dataset de Stanford para evaluar su viabilidad computacional. Dado el desbalanceo extremo (*Long-Tail*) inherente a los datos radiológicos, el objetivo es maximizar la densidad de información clínica útil y minimizar el "ruido semántico" (incertidumbre y ambigüedad en las etiquetas). Esto es crítico porque el decodificador médico necesitará un volumen robusto de ejemplos positivos, anclados firmemente a la matriz visual (la cuadrícula espacial de $10 \times 10$ extraída por la DenseNet), para aprender a generar informes precisos y evitar alucinaciones durante su sintonización posterior.


In [ ]:
# ==============================================================================
# ENCODER VISUAL: AUDITORÍA COMPARATIVA BASAL (GLOBAL VS FRONTAL) - SOLO VISUAL
# ==============================================================================
class CheXpertMasterAnalyzer:
    """
    Clase integral para la auditoría técnica y clínica del dataset CheXpert.
    Diseñada para justificar la selección de patologías para VLMs mediante
    el contraste de proyecciones volumétricas globales frente a restricciones frontales.
    Muestra los resultados únicamente por pantalla mediante gradientes de color.
    """
    def __init__(self, train_csv, valid_csv, img_root):
        self.train_df = pd.read_csv(train_csv)
        self.valid_df = pd.read_csv(valid_csv)
        self.img_root = img_root

        # Las 14 etiquetas originales clínicas del dataset CheXpert de Stanford
        self.todas_las_patologias = [
            'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
            'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
            'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
        ]

        # Estandarización de rutas para el sistema de archivos local/Colab
        for df in [self.train_df, self.valid_df]:
            df['Path_Local'] = df['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)

    def _procesar_subconjunto(self, df_fuente):
        """Método interno vectorial para el cálculo de métricas de viabilidad."""
        total_muestras = len(df_fuente)
        resultados = []

        for patologia in self.todas_las_patologias:
            columna = df_fuente[patologia]

            # Conteos puros forzando tipos nativos estándar (int64)
            # eq() vectorial es más eficiente que == para dataframes grandes
            float_positivos = int(columna.eq(1.0).sum())
            float_inciertos = int(columna.eq(-1.0).sum())
            no_mencionados = int(columna.isna().sum())

            # Porcentajes clave para la toma de decisión clínica
            # Se añade guardia de división por cero por seguridad
            divisor = total_muestras if total_muestras > 0 else 1
            pct_positivos = (float_positivos / divisor) * 100
            pct_inciertos = (float_inciertos / divisor) * 100
            pct_no_mencionados = (no_mencionados / divisor) * 100

            resultados.append({
                'Patología': patologia,
                'Positivos (Ejemplos VLM)': float_positivos,
                '% Positivos': round(pct_positivos, 1),
                '% Inciertos (-1)': round(pct_inciertos, 1),
                '% No Mencionados (NaN)': round(pct_no_mencionados, 1)
            })

        # Consolidación estructurada ordenada de mayor a menor representatividad lingüística
        df_res = pd.DataFrame(resultados)
        return df_res.sort_values(by='Positivos (Ejemplos VLM)', ascending=False).reset_index(drop=True)

    def analizar_viabilidad_patologias(self):
        """
        Escanea secuencialmente el corpus completo y el corpus frontal purificado,
        desplegando los resultados en pantalla con gradientes de color diferenciados.
        No persiste archivos en disco.
        """
        # --- 1. PROCESAMIENTO DEL CONJUNTO GLOBAL DE PARTIDA ---
        print("⏳ Ejecutando auditoría sobre el ecosistema global (Frontales + Laterales)...")
        df_global = self._procesar_subconjunto(self.train_df)

        # --- 2. FILTRADO GEOMÉTRICO Y PROCESAMIENTO FRONTAL ---
        print("✂️ Aislando proyecciones AP/PA y ejecutando auditoría en el plano Frontal estricto...")
        df_train_frontal = self.train_df[self.train_df['Frontal/Lateral'] == 'Frontal']
        df_frontal = self._procesar_subconjunto(df_train_frontal)

        # ==============================================================================
        # DESPLIEGUE POR PANTALLA (RENDERIZADO INTERACTIVO COLAB)
        # ==============================================================================
        print("\n" + "=" * 100)
        print(" 📊 TABLA 1: ANÁLISIS DE VIABILIDAD GLOBAL (HISTÓRICO BASE)")
        print(f" Total de radiografías analizadas: {len(self.train_df):,}")
        print("=" * 100)
        display(df_global.style.background_gradient(cmap='Purples', subset=['% Positivos']).format({
            'Positivos (Ejemplos VLM)': '{:,}',
            '% Positivos': '{:.1f}%',
            '% Inciertos (-1)': '{:.1f}%',
            '% No Mencionados (NaN)': '{:.1f}%'
        }))

        print("\n" + "=" * 100)
        print(" 🎯 TABLA 2: ANÁLISIS DE VIABILIDAD PURIFICADO (PLANO ESTRICTAMENTE FRONTAL)")
        print(f" Total de radiografías analizadas: {len(df_train_frontal):,}")
        print("=" * 100)
        # Usamos un mapa de color distinto (Blues) para diferenciar el resultado purificado
        display(df_frontal.style.background_gradient(cmap='Blues', subset=['% Positivos']).format({
            'Positivos (Ejemplos VLM)': '{:,}',
            '% Positivos': '{:.1f}%',
            '% Inciertos (-1)': '{:.1f}%',
            '% No Mencionados (NaN)': '{:.1f}%'
        }))

        print("\n✅ Auditoría comparativa finalizada. Visualice y copie los porcentajes necesarios para la memoria.")


# ==============================================================================
# EJECUCIÓN CON CONFIGURACIÓN CENTRALIZADA (ÚNICA FUENTE DE VERDAD)
# ==============================================================================
# Instanciamos el objeto audit apuntando a tus archivos de Drive
audit = CheXpertMasterAnalyzer(
    train_csv=CONFIG["files"]["train_csv"],
    valid_csv=CONFIG["files"]["valid_csv"],
    img_root=CONFIG["dir"]["data_local"]
)

# Lanzamos el pipeline dual (Global vs Frontal)
audit.analizar_viabilidad_patologias()

In [ ]:
# 1. INYECTAMOS LAS 14 PATOLOGÍAS AL OBJETO AUDIT
audit.target_cols = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]

A la luz de las métricas empíricas extraídas por la auditoría sobre el dataset, la arquitectura del codificador visual abandona el enfoque reducido tradicional de 5 patologías ("Big 5") para consolidarse en el espectro completo de **14 clases** estandarizadas por CheXpert, operando estrictamente sobre **proyecciones frontales (AP/PA)**.

Esta decisión metodológica responde a tres principios de diseño fundamentales para la viabilidad de la arquitectura multimodal y la mitigación del sesgo en la generación de lenguaje:

1. **Mitigación del Sesgo hacia la Clase Mayoritaria ("Efecto Cajón de Sastre"):**
   Restringir el entrenamiento a un subconjunto de 5 patologías forzaba al modelo a ignorar hallazgos clínicos masivos. Como revela la auditoría, *Support Devices* (56.1%) y *Lung Opacity* (49.3%) dominan el espacio latente real. Al omitir estas categorías, el clasificador carecía de etiquetas para anomalías evidentes (ej. marcapasos o tubos), empujando estadísticamente a la red a predecir **'No Finding'** por descarte. El blindaje estadístico sobre las 14 clases en paralelo permite al modelo identificar y aislar estos artefactos, "limpiando" los falsos negativos y devolviendo a la clase *'No Finding'* a su prevalencia real.

2. **Enriquecimiento del Espacio Semántico y Geométrico para el VLM:**
   Para que el puente multimodal (**Proyector MLP**) y el posterior razonador clínico autorregresivo (**BioGPT**) no incurran en alucinaciones semánticas, necesitan una topología visual extremadamente rica. Discriminar entre matices patológicos sutiles (diferenciar una *Atelectasia* de una *Consolidación* o una *Infiltración*) exige que el codificador convolucional abandone los vectores colapsados unidimensionales y extraiga una **cuadrícula espacial de $10 \times 10$** (100 tokens visuales continuos de 1024 canales). Este nivel de resolución y "vocabulario visual" profundo es matemáticamente imposible de alcanzar si se entrena la red para discriminar únicamente 5 categorías.

3. **Restricción a Proyecciones Estrictamente Frontales (AP/PA, n=191,027):**
   La exclusión de las proyecciones laterales garantiza la **estabilidad geométrica** de los mapas de características. Al someter a la red únicamente a distribuciones anatómicas constantes (silueta cardíaca central, diafragma basal), se acelera drásticamente la convergencia del modelo y se optimiza el consumo de VRAM, un factor crítico dado el hardware empleado (NVIDIA T4). Además, esta consistencia visual anula la ambigüedad posicional durante el **alineamiento geométrico-generativo (Fase 2)**, garantizando que la inyección de **Embeddings Posicionales 2D** mapee coordenadas anatómicas unívocas, libres de ruido perspectivo.

Finalmente, el mantenimiento del espectro completo permite tratar la clase **No Finding** como un verdadero "estado base" de normalidad. Esto garantiza que el codificador visual construya un espacio latente capaz de discernir matemáticamente entre un tórax sano y las alteraciones morfológicas, sirviendo como señal de control robusta y fidedigna para el generador autorregresivo final durante su sintonización con LoRA.

Luego, se constata una alta prevalencia de valores nulos (`NaN`) y etiquetas de incertidumbre (**-1.0**) en la matriz de anotaciones. Lejos de constituir anomalías de tabulación, esta dispersión refleja la naturaleza inherente de la redacción clínica: el facultativo omite sistemáticamente referenciar las estructuras sanas (generando los valores `NaN`) y emplea un lenguaje probabilístico ante hallazgos ambiguos (induciendo el **-1.0** durante la extracción automatizada por NLP).

Para garantizar la estabilidad matemática del ecosistema multimodal en fases posteriores, es un requisito metodológico ineludible aplicar un tratamiento de etiquetas asimétrico. Los nulos deben ser binarizados de forma determinista al valor **0.0** (asumiendo la ausencia clínica implícita). Por su parte, el ruido semántico de la incertidumbre se aborda mediante un **mapeo de incertidumbre clínico (*Soft Targets*)**.

A nivel de implementación computacional, esta regularización se ejecuta inyectando ruido estocástico mediante una **distribución uniforme continua en el intervalo [0.55 - 0.85]** sobre las etiquetas dudosas durante el entrenamiento (fijándose estáticamente en 0.55 para validación). Esta estrategia geométrica, integrada de forma conjunta con una pérdida asimétrica ponderada (*BCE + Pos-Weight*), actúa como un blindaje estadístico crítico para combatir el desbalanceo extremo (*Long-Tail*), mitigando el aprendizaje de atajos predictivos y flexibilizando la optimización de la red ante características visuales difusas.

### <font color='darkblue'>2.2.2. Integridad de Etiquetas y Cuantificación de la Incertidumbre</font>

El dataset CheXpert introduce una complejidad inherente: las etiquetas de incertidumbre (asignadas como -1.0), que representan hallazgos donde el radiólogo o el etiquetador automático no pudieron confirmar ni descartar la patología. El cálculo del **'Uncertainty Index'** permite determinar el impacto de estas observaciones ambiguas, fundamentando la elección de una estrategia de mapeo específica:

Estrategias de Mapeo de Incertidumbre:

1. **U-Zeros (Mapeo a Hallazgo Negativo):**
   * **En qué consiste:** Todos los valores -1.0 se transforman en 0.
   * **Impacto:** Se adopta una postura conservadora. Si el modelo no tiene la certeza absoluta, prefiere no "inventar" un positivo. Útil para evitar falsos positivos, aunque penaliza severamente la sensibilidad clínica.
    
2. **U-Ones (Mapeo a Hallazgo Positivo):**
   * **En qué consiste:** Todos los valores -1.0 se transforman en 1.
   * **Problema detectado:** Aunque históricamente se usaba para priorizar la sensibilidad, en patologías con un alto Índice de Incertidumbre inyecta un ruido masivo. Genera un error extremo en la función de pérdida que obliga a la red neuronal a buscar atajos predictivos (*Shortcut Learning*) fuera de la anatomía real para forzar la predicción a 1.0, arruinando la explicabilidad.

3. **U-Ignore (Exclusión):**
   * **En qué consiste:** Se ignoran las muestras con -1.0 durante el cálculo del gradiente.
   * **Impacto:** Permite que el modelo aprenda solo de casos "puros", pero supone la eliminación de un volumen inaceptable de datos clínicos valiosos (ej. miles de pacientes con signos iniciales o dudosos).

4. **Mapeo de Incertidumbre Clínico / Soft Targets (Estrategia Aplicada):**
   * **En qué consiste:** Sustituir la etiqueta rígida de incertidumbre (-1.0) por una distribución de probabilidad continua orientada al sesgo clínico positivo. A nivel computacional, durante la fase de entrenamiento, estas etiquetas se transforman dinámicamente inyectando ruido uniforme en el rango **[0.55, 0.85]**. En la fase de validación, se fijan en un valor determinista de **0.55** para garantizar la reproducibilidad y estabilización estructural.
   * **Por qué usarlo:** Representa una solución robusta para la explicabilidad médica. Al integrar esta estrategia con la pérdida asimétrica ponderada (*BCE + Pos-Weight*), la red no sufre una penalización extrema ante imágenes ambiguas, eliminando la necesidad de buscar correlaciones espurias. Paralelamente, superar el umbral probabilístico de 0.5 mantiene viva la señal matemática de sospecha clínica.

> **Justificación Técnica:** Tras el Análisis Exploratorio de Datos (EDA), se descartó la estrategia **U-Ones** debido a que la alta proporción de casos inciertos (especialmente en patologías límite) estaba corrompiendo la función de pérdida y provocando un desalineamiento visual evidente. Para el entrenamiento definitivo de nuestro modelo, implementamos la política de **Soft Targets estocásticos en el intervalo [0.55, 0.85] para el set de entrenamiento, consolidando la estabilidad del sistema mediante un valor de control de 0.55 en validación**. Esta decisión protege la integridad clínica del pipeline: estabiliza los gradientes en la optimización de la *DenseNet121*, ancla la atención de la red estrictamente en las estructuras anatómicas (certificado empíricamente mediante **Grad-CAM**) y garantiza que la **cuadrícula espacial de $10 \times 10$** extraída no dependa de artefactos ni atajos visuales, preservando la topología requerida para el posterior puente multimodal.

La elección del intervalo dinámico estocástico **[0.55, 0.85]** (mapeo de *Soft Targets*) para el conjunto de entrenamiento responde a una necesidad dual de regularización matemática y praxis clínica. Al situar el límite inferior en 0.55, se supera el umbral de máxima entropía de la función sigmoide (0.5), aplicando un sesgo conservador que prioriza la sensibilidad radiológica ante sospechas no confirmadas. Simultáneamente, el techo del 0.85 evita forzar a la red a converger hacia una certeza absoluta (1.0) en muestras intrínsecamente ambiguas. Esto previene gradientes excesivamente penalizadores en la función de **pérdida asimétrica ponderada (*BCE + Pos-Weight*)**, la cual actúa como blindaje estadístico frente al desbalanceo extremo (*Long-Tail*). La inyección estocástica de ruido dentro de este rango actúa como un mecanismo regularizador que impide el sobreajuste a un valor de incertidumbre estático y mitiga el aprendizaje de atajos predictivos.

Por otro lado, la fijación del hiperparámetro en un valor escalar determinista de **0.55** durante la fase de validación es un requisito metodológico ineludible. Elimina la varianza estocástica en el cálculo de la pérdida en validación, permitiendo una monitorización objetiva de la convergencia de la *DenseNet121* y asegurando que las métricas de evaluación (como el Macro-AUC global) sean estrictamente reproducibles. Esta estabilización estructural ancla el rendimiento del modelo a su capacidad de detectar la mínima señal clínica razonable, garantizando que la **cuadrícula espacial de $10 \times 10$** extraída mantenga la alta fidelidad topológica requerida para las siguientes fases del ecosistema multimodal.

In [ ]:
# ==============================================================================
# AUDITORÍA DE INTEGRIDAD: ANÁLISIS DE RUIDO SEMÁNTICO EN PLANO FRONTAL
# ==============================================================================
import pandas as pd

def analizar_incertidumbre(analyzer):
    """
    Escanea la distribución de etiquetas de incertidumbre (-1) sobre las cohortes
    restringidas al plano frontal y calcula el Uncertainty Index formal para
    auditar el impacto del ruido semántico en el espacio de hipótesis.
    """
    stats = []

    # Iteración sobre los dataframes purificados en el constructor de 'audit'
    for name, df in [("Entrenamiento (Frontal)", analyzer.train_df), ("Validación (Frontal)", analyzer.valid_df)]:

        # 1. FILTRO ESTRICTO DE PROYECCIÓN FRONTAL
        # Nos aseguramos de operar solo sobre las filas donde la imagen es frontal
        # (Si tu columna tiene otro nombre exacto como 'AP/PA', cámbialo aquí)
        if 'Frontal/Lateral' in df.columns:
            df_frontal = df[df['Frontal/Lateral'] == 'Frontal']
        else:
            # Si el dataframe ya venía filtrado y no tiene la columna, usamos df tal cual
            df_frontal = df

        for col in analyzer.target_cols:
            # Conteo indexado usando el dataframe ya purificado a frontales
            pos = int(df_frontal[col].eq(1).sum())
            unc = int(df_frontal[col].eq(-1).sum())

            # Ratio formal de incertidumbre: proporción de dudas frente al total expuesto
            uncertainty_index = unc / (pos + unc) if (pos + unc) > 0 else 0.0

            stats.append({
                "Dataset": name,
                "Patología": col,
                "Positivos": pos,
                "Inciertos (-1)": unc,
                "Uncertainty Index": round(uncertainty_index, 3)
            })

    return pd.DataFrame(stats).set_index(["Dataset", "Patología"])

# ==============================================================================
# EJECUCIÓN Y RENDERIZADO EN PANTALLA (ANÁLISIS DE INTEGRIDAD)
# ==============================================================================

# El motor 'audit' ahora sí opera de forma nativa con el sesgo geométrico lateral corregido
df_incertidumbre = analizar_incertidumbre(audit)

# Despliegue estético mediante un mapa de calor degradado (Cmap Blues) enfocado en el índice
display(df_incertidumbre.style.background_gradient(cmap='Blues', subset=['Uncertainty Index']))

El análisis exploratorio de la distribución de etiquetas en el conjunto de entrenamiento, restringido al plano estrictamente frontal, evidencia la severa carga de ambigüedad clínica que caracteriza al dataset de CheXpert. Este fenómeno se refleja en una alta prevalencia de observaciones catalogadas como inciertas (-1.0). El problema es especialmente crítico en patologías como la **Neumonía**, con un *Uncertainty Index* de **0.774** (15.981 casos inciertos frente a 4.675 positivos), y la **Consolidación**, con un índice de **0.653** (24.381 inciertos frente a 12.983 positivos). Esto implica que la inmensa mayoría de la masa crítica expuesta para estas categorías carece de una frontera de decisión nítida en los reportes radiológicos originales, comprometiendo de partida la consistencia del aprendizaje multietiqueta en un entorno de 14 clases en paralelo.

Bajo estas condiciones de asimetría y dispersión extrema (*Long-Tail*), la aplicación de una política determinista clásica como **U-Ones** (el mapeo axiomático de la incertidumbre al valor positivo 1.0) introduce un ruido de etiqueta inadmisible en la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)**. Al forzar al optimizador a tratar meras sospechas diagnósticas como certezas clínicas absolutas, el sistema incurre de forma prematura en atajos predictivos (*Shortcut Learning*). Esto obliga a la arquitectura visual a correlacionar artefactos visuales periféricos con la patología, degradando críticamente la especificidad del modelo y destruyendo la fidelidad topológica espacial, un factor que debe ser empíricamente auditado y prevenido mediante mapas **Grad-CAM**.

Por el contrario, la adopción estratégica de un **mapeo de incertidumbre clínico (*Soft Targets*) en el rango estocástico [0.55, 0.85]** durante el entrenamiento resuelve esta tensión metodológica. Al inyectar una distribución de probabilidad continua orientada al sesgo clínico positivo, se capitaliza de forma segura el valor estadístico de cohortes masivas pero ambiguas (como los 29.863 registros inciertos de Atelectasia o los 10.286 de Enlarged Cardiomediastinum). Esta aproximación matemática opera como un blindaje estadístico sobre el espacio de hipótesis: suaviza la penalización de los gradientes, previene la saturación temprana y ancla con éxito la atención de la red *DenseNet121* sobre los verdaderos biomarcadores morfológicos. Esto garantiza la extracción intacta de la **cuadrícula espacial de $10 \times 10$** (100 tokens visuales), manteniendo la estabilidad estructural y clínica del ecosistema multimodal mediante la fijación en un valor determinista de **0.55** durante la validación.

### <font color='darkblue'>2.2.3. Análisis de Prevalencia Patológica</font>
Evaluación del volumen de casos positivos reales por clase. Este análisis es crítico para identificar desequilibrios en el conjunto de entrenamiento que podrían sesgar el aprendizaje del modelo hacia las patologías con mayor representación (ej. Derrame Pleural).

In [ ]:
# ==============================================================================
# ENCODER VISUAL: RENDERIZADO DE LA PREVALENCIA DE CASOS (COHORTE FRONTAL)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('default')
plt.figure(figsize=(12, 5), dpi=120)

# ----------------------------------------------------------------------
# 1. DECLARACIÓN ESTRICTA DEL DATAFRAME FRONTAL (Filtro previo de seguridad)
# ----------------------------------------------------------------------
if 'Frontal/Lateral' in audit.train_df.columns:
    df_frontal = audit.train_df[audit.train_df['Frontal/Lateral'] == 'Frontal'].copy()
else:
    df_frontal = audit.train_df.copy()

# ----------------------------------------------------------------------
# 2. Cálculo de frecuencias ordenadas consumiendo la cohorte frontal purificada
# ----------------------------------------------------------------------
train_positives = df_frontal[audit.target_cols].eq(1).sum().sort_values(ascending=False)

# Renderizado estético usando Seaborn (Fix: Asignación explícita de hue para evitar FutureWarnings)
sns.barplot(
    x=train_positives.values,
    y=train_positives.index,
    hue=train_positives.index,
    palette="Blues_r",
    legend=False
)

# Estética de alta gama para la maquetación de la memoria del TFG
plt.title("Distribución de Casos Positivos por Patología (Plano Estrictamente Frontal)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Número Absoluto de Muestras Positivas (1)", fontsize=11, fontweight='bold')
plt.ylabel("Clases Objetivo (Target)", fontsize=11, fontweight='bold')
plt.grid(axis='x', linestyle=':', alpha=0.6)

# Inyección de etiquetas numéricas con separadores de miles en la punta de cada barra
for i, v in enumerate(train_positives.values):
    plt.text(v + (max(train_positives.values) * 0.005), i, f" {v:,}", va='center', fontsize=10, fontweight='bold', color='darkblue')

plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD - CONFIG)
# ----------------------------------------------------------------------
save_path_barras = os.path.join(CONFIG["dir"]["eda"], "distribucion_casos_positivos_frontal.png")

# Guardamos a 300 DPI asegurando el empaquetado marginal estricto
plt.savefig(save_path_barras, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Histograma de prevalencia frontal exportado con éxito a Drive: {save_path_barras}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

El perfil de prevalencia indexado revela una **severa asimetría estructural** en la distribución de casos positivos que componen el vector objetivo extendido a 14 clases (`target_cols`). La clase dominante, **Support Devices**, acapara drásticamente el lienzo estadístico con 107.170 muestras positivas, superando en una proporción de más de 42 a 1 el volumen de la clase minoritaria, **Pleural Other**, que cuenta únicamente con 2.505 registros confirmados. Las patologías intermedias pero clínicamente críticas, como **Edema** (49.675), **Atelectasis** (29.720) y **Cardiomegaly** (23.385), evidencian escalones de descenso representativo muy marcados, integrándose en un entorno dominado por hallazgos masivos como **Lung Opacity** (94.211) y flanqueadas por la cohorte de control **No Finding** (16.974).

Desde la perspectiva de la optimización estocástica en aprendizaje profundo, este desbalanceo intrínseco (típico de las distribuciones clínicas de "cola larga" o *long-tail*) introduce un riesgo crítico de sesgo algorítmico y colapso de gradiente. Si se entrenara el codificador visual (*DenseNet121*) bajo una función de pérdida simétrica convencional, el optimizador convergería prematuramente hacia un mínimo local subóptimo. En este escenario, la red maximizaría la métrica de acierto global (*Accuracy*) especializándose en las firmas radiológicas densas o artefactos evidentes de las clases mayoritarias, e ignorando las sutiles opacidades reticulares o focos de ocupación alveolar que delimitan patologías infrarrepresentadas como una **Neumonía** (4.675) o una **Consolidación** (12.983).

**Estrategias Arquitectónicas de Compensación:**
Para neutralizar esta asimetría de las 14 clases en paralelo y evitar el colapso unidimensional de las características, garantizando que el clasificador extraiga una **cuadrícula espacial de $10 \times 10$** (100 tokens visuales continuos de 1024 canales) equitativa y de alta resolución antes de transferirla al Proyector MLP (Fase 2), se justifican y validan dos contramedidas algorítmicas en nuestro pipeline:

1. **Blindaje Estadístico por Ponderación de Pérdida (*BCE + Pos-Weight*):** A nivel matemático, el uso de una entropía cruzada binaria estándar resultaría punitivo para la sensibilidad del modelo en las clases minoritarias. Se introduce un tensor de pesos específicos en una función de **pérdida asimétrica ponderada**, calculado de manera inversa a la prevalencia de cada patología. Este tensor actúa como un factor de escala sobre los gradientes de los falsos negativos, penalizando con severidad desproporcionada los errores de omisión diagnóstica en patologías con bajo soporte muestral, integrándose orgánicamente con el mapeo de incertidumbre (*Soft Targets*) abordado previamente.
2. **Calibración Operativa Post-Entrenamiento (Índice de Youden) y Auditoría Visual:** Dado que el desbalanceo desplaza de forma natural las probabilidades brutas de salida de la función sigmoide hacia valores cercanos a cero en las clases menos representadas, el umbral de decisión estándar de la industria (0.5) queda invalidado. La determinación no paramétrica de un punto de corte óptimo independiente para cada una de las 14 patologías compensa el sesgo de prevalencia directamente en la frontera de inferencia.


### <font color='darkblue'>2.2.4. Estudio de Co-ocurrencia y Comorbilidad Clínica</font>
Se investiga la correlación entre diferentes patologías. Una alta co-ocurrencia entre hallazgos (como Edema y Cardiomegalia) refleja patrones clínicos reales de insuficiencia cardíaca, lo cual debe ser capturado por la naturaleza multietiqueta del modelo.

In [ ]:
# ==============================================================================
# ENCODER VISUAL: MATRIZ DE CO-OCURRENCIA CLÍNICA (COHORTE FRONTAL)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

plt.style.use('default')
plt.figure(figsize=(11, 9), dpi=120)


# Clonamos el subset para no contaminar el dataframe original de la memoria RAM
df_coocurrencia = df_frontal[audit.target_cols].copy()

# ----------------------------------------------------------------------
# CORRECCIÓN DE FRONTERA: Binarización adaptada a las 14 patologías
# ----------------------------------------------------------------------
# Para las patologías, tratamos la incertidumbre (-1) como sospecha positiva (1) según la política base
columnas_enfermedades = [c for c in audit.target_cols if c != 'No Finding']
df_coocurrencia[columnas_enfermedades] = df_coocurrencia[columnas_enfermedades].fillna(0).replace(-1, 1)

# Para 'No Finding', los NaN son estrictamente 0 (ya que implican que el paciente presenta hallazgos)
df_coocurrencia['No Finding'] = df_coocurrencia['No Finding'].fillna(0)

# Cálculo de la matriz de correlación de Pearson pura sobre el plano frontal estricto
correlation_matrix = df_coocurrencia.corr()

# Máscara geométrica para ocultar la matriz espejo superior (Estilo de publicación SOTA)
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

# Renderizado del mapa de calor térmico
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.8,
    cbar=True,
    cbar_kws={"shrink": 0.75},
    annot_kws={"size": 9, "weight": "bold"} # Reducido ligeramente a 9 para albergar las 14 etiquetas cómodamente
)

# Estética y formateo institucional para el documento de la memoria del TFM
plt.title("Matriz de Co-ocurrencia Clínica en el Plano Frontal (Pearson)", fontsize=13, fontweight='bold', pad=20)
plt.xticks(fontweight='bold', rotation=45, ha='right')
plt.yticks(fontweight='bold', rotation=0)

plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD - CONFIG)
# ----------------------------------------------------------------------
save_path_heatmap = os.path.join(CONFIG["dir"]["eda"], "matriz_correlacion_patologias_frontal.png")

plt.savefig(save_path_heatmap, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Mapa de calor de co-ocurrencia estrictamente frontal exportado a Drive: {save_path_heatmap}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()



La matriz de correlación lineal de Pearson (Figura X) revela la estructura de co-ocurrencia de las 14 clases seleccionadas sobre la cohorte estrictamente frontal. Mapear estas dependencias estadísticas de forma previa al entrenamiento es un requisito metodológico crucial, ya que permite anticipar cómo interactuarán las fronteras de decisión dentro del espacio de características latentes de la *DenseNet121* y evitar el aprendizaje de sesgos espurios.

Del análisis térmico del mapa de calor, se extraen cuatro conclusiones clínico-estructurales clave:

* **1. El Eje Consolidación-Atelectasia (Correlación: 0.37):** Representa la asociación positiva más robusta del sistema. Fisiopatológicamente, existe un solapamiento directo: los procesos de ocupación del espacio alveolar (Consolidación) y el colapso pulmonar por pérdida de volumen (Atelectasia) coexisten con frecuencia en síndromes pleuropulmonares complejos. Desde la perspectiva de la visión por computador, este valor alerta sobre un riesgo potencial de **entrelazamiento de características (*Feature Entanglement*)**. Dado que ambas entidades comparten firmas densitométricas visuales similares en la radiografía digital (zonas radiopacas o parénquima "blanco"), se justifica arquitectónicamente la extirpación del *Global Average Pooling* en la *DenseNet121*. Al extraer una **cuadrícula espacial de $10 \times 10$** (100 tokens visuales), el codificador preserva la alta resolución espacial y los descriptores locales finos necesarios para discriminar correctamente las sutiles diferencias de contorno.
* **2. El Carácter Generalista de la Opacidad Pulmonar (*Lung Opacity*):** La clase *Lung Opacity* exhibe correlaciones positivas consistentes con las patologías de afectación del parénquima, destacando su relación con **Pleural Effusion** (0.22), **Pneumonia** (0.21) y **Edema** (0.18). Este comportamiento valida estadísticamente su descarte inicial como variable aislada en modelos reducidos y justifica su tratamiento actual como un "término paraguas" visual. Al mapear las 14 clases en paralelo, el modelo aprende a desagregar la opacidad genérica en entidades clínicas específicas, enriqueciendo la topología visual latente que se transferirá al **Proyector MLP (Fase 2)**.
* **3. La Cascada Fisiológica Cardiomegalia-Edema (0.18):** Refleja una dependencia clínico-causal perfectamente modelada por el dataset. Un remodelado cardíaco adverso (un corazón agrandado) altera la hemodinámica pulmonar, derivando de forma natural en una extravasación de líquido en el intersticio (Edema pulmonar). Esta correlación moderada actúa como un **sesgo inductivo semántico muy valioso para el razonador clínico autorregresivo (BioGPT) en la Fase 3**. El puente multimodal capitalizará esta correlación, permitiendo que la inyección de la matriz visual (junto a los Embeddings Posicionales 2D) aumente la probabilidad condicional de que el decodificador, sintonizado de forma quirúrgica mediante **LoRA**, atienda y redacte tokens asociados a la congestión vascular o soporte ventilatorio.
* **4. Comportamiento de Control de la Normalidad (*No Finding*):** La columna de control **No Finding** valida la integridad matemática del filtrado al registrar correlaciones negativas sólidas frente a todo el espectro patológico (alcanzando el **-0.32** contra *Lung Opacity*, **-0.28** contra *Pleural Effusion* y **-0.21** contra *Atelectasis*). Esto confirma que opera como un ancla de exclusión perfecta para el optimizador, mientras que la práctica ortogonalidad de variables independientes (como *Support Devices* vs. *Pneumonia* [-0.10] o *Fracture* vs. *Edema* [-0.07]) demuestra que las variables aportan información diagnóstica única no redundante.

**Justificación de la Arquitectura Multietiqueta:**
Los datos de co-ocurrencia obtenidos justifican categóricamente el descarte de aproximaciones de clasificación exclusivas. Dado que las patologías presentan dependencias clínico-estadísticas demostrables y no son mutuamente excluyentes (un paciente puede portar simultáneamente dispositivos de soporte, derrame pleural y edema), el bloque visual tiene **estrictamente restringido el uso de la función de activación Softmax**. En su lugar, es metodológicamente obligatorio implementar **activaciones Sigmoide independientes mapeadas a una función de pérdida asimétrica ponderada (*BCE + Pos-Weight*)**. Integrada junto con el mapeo de incertidumbre clínico (*Soft Targets*), esta arquitectura permite al sistema converger de forma simultánea en múltiples frentes patológicos sobre una misma placa de tórax, blindando matemáticamente al codificador frente al desbalanceo extremo.



### <font color='darkblue'>2.2.5. Perfil Demográfico y Sesgo de Datos</font>
Análisis de la distribución de edad y sexo en la cohorte. Este estudio es fundamental para garantizar la equidad algorítmica y verificar que el modelo no aprenda características demográficas en lugar de radiológicas.

In [ ]:
# ==============================================================================
# ENCODER VISUAL: DISTRIBUCIÓN DEMOGRÁFICA DE LA COHORTE PURIFICADA (FRONTAL)
# ==============================================================================

plt.style.use('default')
plt.figure(figsize=(12, 6), dpi=120)

# Renderizado del histograma demográfico con curvas de densidad (KDE) superpuestas
# Cambiamos 'multiple' a 'layer' con transparencia (alpha) para una comparación rigurosa
sns.histplot(
    data=df_frontal,
    x='Age',
    hue='Sex',
    multiple="layer",
    kde=True,
    bins=30,
    palette="viridis",
    alpha=0.4,
    edgecolor="black",
    linewidth=0.7
)

# Estética institucional avanzada para la memoria del TFG
plt.title("Distribución Demográfica de la Cohorte Purificada (Edad y Sexo - Plano Frontal)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Edad del Paciente (Años)", fontsize=11, fontweight='bold')
plt.ylabel("Frecuencia Absoluta (Número de Estudios)", fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle=':', alpha=0.6)

# Ajuste fino de los límites para evitar colas de interpolación vacías en el KDE
plt.xlim(18, 95)

plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD - CONFIG)
# ----------------------------------------------------------------------
save_path_demografia = os.path.join(CONFIG["dir"]["eda"], "distribucion_demografica_cohorte.png")

# Guardamos a 300 DPI asegurando el encuadre perfecto de la leyenda y los ejes
plt.savefig(save_path_demografia, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Histograma demográfico frontal exportado con éxito a Drive: {save_path_demografia}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()



El análisis de la estructura demográfica de la cohorte purificada frontal (Figura X) es un requisito metodológico crucial para evaluar la representatividad del dataset de entrenamiento e identificar posibles sesgos inductivos que pudieran comprometer la equidad (*fairness*) y la capacidad de generalización del clasificador convolucional. A partir del histograma de frecuencias y las curvas de estimación de densidad por kernel (*KDE*) segmentadas por sexo, se extraen las siguientes conclusiones analíticas fundamentales:

* **1. Distribución Etaria y Asimetría Negativa:** El perfil de edad exhibe una distribución unimodal con una marcada **asimetría hacia la izquierda (asimetría negativa)**. La masa estadística crítica se concentra de manera densa en el segmento de pacientes de edad mediana y avanzada, concretamente en el intervalo intercuartílico de los **55 a los 75 años**, situando el pico de densidad máxima operativa en el entorno de los 60 a 65 años. Clínicamente, esta distribución refleja fielmente la epidemiología hospitalaria real: las afecciones cardiorrespiratorias agudas y crónicas (como el edema pulmonar, la cardiomegalia o el derrame pleural) presentan curvas de incidencia que escalan de manera natural con el envejecimiento biológico de la población.
* **2. Distribución y Representación por Sexo:** A lo largo de todo el espectro etario, la cohorte frontal purificada muestra una distribución volumétrica dominada por la población masculina (*Male*, barras y traza turquesa) en términos absolutos, seguida de cerca por la población femenina (*Female*, barras y traza azul grisáceo). A pesar de la diferencia en los máximos absolutos de frecuencia por contenedor (*bin*), ambas curvas de densidad (*KDE*) exhiben perfiles morfológicos paralelos y un solapamiento geométrico simétrico en sus trayectorias. Esto garantiza que el optimizador de la *DenseNet121* se someterá a un aprendizaje balanceado, extrayendo descriptores anatómicos sin sesgos de género históricos. Por su parte, la fracción de registros indeterminados (*Unknown*, traza verde claro) es marginalmente nula, validando la exhaustividad del filtrado administrativo.
* **3. Criterio de Exclusión Pediátrica y Truncado de Privacidad ($90+$):** Se constata la ausencia absoluta de registros correspondientes a la población pediátrica ($<18$ años), delimitando el alcance operativo de la red a la anatomía del tórax adulto. Por otro lado, **el repunte abrupto y anómalo observable en la frontera de los 90 años no responde a una distribución biológica real**, sino a una estrategia estricta de anonimización en el dataset de origen. El sistema agrupa e indexa de forma acumulativa a todos los pacientes longevos de centiles superiores en este único contenedor de control para salvaguardar la privacidad de sus datos de salud, un factor que la función de coste asimila sin distorsionar la convergencia del vector latente.

### <font color='darkblue'>2.2.6 Auditoría de Artefactos y Sesgos Periféricos</font>
Inspección visual de las radiografías crudas para identificar marcadores radiopacos y textos institucionales incrustados, justificando la necesidad metodológica de aplicar una guillotina geométrica preventiva antes de la optimización del modelo.

In [ ]:
# ==============================================================================
# EDA: DETECCIÓN DE ARTEFACTOS Y TEXTOS PERIFÉRICOS (SHORTCUT LEARNING)
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
import random

plt.style.use('default')

# Inicialización de cuadrícula para auditoría de márgenes (1 fila, 3 columnas)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=120)

for i in range(3):
    # Selección aleatoria indexada sobre el volumen frontal
    idx = random.randint(0, len(df_frontal) - 1)
    row = df_frontal.iloc[idx]

    img_path = os.path.join(audit.img_root, row['Path_Local'])
    img = Image.open(img_path).convert('L')
    w, h = img.size

    # Despliegue fotométrico
    axes[i].imshow(img, cmap='bone')

    # ------------------------------------------------------------------
    # SIMULACIÓN DE LA GUILLOTINA: Sombreado de las zonas de exclusión (8%)
    # ------------------------------------------------------------------
    margen_y = h * 0.08
    margen_x = w * 0.08

    # Zona Superior (Riesgo de textos técnicos tipo "AP ERECT" o "PORTABLE")
    rect_top = patches.Rectangle((0, 0), w, margen_y, linewidth=1, edgecolor='red', facecolor='red', alpha=0.25)
    # Zona Lateral Izquierda (Riesgo de marcas "L" o "R" de plomo)
    rect_left = patches.Rectangle((0, 0), margen_x, h, linewidth=1, edgecolor='red', facecolor='red', alpha=0.25)
    # Zona Lateral Derecha (Riesgo de marcas "L" o "R" de plomo)
    rect_right = patches.Rectangle((w - margen_x, 0), margen_x, h, linewidth=1, edgecolor='red', facecolor='red', alpha=0.25)

    # Proyección de las máscaras de advertencia sobre la imagen
    axes[i].add_patch(rect_top)
    axes[i].add_patch(rect_left)
    axes[i].add_patch(rect_right)

    axes[i].set_title(f"Muestra Cruda Index: {idx}\nDimensiones Nativas: {w}x{h}", fontsize=11, fontweight='bold', pad=10)
    axes[i].axis('off')

# Título de cabecera formal institucional
plt.suptitle("Fase 1: Auditoría de Zonas de Riesgo Periférico (Textos Institucionales y Marcas de Plomo)", fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD)
# ----------------------------------------------------------------------
save_path_fase1 = os.path.join(CONFIG["dir"]["eda"], "fase1_artefactos_perifericos.png")
plt.savefig(save_path_fase1, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Auditoría de Fase 1 (Artefactos Periféricos) exportada a Drive: {save_path_fase1}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

La inspección visual directa de las radiografías crudas (Figura X) revela una vulnerabilidad estructural en la ingesta nativa del dataset CheXpert: la presencia sistemática de ruido periférico. Tal y como respalda la literatura metodológica de vanguardia, el rendimiento de aprendizaje de las redes neuronales profundas sobre radiografías crudas puede verse seriamente afectado por áreas ruidosas irrelevantes, como textos o la existencia de bordes irregulares. Como se resalta en las franjas sombreadas en rojo (equivalentes al 8% de los márgenes), las imágenes presentan con alta frecuencia marcas tipográficas de plomo (las letras "L" o "R" indicativas de lateralidad) y textos institucionales incrustados por el escáner (tales como "AP ERECT" o "PORTABLE").

Desde la perspectiva del modelado profundo (*Deep Learning*), estos elementos radiopacos de alto contraste actúan como atajos espurios (*Shortcut Learning*). Si el optimizador detecta que el texto "PORTABLE" correlaciona estadísticamente con pacientes encamados en unidades de cuidados intensivos, la red convolucional aprenderá a minimizar su función de pérdida prediciendo patologías graves basándose en el reconocimiento tipográfico, ignorando por completo la extracción de características anatómicas del parénquima pulmonar.

Esta auditoría empírica justifica categóricamente la programación de nuestra **guillotina anatómica asimétrica (`SafeCropMargind`)** en el pipeline de preprocesamiento de MONAI. La ablación determinista del **8%** de los márgenes superior y laterales blinda matemáticamente a la red contra estos artefactos no biológicos. Paralelamente, el análisis visual corrobora que el margen inferior debe mantenerse intacto (0% de recorte); aplicar una exclusión en la base amputaría los senos costodiafragmáticos, destruyendo la única región de interés (ROI) válida para la detección de la clase *Pleural Effusion* (derrame pleural).

### <font color='darkblue'>2.2.7. Auditoría de Geometría y Relación de Aspecto</font>
Análisis de las dimensiones nativas de las radiografías para evidenciar la extrema variabilidad en la relación de aspecto (*Aspect Ratio*), fundamentando la necesidad ineludible de aplicar una estandarización isométrica con relleno espacial (*padding*) frente a un redimensionado tradicional.

In [ ]:
# ==============================================================================
# EDA FASE 2: AUDITORÍA DE RELACIÓN DE ASPECTO Y DEFORMACIÓN ANATÓMICA
# ==============================================================================
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import os
import random

plt.style.use('default')

# Inicialización de la cuadrícula: 3 columnas (muestras) x 3 filas (estados)
fig, axes = plt.subplots(3, 3, figsize=(16, 14), dpi=120)

random.seed(42)

for i in range(3):
    # 1. Extracción de la muestra
    idx = random.randint(0, len(df_frontal) - 1)
    row = df_frontal.iloc[idx]
    img_path = os.path.join(audit.img_root, row['Path_Local'])

    # Carga de la imagen original en escala de grises
    img_original = Image.open(img_path).convert('L')
    w, h = img_original.size
    aspect_ratio = w / h

    # ------------------------------------------------------------------
    # FILA 1: ESTADO NATIVO
    # ------------------------------------------------------------------
    axes[0, i].imshow(img_original, cmap='bone')
    axes[0, i].set_title(f"Muestra {idx} - NATIVA\nResolución: {w}x{h}\nAspect Ratio: {aspect_ratio:.2f}", fontsize=11, fontweight='bold', pad=10, color='black')
    axes[0, i].axis('off')

    # ------------------------------------------------------------------
    # FILA 2: EL PELIGRO (Resize Estándar Destructivo 320x320)
    # ------------------------------------------------------------------
    img_bad_resize = img_original.resize((320, 320))
    axes[1, i].imshow(img_bad_resize, cmap='bone')
    axes[1, i].set_title("BAD PRACTICE: Resize Forzado\nAnatomía aplastada/estirada", fontsize=11, fontweight='bold', pad=10, color='darkred')
    axes[1, i].axis('off')

    # ------------------------------------------------------------------
    # FILA 3: LA SOLUCIÓN SOTA (Preservación Isométrica + Padding)
    # ------------------------------------------------------------------
    # Simulamos el comportamiento de MONAI (size_mode="longest" + SpatialPadd)
    if w > h:
        new_w = 320
        new_h = int(h * (320 / w))
    else:
        new_h = 320
        new_w = int(w * (320 / h))

    img_isometric = img_original.resize((new_w, new_h))

    # Creamos un lienzo negro absoluto simulando el padding del tensor (-1024)
    canvas = Image.new('L', (320, 320), color=0)
    # Pegamos la imagen isométrica en el centro
    offset_x = (320 - new_w) // 2
    offset_y = (320 - new_h) // 2
    canvas.paste(img_isometric, (offset_x, offset_y))

    axes[2, i].imshow(canvas, cmap='bone')
    axes[2, i].set_title("SOTA PIPELINE (MONAI)\nLongest: 320px + Padding Simétrico", fontsize=11, fontweight='bold', pad=10, color='darkgreen')
    axes[2, i].axis('off')

# Cabeceras de fila para mayor claridad
axes[0, 0].set_ylabel("1. RAW", fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel("2. DESTROY", fontsize=14, fontweight='bold')
axes[2, 0].set_ylabel("3. SOTA", fontsize=14, fontweight='bold')

plt.suptitle("Fase 2: Impacto del Redimensionado en la Proporción Anatómica (Prevención de Falsas Cardiomegalias)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE
# ----------------------------------------------------------------------
save_path_fase2 = os.path.join(CONFIG["dir"]["eda"], "fase2_geometria_aspect_ratio.png")
plt.savefig(save_path_fase2, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Auditoría de Fase 2 (Geometría y Proporciones) exportada a Drive: {save_path_fase2}")
print("="*80 + "\n")

plt.show()

El análisis de la dimensionalidad de las muestras extraídas del dataset CheXpert pone de manifiesto una variabilidad extrema en las resoluciones nativas y, más críticamente, en la relación de aspecto (*Aspect Ratio*). Como se observa en la primera fila de la Figura X, las placas oscilan desde formatos marcadamente rectangulares verticales hasta configuraciones más cuadradas, reflejando diferencias en el hardware de captura y en el tamaño físico del paciente.

El diseño del preprocesamiento debe abordar un conflicto computacional estricto: las arquitecturas convolucionales exigen tensores de entrada cuadrados y uniformes (en nuestro caso, **320x320** píxeles). La solución adoptada por defecto en múltiples pipelines básicos consiste en forzar un redimensionamiento bidimensional directo. Sin embargo, como evidencia la segunda fila de nuestra simulación (*BAD PRACTICE*), forzar el encuadre destruye la isometría anatómica. En radiografías marcadamente estrechas, el aplastamiento vertical induce un ensanchamiento artificial de la silueta cardíaca. Esta deformación es clínicamente inaceptable, ya que la red neuronal interpretaría esta distorsión geométrica como un aumento del índice cardiotorácico, disparando de forma sistemática falsos positivos en la predicción de **Cardiomegalia** y corrompiendo la fiabilidad diagnóstica del modelo.

Esta demostración empírica justifica y exige la adopción de la arquitectura SOTA programada en nuestros pipelines de MONAI. Al configurar la transformación `mt.Resized` con el parámetro `size_mode="longest"`, se garantiza que el eje mayor escale exactamente a **320** píxeles, arrastrando al eje menor de forma matemática para mantener un 100% de fidelidad biológica. Seguidamente, la inyección de `mt.SpatialPadd` actúa como un estabilizador tensorial, completando el espacio remanente con un relleno simétrico de valor constante correspondiente al negro radiológico absoluto (**-1024.0**). El resultado (tercera fila) es un tensor cuadrado perfecto que satisface los requisitos de la GPU sin amputar estructuras anatómicas ni alterar las proporciones patológicas originales.

### <font color='darkblue'>2.2.8. Fase 3: Auditoría de Rango Dinámico (Fotometría)</font>
Análisis de las distribuciones de intensidad de los píxeles (histogramas) para evidenciar la inconsistencia en la exposición radiológica nativa y fundamentar la proyección matemática de los tensores al estándar continuo [-1024.0, 1024.0].

In [ ]:
# ==============================================================================
# EDA FASE 3: AUDITORÍA DE RANGO DINÁMICO Y ESTABILIZACIÓN FOTOMÉTRICA
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
import random

plt.style.use('default')

# Inicialización de la cuadrícula: 2 filas (Imágenes vs Histogramas) x 2 columnas (Crudo vs Normalizado)
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=120)

# 1. Extracción de la muestra
idx = random.randint(0, len(df_frontal) - 1)
row = df_frontal.iloc[idx]
img_path = os.path.join(audit.img_root, row['Path_Local'])

# Carga en matriz NumPy
img_original = Image.open(img_path).convert('L')
img_array_raw = np.array(img_original).astype('float32')

# ------------------------------------------------------------------
# COLUMNA IZQUIERDA: ESTADO NATIVO (Escala arbitraria 0-255)
# ------------------------------------------------------------------
# Imagen Cruda
im_raw = axes[0, 0].imshow(img_array_raw, cmap='bone')
axes[0, 0].set_title(f"Estado Nativo (Escala de Grises Estándar)\nRango Real: [{img_array_raw.min():.0f}, {img_array_raw.max():.0f}]", fontsize=11, fontweight='bold', pad=10)
axes[0, 0].axis('off')
fig.colorbar(im_raw, ax=axes[0, 0], fraction=0.046, pad=0.04)

# Histograma Crudo
axes[1, 0].hist(img_array_raw.flatten(), bins=50, color='gray', edgecolor='black', alpha=0.7)
axes[1, 0].set_title("Distribución de Frecuencias (Raw)", fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel("Intensidad del Píxel", fontsize=10)
axes[1, 0].set_ylabel("Frecuencia", fontsize=10)
axes[1, 0].grid(axis='y', linestyle=':', alpha=0.6)

# ------------------------------------------------------------------
# COLUMNA DERECHA: LA SOLUCIÓN SOTA (Proyección Lineal Médica)
# ------------------------------------------------------------------
# Simulación exacta de mt.ScaleIntensityd(minv=-1024.0, maxv=1024.0)
v_min, v_max = img_array_raw.min(), img_array_raw.max()
rango_deseado = 1024.0 - (-1024.0)
img_array_norm = (img_array_raw - v_min) / (v_max - v_min) * rango_deseado + (-1024.0)

# Imagen Normalizada
im_norm = axes[0, 1].imshow(img_array_norm, cmap='bone', vmin=-1024.0, vmax=1024.0)
axes[0, 1].set_title(f"Normalización Médica (MONAI ScaleIntensity)\nRango Proyectado: [-1024.0, 1024.0]", fontsize=11, fontweight='bold', pad=10, color='darkgreen')
axes[0, 1].axis('off')
fig.colorbar(im_norm, ax=axes[0, 1], fraction=0.046, pad=0.04)

# Histograma Normalizado
axes[1, 1].hist(img_array_norm.flatten(), bins=50, color='#1f4e79', edgecolor='black', alpha=0.8)
axes[1, 1].set_title("Distribución de Frecuencias (Normalizada)", fontsize=11, fontweight='bold', color='darkgreen')
axes[1, 1].set_xlabel("Intensidad Proyectada", fontsize=10)
axes[1, 1].grid(axis='y', linestyle=':', alpha=0.6)

plt.suptitle("Fase 3: Estabilización del Rango Dinámico (De Escala Nativa a Rango Tensor Matemático)", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE
# ----------------------------------------------------------------------
save_path_fase3 = os.path.join(CONFIG["dir"]["eda"], "fase3_fotometria_rango_dinamico.png")
plt.savefig(save_path_fase3, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Auditoría de Fase 3 (Fotometría y Normalización) exportada a Drive: {save_path_fase3}")
print("="*80 + "\n")

plt.show()

En la práctica clínica real, las radiografías provienen de múltiples equipos con configuraciones de hardware heterogéneas, kilovoltajes variables y diferentes tiempos de exposición. Como se aprecia en la columna izquierda de la Figura X, la carga nativa de las imágenes despliega histogramas de intensidad anclados a la escala de grises de 8-bits estandarizada (intervalos de **0 a 255**). Más críticamente, dependiendo del grado de exposición de la placa, el rango dinámico rara vez ocupa todo el espectro disponible, concentrando la información visual en picos asimétricos.

En el ecosistema del aprendizaje profundo (*Deep Learning*), inyectar estos tensores crudos con escalas numéricas incompatibles o descompensadas resulta destructivo para el optimizador matemático. Un rango dinámico inconsistente entre lotes (*batches*) provoca severas oscilaciones en el cálculo de la función de pérdida y una propagación de gradientes inestable durante el *backpropagation*, impidiendo la convergencia de los pesos de la red neuronal.

Para mitigar este colapso numérico, el pipeline de MONAI exige la implementación de la transformación fotométrica `ScaleIntensityd`. Como ilustra la columna derecha, esta función ejecuta una proyección lineal que expande el rango de los píxeles hacia el estándar continuo definido entre **[-1024.0, 1024.0]**. Esta normalización específica cumple un doble propósito metodológico: primero, emula matemáticamente el comportamiento de las Unidades Hounsfield (HU) propias de los escáneres volumétricos (TAC), estableciendo un negro radiológico absoluto en **-1024.0**; y segundo, centra la matriz de datos en un rango simétrico con media tendente a cero, lo cual es un requisito estructural para garantizar una activación fluida y numéricamente estable en las capas preentrenadas de nuestra arquitectura *DenseNet121*.

### <font color='darkblue'>2.2.10. Auditoría de Variabilidad Clínica y Robustez Operativa (Entorno UCI)</font>
Simulación empírica de las condiciones de adquisición en Unidades de Cuidados Intensivos (UCI) para justificar la inyección de ruido geométrico y fotométrico (*Data Augmentation*) como mecanismo de regularización frente al sobreajuste.

In [ ]:
# ==============================================================================
# EDA FASE 4: SIMULACIÓN ESTOCÁSTICA DEL ENTORNO CLÍNICO (AUMENTACIÓN DE DATOS)
# ==============================================================================
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance
import numpy as np
import os
import random

plt.style.use('default')

# Inicialización de la cuadrícula: 1 fila x 4 columnas (Demostración secuencial)
fig, axes = plt.subplots(1, 4, figsize=(20, 6), dpi=120)

# 1. Extracción de muestra base representativa
idx = random.randint(0, len(df_frontal) - 1)
row = df_frontal.iloc[idx]
img_path = os.path.join(audit.img_root, row['Path_Local'])
img_original = Image.open(img_path).convert('L')

# ------------------------------------------------------------------
# PANEL 1: ESTADO BASAL (Control)
# ------------------------------------------------------------------
axes[0].imshow(img_original, cmap='bone')
axes[0].set_title("1. Estado Basal (Ideal)\nPaciente alineado y exposición óptima", fontsize=11, fontweight='bold', pad=10)
axes[0].axis('off')

# ------------------------------------------------------------------
# PANEL 2: VARIABILIDAD DE EXPOSICIÓN (RandAdjustContrastd)
# ------------------------------------------------------------------
# Simulamos una radiografía infraexpuesta (baja penetración de rayos X)
enhancer = ImageEnhance.Contrast(img_original)
img_contrast = enhancer.enhance(0.55)
axes[1].imshow(img_contrast, cmap='bone')
axes[1].set_title("2. Variabilidad de Exposición (UCI)\n(Justifica: RandAdjustContrastd)", fontsize=11, fontweight='bold', pad=10, color='#1f4e79')
axes[1].axis('off')

# ------------------------------------------------------------------
# PANEL 3: TORSIÓN POSTURAL ASIMÉTRICA (RandAffined)
# ------------------------------------------------------------------
# Simulamos paciente encamado: Rotación de 8 grados y traslación lateral de 30px (eje Y bloqueado)
img_affine = img_original.rotate(-8, translate=(30, 0), fillcolor=0)
axes[2].imshow(img_affine, cmap='bone')
axes[2].set_title("3. Torsión Postural en Camilla\n(Justifica: RandAffined Asimétrico)", fontsize=11, fontweight='bold', pad=10, color='darkorange')
axes[2].axis('off')

# ------------------------------------------------------------------
# PANEL 4: OCLUSIÓN REGULARIZADORA (RandCoarseDropoutd / Cutout)
# ------------------------------------------------------------------
img_array = np.array(img_original)
h, w = img_array.shape

# Inyectamos dos parches ciegos simulando la técnica Cutout
cx1, cy1 = int(w * 0.25), int(h * 0.35)
cx2, cy2 = int(w * 0.65), int(h * 0.55)
size = 70

img_cutout = img_array.copy()
# Aplicamos parches de color negro absoluto
img_cutout[cy1:cy1+size, cx1:cx1+size] = 0
img_cutout[cy2:cy2+size, cx2:cx2+size] = 0

axes[3].imshow(img_cutout, cmap='bone')
axes[3].set_title("4. Oclusión de Características\n(Justifica: CoarseDropout / Cutout)", fontsize=11, fontweight='bold', pad=10, color='darkred')
axes[3].axis('off')

plt.suptitle("Fase 4: Justificación Empírica de la Aumentación de Datos (Prevención de Memorización y Sobreajuste)", fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE
# ----------------------------------------------------------------------
save_path_fase4 = os.path.join(CONFIG["dir"]["eda"], "fase4_aumentacion_entorno_uci.png")
plt.savefig(save_path_fase4, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Auditoría de Fase 4 (Variabilidad y Aumentación) exportada a Drive: {save_path_fase4}")
print("="*80 + "\n")

plt.show()

El corpus clínico de CheXpert presenta una particularidad fundamental: una gran proporción de las radiografías fueron adquiridas utilizando equipos portátiles en Unidades de Cuidados Intensivos (placas tipo *AP Supine*). A diferencia de las exploraciones ambulatorias (donde el paciente colabora y se posiciona de forma simétrica), el paciente crítico suele presentar posturas asimétricas, rotaciones torácicas involuntarias y condiciones de exposición radiológica impredecibles (Figura X, Paneles 1 y 2).

Si la arquitectura *DenseNet121* fuera optimizada exclusivamente sobre el estado basal de las placas, el modelo colapsaría en la memorización de configuraciones anatómicas exactas, destruyendo su capacidad de generalización en entornos hospitalarios reales. Para blindar al modelo, se justifica clínicamente la inyección estocástica de transformaciones en el pipeline de entrenamiento de MONAI:

1. **Simulación Dinámica de Exposición (`RandAdjustContrastd`):** Al perturbar aleatoriamente el contraste fotométrico, se simulan las fluctuaciones de kilovoltaje (kVp) y tiempo de exposición inherentes a las máquinas de rayos X portátiles.
2. **Invarianza Postural Asimétrica (`RandAffined`):** Para emular la torsión del paciente en la camilla (Panel 3), se permite una rotación afín moderada. De forma crítica, la traslación espacial se programa de manera asimétrica: se permite un desplazamiento horizontal significativo (**30px**), pero se restringe severamente el desplazamiento vertical (**5px**). Esta asimetría matemática no es aleatoria; es una salvaguarda clínica programada específicamente para evitar amputar las bases pulmonares del marco del tensor.
3. **Oclusión de Características / Amnesia Inducida (`RandCoarseDropoutd`):** La simulación de la técnica *Cutout* (Panel 4) inyecta parches de ruido oscuro en el parénquima pulmonar. Al cegar aleatoriamente fracciones de la imagen, se neutraliza la tendencia de la red a basar su diagnóstico en un único parche hiper-localizado de alta activación. Esta perturbación fuerza al codificador a distribuir sus mapas de atención (*attention maps*) de manera global por todo el volumen torácico, extrayendo correlaciones redundantes y aumentando drásticamente la robustez de los *embeddings* que posteriormente alimentarán al modelo de lenguaje generativo.

### <font color='darkblue'>2.2.11. Validación Visual y Control de Calidad del Ecosistema de Datos</font>
Inspección cualitativa de muestras aleatorias para certificar el éxito del filtrado geométrico, validar la integridad binaria de los archivos y observar la coherencia semántica entre las etiquetas y los biomarcadores radiológicos.

In [ ]:
# ==============================================================================
# EDA GALERÍA DE CONTROL DE CALIDAD Y CONSISTENCIA SEMÁNTICA (FRONTAL)
# ==============================================================================
import random
import matplotlib.pyplot as plt
from PIL import Image
import os

plt.style.use('default')

# Inicialización de la cuadrícula de inspección (1 fila, 5 columnas de imágenes)
fig, axes = plt.subplots(1, 5, figsize=(20, 5), dpi=120)

# Fijamos una semilla local para que la galería sea reproducible en cada ejecución
random.seed(42)

for i in range(5):
    # Selección aleatoria indexada sobre el volumen filtrado frontal
    idx = random.randint(0, len(df_frontal) - 1)
    row = df_frontal.iloc[idx]

    # Construcción dinámica de la ruta segura sobre el almacenamiento local
    img_path = os.path.join(audit.img_root, row['Path_Local'])
    img = Image.open(img_path)

    # Buscamos qué patología tiene un '1' (Positivo) para mostrarla en el título
    patologias_activas = [col for col in audit.target_cols if row[col] == 1.0]

    # Construimos un string elegante con los hallazgos radiológicos
    if 'No Finding' in patologias_activas:
        etiqueta_titulo = "Control: No Finding"
    elif patologias_activas:
        # Limitamos a un máximo de 2 o 3 para que el título no desconfigure la trama visual
        etiqueta_titulo = f"Positivo: {', '.join(patologias_activas[:2])}"
        if len(patologias_activas) > 2:
            etiqueta_titulo += "..."
    else:
        etiqueta_titulo = "Hallazgo Sutil / Incierto"

    # Despliegue fotométrico usando el mapa de color clínico por excelencia ('bone')
    axes[i].imshow(img, cmap='bone')

    # Título refinado: Muestra el índice y su etiqueta clínica pura
    axes[i].set_title(f"Muestra Index: {idx}\n[{etiqueta_titulo}]", fontsize=10, fontweight='bold', pad=8)
    axes[i].axis('off')

# Título de cabecera formal institucional para el TFG
plt.suptitle("Galería de Control de Calidad Visual: Cohorte Purificada en Plano Frontal", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (ÚNICA FUENTE DE VERDAD)
# ----------------------------------------------------------------------
save_path_galeria = os.path.join(CONFIG["dir"]["eda"], "galeria_control_calidad_frontal.png")

plt.savefig(save_path_galeria, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Galería de control de calidad estrictamente frontal exportada a Drive: {save_path_galeria}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

Como último eslabón de la fase de auditoría técnica y de forma previa a la inyección de los tensores en el motor de grafos de cómputo del modelo, se ha implementado un protocolo estocástico de **Control de Calidad Visual**. Esta galería de inspección (Figura X) actúa como un mecanismo de verificación cualitativa para confirmar el éxito de las políticas de depuración ejecutadas sobre la base de datos cruda. La monitorización visual y su correspondencia directa con los metadatos clínicos permiten certificar tres hitos operativos críticos para la viabilidad de la arquitectura multimodal:

1. **Homogeneidad Absoluta de la Perspectiva de Entrada:**
   El renderizado estocástico confirma que el 100% de las matrices de píxeles expuestas corresponden efectivamente al plano anatómico frontal (AP/PA). Se valida así la purga completa de las proyecciones laterales (LAT), lo que asegura la consistencia estructural necesaria para que el codificador visual logre extraer con alta fidelidad topológica la **cuadrícula espacial de $10 \times 10$** (100 tokens visuales continuos), garantizando la invarianza geométrica requerida.

2. **Integridad Fotométrica y Calibración en Escala de Grises:**
   El despliegue bajo el mapa de color clínico estándar (`cmap='bone'`) permite ratificar la preservación de los gradientes densitométricos. La visualización nítida de las estructuras de referencia (parénquima pulmonar, silueta cardíaca, arcos costales y cúpulas diafragmáticas) descarta la presencia de archivos binarios corruptos o artefactos de descompresión, y valida la viabilidad de la normalización tensorial al rango continuo de **[-1024.0, 1024.0]** abordada previamente.

3. **Consistencia Semántica de la Indexación de Etiquetas:**
   La extracción dinámica de las variables objetivo (`target_cols`) sobre la cabecera de las muestras corrobora la coherencia diagnóstica del pipeline. Se verifica de forma empírica que las radiografías clasificadas como patológicas en el vector de entrenamiento exhiben los biomarcadores macroscópicos esperados (tales como la obliteración de los senos costofrénicos en muestras con *Pleural Effusion*, el ensanchamiento del contorno cardíaco en casos de *Cardiomegaly* o la presencia de opacidades algodonosas en *Lung Opacity*). Esta correspondencia clínica inicial es un requisito sine qua non para que la futura auditoría empírica mediante mapas **Grad-CAM** pueda certificar el anclaje anatómico de la red y la ausencia de atajos predictivos (*Shortcut Learning*).

Este diagnóstico cualitativo de consistencia cierra formalmente la fase de ingeniería y preparación del ecosistema de datos. Se garantiza de este modo que el codificador visual (*DenseNet121* adaptado mediante *Weight Surgery*) iniciará su optimización matemática sobre una base de conocimiento limpia, trazable y metodológicamente blindada frente a sesgos geométricos o solapamientos semánticos destructivos, lista para su posterior acoplamiento con el **Proyector MLP (Fase 2)** y el razonador clínico **BioGPT (Fase 3)**.


## <span style="color:#003366;">2.3. Preprocesamiento y Pipeline de Datos</span>

### <font color='darkblue'> 2.3.1. Estructuración del Pipeline de Datos y Carga Estocástica en MONAI </font>

Una vez depurada la base de datos y consolidada la cohorte estrictamente frontal (**n = 191.027**), se procedió a la estructuración del pipeline de carga de datos utilizando la librería especializada en entornos médicos **MONAI** (*Medical Open Network for AI*). Para ello, se diseñó un algoritmo de transformación matricial optimizado encargado de convertir los metadatos tabulares en colecciones indexadas de diccionarios, estructurados bajo el patrón clave-valor `{"image": ruta_absoluta, "label": vector_probabilidad}`.

El núcleo de esta implementación radica en la vectorización eficiente de la política de **mapeo de incertidumbre clínico (*Soft Targets*)** definida en secciones previas. El proceso de mapeo opera a través de las siguientes etapas lógicas directamente sobre la memoria RAM:

<font color='black'>

1. **Binarización por Defecto de la Ausencia:** Se realiza un volcado matricial del vector objetivo de 14 clases en paralelo donde los valores nulos (`NaN`) son mapeados de forma determinista al valor **0.0**. Esta operación asume la ausencia implícita de la patología, alineándose con la praxis de redacción por omisión característica de los reportes radiológicos originales.
2. **Amortiguación Estocástica Continua (Set de Entrenamiento):** Mediante la instanciación de una máscara booleana indexada sobre las posiciones de incertidumbre (valores originales de **-1.0**), se inyecta ruido uniforme continuo en el intervalo **[0.55, 0.85]**. Esta técnica actúa como un regularizador liso que mitiga el aprendizaje de atajos predictivos (*Shortcut Learning*) y flexibiliza la penalización de la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)** ante patrones visuales ambiguos, actuando como un blindaje frente al desbalanceo extremo (*Long-Tail*).
3. **Anclaje Determinista de Control (Set de Validación Interna):** Durante la estructuración de la cohorte de validación interna, la varianza estocástica se anula sustituyendo la incertidumbre por un valor escalar fijo de **0.55**. Esta guardia estructural blinda la reproducibilidad de los ensayos inter-época y ancla la evaluación a la detección de la mínima señal clínica de sospecha diagnóstica.
4. **Preservación del Patrón Oro (Set de Prueba):** El conjunto de prueba definitivo (*Test Set*), derivado del consenso de expertos, se procesa manteniendo intacta su binarización estricta original (**0.0** y **1.0**). Al carecer de ruido semántico, el algoritmo omite cualquier alteración, garantizando que el rendimiento final del modelo (evaluado en métricas como el Macro-AUC global) se mida contra verdades clínicas absolutas.

Esta estructuración de datos de alta velocidad optimiza el consumo del almacenamiento local y prepara los tensores de entrada para las transformaciones espaciales y fotométricas paralelas que tendrán lugar en los grafos de computación de la GPU.
</font>

El tratamiento de la incertidumbre diagnóstica (etiquetada como -1.0) descarta las aproximaciones de binarización estricta (estrategias *U-Zeros* o *U-Ones* propuestas originalmente por Irvin et al., 2019) dado que tienden a inyectar falsos negativos o a sobreestimar los falsos positivos, respectivamente. Para superar esta limitación estructural, nuestra arquitectura implementa la estrategia de **mapeo de incertidumbre clínico (*Soft Targets*)**, una evolución geométrica de la regularización validada por Pham et al. (2021).

Siguiendo esta metodología de vanguardia, el ruido semántico presente en la cohorte de entrenamiento se proyecta sobre una distribución estocástica uniforme $U(0.55, 0.85)$. Esta parametrización empírica demostró alcanzar el estado del arte (SOTA) en la clasificación multietiqueta del corpus CheXpert. Su éxito matemático radica en proporcionar a la red objetivos probabilísticos suaves (*soft targets*) que mitigan la penalización de la función de pérdida ante hallazgos clínicos ambiguos, logrando extraer el valor predictivo de la duda médica sin comprometer la confianza del modelo sobre los diagnósticos confirmados y operando como un blindaje estadístico frente al desbalanceo extremo (*Long-Tail*).

Por su parte, el tratamiento de la cohorte de validación interna exige un entorno de evaluación estrictamente determinista para garantizar la comparabilidad objetiva del rendimiento entre épocas. Inyectar ruido estocástico durante la inferencia provocaría oscilaciones artificiales en las métricas de error, enmascarando la verdadera tasa de convergencia del codificador visual (*DenseNet121*). Por consiguiente, la varianza matemática se suprime anclando los registros inciertos en un valor escalar fijo de **0.55**. La adopción de este umbral exacto —el límite inferior de la distribución de entrenamiento— responde a un principio de precaución clínica: exige al modelo reconocer la mínima señal de sospecha diagnóstica al superar ligeramente la frontera binaria del **0.5**, actuando como un ancla conservadora que previene el sobreajuste y estabiliza el cálculo de la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)**.


In [ ]:
# ==============================================================================
# ENCODER VISUAL: ESTRUCTURACIÓN DEL PIPELINE DE DATOS MONAI Y SPLIT POR PACIENTE
# ==============================================================================
import os
import numpy as np
import pandas as pd
import warnings
from sklearn.model_selection import train_test_split

# Supresión defensiva de advertencias estéticas del compilador
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

def crear_lista_monai_optimizada(df, img_root, target_cols, is_train=True):
    """
    Transforma los metadatos tabulares purificados en colecciones estructuradas
    de diccionarios (Key-Value) optimizadas para los grafos de carga de MONAI.
    """
    print(f" 📦 Estructurando {len(df):,} muestras clínicas para MONAI...")

    # Conversión matricial de etiquetas rellenando valores nulos (NaN -> 0: Ausencia)
    etiquetas_limpias = df[target_cols].fillna(0).values.astype('float32')

    # Máscara booleana para aislar el ruido semántico de la duda diagnóstica (-1)
    mask_uncertain = (etiquetas_limpias == -1)

    if is_train:
        # Modo Entrenamiento: Amortiguación estocástica continua uniforme
        ruido_aleatorio = np.random.uniform(low=0.55, high=0.85, size=etiquetas_limpias.shape)
        etiquetas_limpias[mask_uncertain] = ruido_aleatorio[mask_uncertain].astype('float32')
        print("    ➡️ Regularización: Label Smoothing [0.55 - 0.85] inyectado en incertidumbres.")
    else:
        # Modo Validación/Test: Valor estático de control (Guardia defensiva estructural)
        etiquetas_limpias[mask_uncertain] = 0.55
        print("    ➡️ Validación/Test: Estabilización determinista (0.55) o paso limpio ejecutado.")

    # Reconstrucción vectorial de rutas absolutas locales de alta velocidad (SSD)
    rutas_limpias = [os.path.join(img_root, p) for p in df['Path_Local']]

    # Compresión indexada en diccionarios nativos para MONAI Dataset
    lista_datos = [
        {"image": ruta, "label": label}
        for ruta, label in zip(rutas_limpias, etiquetas_limpias)
    ]

    return lista_datos

# ==============================================================================
# PURIFICACIÓN GEOMÉTRICA ANTES DEL SPLIT (BLOQUEO DE VISTAS LATERALES)
# ==============================================================================
print("\n" + "="*80)
print(" 🛡️ FILTRADO GEOMÉTRICO ESTRICTO: EXTRAYENDO COHORTES FRONTALES PURAS")
print("="*80)

# Filtramos los dataframes originales para quedarnos ÚNICA Y EXCLUSIVAMENTE con Frontales
df_train_frontal_puro = audit.train_df[audit.train_df['Frontal/Lateral'] == 'Frontal'].copy()
df_test_frontal_puro = audit.valid_df[audit.valid_df['Frontal/Lateral'] == 'Frontal'].copy()

# ==============================================================================
# RE-PARTICIÓN VECTORIZADA (SPLIT POR PACIENTE SOBRE LA COHORTE FRONTAL PURA)
# ==============================================================================

# 1. Extracción de Patient_ID en una columna temporal sobre el dataframe ya purificado
df_train_frontal_puro['Patient_ID'] = df_train_frontal_puro['Path'].str.extract(r'(patient\d+)')[0]
pacientes_frontales_unicos = df_train_frontal_puro['Patient_ID'].unique()

print(f" 👥 Pacientes únicos en la cohorte estrictamente frontal: {len(pacientes_frontales_unicos):,}")

# 2. Split 90/10 sobre los IDs de pacientes reales de frente
train_patients, val_patients = train_test_split(pacientes_frontales_unicos, test_size=0.10, random_state=42)

# 3. Filtrado ultrarrápido por .isin()
df_train_interno = df_train_frontal_puro[df_train_frontal_puro['Patient_ID'].isin(train_patients)].copy()
df_val_interno = df_train_frontal_puro[df_train_frontal_puro['Patient_ID'].isin(val_patients)].copy()

# ==============================================================================
# PIPELINE DE DESPLIEGUE: ASIGNACIÓN FINAL DE LAS 3 MATRICES FRONTALES A MONAI
# ==============================================================================

print("\n[FASE 1] Cargando Set de ENTRENAMIENTO (Frontal Puro)...")
train_data = crear_lista_monai_optimizada(
    df=df_train_interno,
    img_root=CONFIG["dir"]["data_local"],
    target_cols=audit.target_cols,
    is_train=True
)

print("\n[FASE 2] Cargando Set de VALIDACIÓN INTERNA (Frontal Puro)...")
val_data = crear_lista_monai_optimizada(
    df=df_val_interno,
    img_root=CONFIG["dir"]["data_local"],
    target_cols=audit.target_cols,
    is_train=False
)

print("\n[FASE 3] Cargando Set de TEST / PATRÓN ORO (Frontal Puro)...")
test_data = crear_lista_monai_optimizada(
    df=df_test_frontal_puro,
    img_root=CONFIG["dir"]["data_local"],
    target_cols=audit.target_cols,
    is_train=False
)

# ==============================================================================
# RESUMEN EJECUTIVO DE VERIFICACIÓN (PULIDO PARA EL TFG)
# ==============================================================================
print("\n" + "="*80)
print(" 🚀 PIPELINE DE DATOS MONAI CONSOLIDADO (100% PLANO FRONTAL)")
print("="*80)
print(f"  • Clases Activas (Target)      : {audit.target_cols}")
print(f"  • Matriz de Entrenamiento      : {len(train_data):,} muestras (Suavizado Continuo).")
print(f"  • Matriz de Validación Interna : {len(val_data):,} muestras (Determinista 0.55).")
print(f"  • Matriz de Test (Patrón Oro)  : {len(test_data):,} muestras (Consenso de Expertos).")
print("="*80 + "\n")

### <font color='darkblue'> 2.3.2. Arquitectura SOTA de Preprocesamiento y Aumentación de Datos en MONAI

Para la ingesta y adecuación de los tensores de imagen a la arquitectura *DenseNet121*, se ha diseñado un pipeline de transformaciones espaciales y fotométricas altamente optimizado mediante la librería MONAI. Este flujo de operaciones se fundamenta en las estrategias de los modelos de vanguardia (SOTA) en el análisis del dataset CheXpert, priorizando la preservación de los biomarcadores anatómicos periféricos y la mitigación del aprendizaje de atajos (*shortcut learning*).

El pipeline se desglosa en cuatro fases de transformación secuencial:

**1. Purificación Espacial y Recorte Anatómico Selectivo:**
Una vulnerabilidad crítica en la radiografía digital es la presencia de marcadores de plomo (como las letras "L" o "R") y etiquetas de texto superpuestas en los bordes superiores y laterales de la placa. Si no se eliminan, las redes convolucionales tienden a usarlos como atajos predictivos espurios. Para solucionar esto, se ha programado una clase personalizada (`SafeCropMargind`) que actúa como una guillotina geométrica asimétrica: recorta de forma determinista el **8%** de los márgenes superior e izquierdo/derecho, pero mantiene intacto el 100% del margen inferior. Esta asimetría es un requisito clínico ineludible, ya que amputar la base de la imagen destruiría los senos costodiafragmáticos, eliminando la única evidencia visual necesaria para diagnosticar la clase *Pleural Effusion* (Derrame Pleural).

**2. Resolución Escalada y Padding Simétrico:**
Superando el estándar industrial de 224x224, se ha incrementado la resolución espacial del pipeline a **320x320 píxeles** (`spatial_size=320`). Al escalar por el lado más largo (`size_mode="longest"`), se mantiene intacta la relación de aspecto original (mitigando el estiramiento anamórfico). Para convertir el resultado en un tensor cuadrado perfecto sin deformaciones, se aplica un relleno espacial simétrico (`SpatialPadd`) utilizando un valor de fondo constante equivalente al negro radiológico absoluto (**-1024.0**), evitando la creación de fronteras artificiales.

**3. Simulación Estocástica de Entornos UCI (Aumentación de Datos):**
Para el conjunto de entrenamiento, se inyectan transformaciones estocásticas diseñadas específicamente para emular la variabilidad geométrica de la práctica clínica real, especialmente en pacientes encamados de la Unidad de Cuidados Intensivos (UCI):
* **Ajuste Dinámico de Contraste:** (`RandAdjustContrastd`) Simula variaciones en el tiempo de exposición y kilovoltaje de la máquina de rayos X portátil.
* **Transformación Afín Asimétrica:** (`RandAffined`) Se aplica una rotación moderada para simular torsiones posturales del paciente en la camilla. Crucialmente, el rango de traslación espacial se configura de manera asimétrica: permite un desplazamiento horizontal de hasta **30 píxeles** (eje X), pero restringe el desplazamiento vertical a un máximo de **5 píxeles** (eje Y) para garantizar que las bases pulmonares o los ápices no salgan del marco del tensor.
* **Oclusión Espacial Regularizadora:** (`RandCoarseDropoutd`) Basado en la técnica *Cutout*, inyecta parches ciegos aleatorios en la imagen. Esto fuerza a la *DenseNet121* a no depender de una única región local de alta activación, obligando a los mapas de características a distribuir la atención por todo el parénquima pulmonar.

**4. Normalización Fotométrica Final:**
Previamente a la inyección en el optimizador, se aplica una re-escalada de intensidad lineal (`ScaleIntensityd`) que proyecta los valores de los píxeles hacia un rango continuo definido entre **[-1024.0, 1024.0]**. Esta normalización específica es requerida por los pesos preentrenados del ecosistema radiológico empleado, estabilizando los gradientes durante la propagación hacia atrás (*backpropagation*) e induciendo una convergencia más rápida.

Para las cohortes de **Validación Interna** y **Test**, el pipeline se ejecuta como un túnel de procesado determinista, aplicando de forma estricta las correcciones geométricas y la normalización fotométrica, pero deshabilitando por completo la inyección de ruido estocástico para garantizar una evaluación objetiva.

In [ ]:
# ==============================================================================
# 1. CLASES CUSTOM: INGENIERÍA DE MITIGACIÓN DE SESGOS CLÍNICOS
# ==============================================================================
class SafeCropMargind(mt.MapTransform):
    """
    Guillotina Anatómica Selectiva (Alineada con la Competición CheXpert).
    Recorta los márgenes laterales y superior para eliminar marcas L/R y ruido
    de posicionamiento, pero respeta el 100% de la base inferior para no amputar
    los senos costodiafragmáticos (crítico para detectar Derrame Pleural).
    """
    def __init__(self, keys, margin_pct=0.08):
        super().__init__(keys)
        self.margin_pct = margin_pct

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            _, h, w = img.shape

            crop_top = int(h * self.margin_pct)
            crop_side = int(w * self.margin_pct)

            # Cirugía geométrica: arriba y lados recortados, abajo intacto (h)
            d[key] = img[:, crop_top:h, crop_side:w-crop_side]
        return d

# ==============================================================================
# 2. PIPELINES DE MONAI (Arquitectura SOTA - Matemáticamente Segura)
# ==============================================================================
print("⏳ Ensamblando el pipeline de preprocesado de alta resolución (320x320)...")

train_transforms_sota = mt.Compose([
    mt.LoadImaged(keys=["image"], image_only=True),
    mt.EnsureChannelFirstd(keys=["image"]),
    mt.Transposed(keys=["image"], indices=[0, 2, 1]),

    # 1. PURIFICACIÓN ESPACIAL (Filtro anti-sesgos textuales)
    SafeCropMargind(keys=["image"], margin_pct=0.08),

    # 2. SIMULACIÓN DE EXPOSICIÓN (Se aplica ANTES de introducir valores negativos)
    mt.RandAdjustContrastd(keys=["image"], prob=0.5, gamma=(0.8, 1.2)),

    # 3. RESOLUCIÓN DE COMPETICIÓN (320x320)
    mt.Resized(keys=["image"], spatial_size=320, size_mode="longest"),

    # 4. SIMULACIÓN DE POSTURA HOSPITALARIA ASIMÉTRICA
    mt.RandAffined(
        keys=["image"],
        prob=0.7,
        rotate_range=0.17,       # +/- 9.7 grados (El paciente se tuerce en la camilla)
        translate_range=(30, 5), # FIX TFG: 30px Eje X (Desplazamiento lateral), 5px Eje Y (Protección de base)
        scale_range=(0.05, 0.05),
        mode="bilinear",
        padding_mode="zeros"
    ),

    # 5. NORMALIZACIÓN MÉDICA FINAL (El salto al rango dinámico [-1024.0, 1024.0])
    mt.ScaleIntensityd(keys=["image"], minv=-1024.0, maxv=1024.0),

    # 6. PADDING ESPACIAL DE SEGURIDAD (Garantiza simetría perfecta 320x320 en el tensor)
    mt.SpatialPadd(keys=["image"], spatial_size=(320, 320), method="symmetric", constant_values=-1024.0),

    # 7. REGULARIZACIÓN SOTA (Cutout robusto para evitar sobreajuste en parches locales)
    mt.RandCoarseDropoutd(keys=["image"], holes=2, spatial_size=70, fill_value=-1024.0, prob=0.5)
])

# Pipeline de Validación (El túnel de lavado limpio, clínico y determinista)
val_transforms_sota = mt.Compose([
    mt.LoadImaged(keys=["image"], image_only=True),
    mt.EnsureChannelFirstd(keys=["image"]),
    mt.Transposed(keys=["image"], indices=[0, 2, 1]),

    SafeCropMargind(keys=["image"], margin_pct=0.08),
    mt.Resized(keys=["image"], spatial_size=320, size_mode="longest"),
    mt.ScaleIntensityd(keys=["image"], minv=-1024.0, maxv=1024.0),
    mt.SpatialPadd(keys=["image"], spatial_size=(320, 320), method="symmetric", constant_values=-1024.0)
])

# ==============================================================================
# 3. INSTANCIACIÓN DE DATASETS Y DATALOADERS OPTIMIZADOS (FIX CONVOLUCIONAL)
# ==============================================================================
print("📦 Empaquetando tensores en los DataLoaders de alta resolución...")

# Uso del Dataset nativo de MONAI
train_dataset = Dataset(data=train_data, transform=train_transforms_sota)
val_dataset = Dataset(data=val_data, transform=val_transforms_sota)

# CORRECCIÓN: Usamos MonaiDataLoader (definido en los imports) para gestionar de forma eficiente
train_loader = MonaiDataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=False)
val_loader = MonaiDataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

# Instanciamos el Dataset y DataLoader para el set de Test que ya tenías creado
test_dataset = Dataset(data=test_data, transform=val_transforms_sota)
test_loader = MonaiDataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f"✅ Test DataLoader instanciado con {len(test_data):,} muestras de Patrón Oro.")
print("\n" + "="*80)
print(" ✅ PIPELINES DE PRODUCCIÓN MULTIMODAL LISTOS")
print("="*80)
print("  • Resolución de Tensores : 320 x 320 píxeles")
print("  • Rango de Normalización : [-1024.0, 1024.0] (Negro radiológico absoluto)")
print("  • Optimización de Cuda   : pin_memory activado para transferencia síncrona")
print("="*80 + "\n")

### <font color='darkblue'>2.3.4. Auditoría Visual y Validación Estocástica del Preprocesamiento</font>



Como paso previo definitivo antes de inyectar los tensores en la arquitectura *DenseNet121*, es imperativo realizar una validación empírica que certifique el correcto funcionamiento de los grafos de computación diseñados en MONAI. La abstracción matemática de las transformaciones espaciales y fotométricas exige una comprobación cualitativa para garantizar que el modelo no asimile artefactos destructivos durante el entrenamiento.

Para ello, se ha implementado un motor de **Auditoría Visual Tripartita**. Este algoritmo extrae de forma estocástica una muestra clínica y renderiza en paralelo las tres etapas vitales de nuestro ecosistema de datos, exportando una infografía de alta resolución estandarizada para la memoria del Trabajo de Fin de Grado. Esta inspección permite corroborar visual y matemáticamente tres hitos estructurales:

1. **El Estado Basal (Radiografía Cruda):** Evidencia la geometría nativa inestable y la presencia de ruido periférico (etiquetas de plomo o texto) que la red podría utilizar como atajos espurios.
2. **El Túnel Determinista (Validación/Inferencia):** Certifica la eficacia de la "guillotina marginal" y la aplicación del relleno espacial simétrico (*Padding* con valor -1024.0). Confirma empíricamente que la imagen se estabiliza en un tensor cuadrado perfecto libre de deformaciones anamórficas, garantizando la resolución exacta requerida para la posterior extracción de la cuadrícula de $10 \times 10$.
3. **La Robustez Estocástica (Entrenamiento):** Demuestra la correcta inyección de la variabilidad operativa. Permite visualizar la torsión geométrica (simulando las condiciones complejas de las placas portátiles en la UCI) y el enmascaramiento regularizador (*Cutout*), verificando empíricamente nuestra principal estrategia arquitectónica contra el aprendizaje de atajos predictivos (*Shortcut Learning*).

A continuación, se despliega el código responsable de esta auditoría clínica:

In [ ]:
# ==============================================================================
# AUDITORÍA VISUAL DEL PIPELINE DE PREPROCESADO Y EXPORTACIÓN DE CONTROL (TFG)
# ==============================================================================
import os
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

print("⏳ Iniciando Auditoría Visual del Pipeline de Preprocesado...")

# Configuración de estilos de publicación científica (Formato TFG)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans']
plt.style.use('default')

# 1. Seleccionamos un paciente aleatorio usando una semilla para reproducibilidad en la memoria
random.seed(42)
idx = random.randint(0, len(val_data) - 1)
muestra_cruda = val_data[idx]
ruta_img = muestra_cruda['image']

# --- A. CARGA ORIGINAL ---
img_pil = Image.open(ruta_img).convert('L')
img_original = np.array(img_pil)

# --- B. PIPELINE DE VALIDACIÓN (Túnel de lavado clínico determinista) ---
tensor_val = val_transforms_sota(muestra_cruda)['image']
img_val = tensor_val[0].numpy() # Conversión a matriz 2D para Matplotlib

# --- C. PIPELINE DE ENTRENAMIENTO (Aumento estocástico de datos) ---
tensor_train = train_transforms_sota(muestra_cruda)['image']
img_train = tensor_train[0].numpy()

# ==============================================================================
# VISUALIZACIÓN EN PARALELO (COMPOSICIÓN TRIPARTITA DE ALTA RESOLUCIÓN)
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 7.5), dpi=120)

# Panel 1: Estado Original Bruto
axes[0].imshow(img_original, cmap='bone')
title_0 = f"1. Radiografía Cruda (Basal)\n\n" \
          f"• Dimensiones nativas: {img_original.shape[1]}x{img_original.shape[0]}\n" \
          f"• Presencia de ruido marginal"
axes[0].set_title(title_0, fontsize=12, fontweight='bold', pad=12, color='#1f4e79', loc='center')
axes[0].axis('off')

# Panel 2: Dataset de Inferencia/Validación (Formato Limpio Estándar)
# Dejamos que Matplotlib escanee los mínimos y máximos reales del tensor para un contraste óptimo
axes[1].imshow(img_val, cmap='bone', vmin=img_val.min(), vmax=img_val.max())
title_1 = f"2. Pipeline de Inferencia (Validación)\n\n" \
          f"• Dimensión: {img_val.shape[1]}x{img_val.shape[0]} (Padding)\n" \
          f"• Guillotina Marginal aplicada\n" \
          f"• Escala: [{img_val.min():.1f}, {img_val.max():.1f}]"
axes[1].set_title(title_1, fontsize=12, fontweight='bold', pad=12, color='#1f4e79', loc='center')
axes[1].axis('off')

# Panel 3: Dataset de Entrenamiento (Robustez ante Variabilidad Operativa)
axes[2].imshow(img_train, cmap='bone', vmin=img_train.min(), vmax=img_train.max())
title_2 = f"3. Pipeline de Aumento (Entrenamiento)\n\n" \
          f"• Transformación Afín (Simulación UCI)\n" \
          f"• Normalización Dinámica\n" \
          f"• Cutout (Oclusión Regularizadora)"
axes[2].set_title(title_2, fontsize=12, fontweight='bold', pad=12, color='#1f4e79', loc='center')
axes[2].axis('off')


# ----------------------------------------------------------------------
# GUARDADO AUTOMÁTICO EN DRIVE (Imágenes Clínicas para la Memoria)
# ----------------------------------------------------------------------
save_path_pipeline = os.path.join(CONFIG["dir"]["eda"], "auditoria_pipeline_preprocesado.png")

plt.savefig(save_path_pipeline, dpi=300, bbox_inches='tight')

print("\n" + "=" * 80)
print(f"✅ Panel comparativo del pipeline exportado a Drive: {save_path_pipeline}")
print(f"🔗 Ruta del estudio analizado: {ruta_img}")
print("=" * 80 + "\n")

plt.show()

El tríptico secuencial (Figura X) ilustra el ciclo de vida de un tensor radiográfico a través de las diferentes etapas de nuestro pipeline de preprocesamiento avanzado en MONAI (`train_transforms_sota` y `val_transforms_sota`). Este análisis visual cualitativo confirma la efectividad operativa de las técnicas de mitigación de sesgos aplicadas:

* **1. Estado Inicial (Radiografía Cruda Basal):**
  La primera imagen expone el estado nativo del dataset CheXpert (con una dimensión original no estandarizada, por ejemplo, **320x390** píxeles, y un formato marcadamente vertical). Se aprecia con total claridad la presencia de la marca tipográfica de plomo "R" (esquina superior izquierda) y el texto técnico de adquisición ("AP ERECT"). Estos elementos periféricos de alto contraste constituyen los "atajos predictivos" (*Shortcut Learning*) que sesgan y comprometen la optimización del modelo convolucional si no son suprimidos.

* **2. Estado de Validación e Inferencia (Purificación Espacial e Isometría):**
  La segunda imagen muestra el resultado del pipeline determinista empleado para la evaluación del modelo.
  * Se comprueba el éxito de la **guillotina anatómica** (`SafeCropMargind`): los textos técnicos y las marcas de plomo han sido completamente extirpados del campo de visión de la red, respetando de forma estricta e íntegra las bases pulmonares.
  * Se evidencia el funcionamiento de la **preservación isométrica** (`size_mode="longest"`): el parénquima pulmonar y la silueta cardíaca mantienen sus proporciones biológicas exactas, previniendo falsos diagnósticos de cardiomegalia inducidos por deformación. Dado que la imagen original era rectangular, el espacio remanente ha sido completado de forma simétrica con relleno constante (*padding*) equivalente al negro radiológico absoluto (**-1024.0**), logrando el formato cuadrado de **320x320** requerido por la GPU de manera matemáticamente segura.

* **3. Estado de Entrenamiento (Aumento de Datos Estocástico):**
  La tercera imagen exhibe la variante de robustez utilizada exclusivamente durante la optimización de los pesos de la *DenseNet121* (previamente adaptada mediante *Weight Surgery*).
  * Se observa una **rotación afín y traslación lateral moderada** (`RandAffined`), simulando con fidelidad el posicionamiento asimétrico real y la torsión postural típica de un paciente crítico encamado en la UCI.
  * Así mismo, se evidencian los efectos de la oclusión espacial o *Cutout*. Esta alteración obliga a la red a desarrollar invarianza posicional, garantizando que los mapas de características (*feature maps*) extraídos para la posterior **cuadrícula espacial de $10 \times 10$** (100 tokens visuales) anclen las patologías a la anatomía pulmonar global, y no a un sistema de coordenadas estático o a un único foco de alta activación. El éxito de esta dispersión de la atención será auditable empíricamente mediante **Grad-CAM**.

> **Conclusión del Control de Calidad:** El panel confirma visualmente que el pipeline multimodal de MONAI actúa como una defensa arquitectónica robusta. Purifica el tensor de artefactos no biológicos, estandariza las dimensiones sin deformar la anatomía del paciente e inyecta la variabilidad geométrica necesaria para garantizar que las representaciones latentes transferidas al **Proyector MLP (Fase 2)** y, en última instancia, al modelo de lenguaje **BioGPT (Fase 3)**, posean la máxima resolución semántica, topológica y clínica posible.


## <span style="color:#003366;">2.4. Definición de la Arquitectura y Criterios de Optimización</span>

### <font color='darkblue'>2.4.1. Selección de la Red Neuronal: DenseNet121</font>
Se ha seleccionado la arquitectura **DenseNet121** debido a su eficiencia probada en el procesamiento de imágenes médicas de tórax. Su diseño, basado en bloques densos con conexiones directas entre todas las capas, permite una mejor propagación del gradiente y fomenta la reutilización de características (*feature reuse*). Esta capacidad es fundamental para detectar patrones sutiles en radiografías, como infiltrados o derrames, con un número de parámetros significativamente menor que otras arquitecturas como ResNet.
#### **Fundamentación Matemática y Estructural**

A diferencia de las arquitecturas residuales (ResNet) que utilizan conexiones de salto basadas en la suma elemental ($x_\ell = f(x_{\ell-1}) + x_{\ell-1}$), DenseNet propone un paradigma de **conectividad densa**. En esta estructura, cada capa recibe como entrada los mapas de características de **todas** las capas precedentes dentro de un bloque denso.

Formalmente, si definimos $H_\ell(\cdot)$ como una función compuesta de *Batch Normalization* (BN), *ReLU* y una convolución de $3 \times 3$, la salida de la capa $\ell$ se expresa mediante la siguiente relación de concatenación:

$$x_\ell = H_\ell([x_0, x_1, \dots, x_{\ell-1}])$$

Donde:
* $[x_0, x_1, \dots, x_{\ell-1}]$ representa la **concatenación** de los mapas de características producidos en las capas $0, \dots, \ell-1$.
* Esta arquitectura permite que la información de bajo nivel (bordes, texturas) persista hasta las capas más profundas, algo vital en radiología donde un detalle de pocos píxeles define un diagnóstico.

#### **Componentes Críticos de la Red**

1.  **Bloques Densos (Dense Blocks):** Son los módulos donde ocurre la concatenación exhaustiva. Gracias a la **Tasa de Crecimiento ($k$)**, cada capa añade solo un número pequeño de nuevos canales ($k=32$ en este modelo), manteniendo la eficiencia de parámetros.
2.  **Capas de Transición (Transition Layers):** Situadas entre bloques densos, utilizan convoluciones de $1 \times 1$ y *Average Pooling* para reducir el número de canales y la resolución espacial, controlando la complejidad computacional.
3.  **Cuello de Botella (Bottleneck Layers):** Introducen convoluciones $1 \times 1$ antes de las de $3 \times 3$ para reducir el número de mapas de entrada, optimizando el uso de la memoria de la GPU.



#### **Justificación de la Elección**
La selección de **DenseNet121** sobre arquitecturas significativamente más complejas o pesadas se fundamenta sólidamente en la literatura empírica de élite para el análisis de radiografías de tórax. Específicamente, en el estudio fundacional del corpus CheXpert desarrollado por **Irvin et al. (2019)**, el equipo de la Universidad de Stanford evaluó comparativamente diversas redes neuronales profundas de vanguardia —incluyendo *ResNet-152*, *Inception-v4* y *SE-ResNeXt101*— concluyendo de forma inequívoca que la arquitectura **DenseNet121** es la que reporta de manera consistente los mejores resultados de clasificación y generalización clínica. Esta elección es secundada por el marco metodológico de **Pham et al. (2021)**, quienes ratifican su uso debido a su extraordinaria eficiencia en la reutilización de características, una propiedad que reduce drásticamente el volumen de parámetros requeridos y mitiga el riesgo de sobreajuste (*overfitting*) en conjuntos de datos médicos.

Estructuralmente, tras procesar las imágenes bajo nuestro pipeline de alta resolución (320x320 píxeles), la capa de agrupación global previa a la clasificación de esta red consolida un vector de características latentes de exactamente **1024 dimensiones**. Este cuello de botella funcional comprime un espacio semántico densamente enriquecido, idóneo para actuar como el extractor de características visuales (*Módulo I*) que posteriormente alimentará de forma estable la alineación contrastiva con CLIP (Fase 2) y el decodificador de lenguaje autorregresivo GPT (Fase 3).


In [ ]:
from IPython.display import Image as IPyImage, display
import os

# @title Mostrar Esquema Arquitectónico - Módulo I (DenseNet121 Encoder)
# ==============================================================================
# DESPLIEGUE DE DIAGRAMA MEDIANTE CONFIGURACIÓN CENTRALIZADA
# ==============================================================================

# 1. Definimos la ruta exacta usando la "Única Fuente de Verdad"
ruta_esquema = os.path.join(CONFIG["dir"]["assets"], "ImagenDense.png")

# 2. Invocamos la visualización real en pantalla
display(IPyImage(filename=ruta_esquema, width=800))

*Figura 1: Arquitectura de un bloque denso (Huang et al., 2017)*

### <font color='darkblue'>2.4.2. Configuración Inicial y Reproducibilidad Científica</font>
En la investigación basada en *Deep Learning*, la validación empírica exige que los experimentos sean estrictamente reproducibles. Las arquitecturas convolucionales y los optimizadores estocásticos introducen múltiples fuentes de aleatoriedad (inicialización de pesos, orden de los *batches*, operaciones internas de la GPU).

Mediante la función `set_seed()`, se ancla la entropía del sistema unificando las semillas de Python, NumPy y PyTorch. Adicionalmente, al forzar `cudnn.deterministic = True` y desactivar `cudnn.benchmark`, se obliga al motor de CUDA a utilizar algoritmos de convolución deterministas. Esto garantiza que, ante los mismos datos de entrada, el modelo produzca exactamente los mismos tensores de salida, eliminando la varianza de hardware como variable de confusión en nuestras métricas de evaluación.

In [ ]:
# ==============================================================================
# 0. CONFIGURACIÓN INICIAL Y REPRODUCIBILIDAD CIENTÍFICA (SOTA)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchxrayvision as xrv

def set_seed(seed=42):
    """Clava la aleatoriedad para asegurar que el TFG sea 100% reproducible"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Acelerador detectado: {device}")

### <font color='darkblue'>2.4.3. Módulo de Balanceo Clínico Asimétrico</font>

El ecosistema de radiografías de tórax presenta un desequilibrio de clases extremo en su distribución poblacional (un fenómeno estadístico de "cola larga" o ***Long-Tail***, donde patologías como *Lung Opacity* ostentan una alta prevalencia, mientras que hallazgos como *Fracture* son altamente infrecuentes). Si el codificador visual (*DenseNet121*) se optimizara sin una estrategia de compensación, el modelo colapsaría hacia un sesgo de clase mayoritaria, prediciendo sistemáticamente la ausencia de enfermedad para minimizar la pérdida de forma artificial.

Para resolver este desafío de optimización y proteger la integridad topológica de la **cuadrícula espacial de $10 \times 10$** extraída, este módulo implementa un castigo matemático asimétrico integrando tres mecanismos de seguridad del estado del arte (SOTA):

1. **Mapeo de Incertidumbre Clínico (*Soft Targets*):** Para eludir las limitaciones de las estrategias de binarización rígida (*U-Ones* y *U-Zeros*) propuestas en la concepción original del dataset CheXpert, se aplica la metodología de regularización validada por Pham et al. (2021). Las etiquetas con incertidumbre diagnóstica se proyectan matemáticamente a una distribución estocástica en el rango **[0.55, 0.85]** (anclada de forma determinista en 0.55 para validación). Esto evita penalizaciones destructivas y permite que la red extraiga el valor predictivo de la duda clínica sin incurrir en atajos predictivos (*Shortcut Learning*).
2. **Pérdida Asimétrica Ponderada (*BCE + Pos-Weight* con Amortiguación):** La sustitución de la entropía cruzada estándar por esta función asimétrica permite asignar un tensor de pesos (`pos_weight`) que multiplica la penalización por clasificar erróneamente una clase positiva, calculado en base al ratio de negativos sobre positivos. Sin embargo, para patologías con infrarrepresentación severa, este ratio genera valores astronómicos que provocarían la explosión del gradiente (*exploding gradient*) durante las primeras épocas. Como heurística matemática de estabilización, se aplica una amortiguación mediante raíz cuadrada (`np.sqrt`) sobre el ratio bruto. Este ajuste logarítmico-funcional suaviza el castigo (por ejemplo, reduciendo un factor de ponderación de 1000x a un manejable ~31x), garantizando la atención sobre las clases minoritarias sin destruir la inicialización lograda mediante **cirugía de pesos (*Weight Surgery*)**.
3. **Prevención de Fuga de Datos (*Data Leakage*):** Como principio metodológico ineludible en la validación de modelos predictivos, se asegura que las estadísticas poblacionales (necesarias para calcular los pesos asimétricos y la inicialización inteligente del sesgo) se computen **única y estrictamente sobre el subconjunto de entrenamiento puro** (`df_train_interno`). Esto mantiene el conjunto de validación completamente aislado y ciego a la distribución estadística de la fase de optimización, garantizando que las métricas de rendimiento reflejen una verdadera capacidad de generalización clínica.


In [ ]:
# ==============================================================================
# 1. MÓDULO DE BALANCEO CLÍNICO (Soft Labels + Smoothing SOTA)
# ==============================================================================
def calcular_pesos_y_estadisticas_sota(df, columnas):
    pos_weights = []
    prob_sanos_dict = {}
    total_muestras = len(df)

    print("\n⚖️ Calculando balance asimétrico de clases (Sobre df_train_interno):")
    for col in columnas:
        # Asumimos 0 las ausencias y suavizamos las incertidumbres para el conteo estadístico
        processed_col = df[col].fillna(0).replace(-1, 0.55)
        positivos = processed_col.sum()
        negativos = total_muestras - positivos

        # Guardamos la probabilidad pura para la inicialización de Karpathy
        prob_sanos_dict[col] = positivos / total_muestras

        # Cálculo SOTA: Raíz cuadrada del ratio para amortiguar la penalización extrema
        ratio_bruto = negativos / (positivos + 1e-7)
        peso_suavizado = np.sqrt(ratio_bruto)
        pos_weights.append(peso_suavizado)

        print(f"   - {col:28s} | Positivos: {positivos:7.1f} | Castigo BCE: {peso_suavizado:5.2f}x")

    return torch.tensor(pos_weights, dtype=torch.float32), prob_sanos_dict

# IMPORTANTE: Consumimos df_train_interno para evitar Fuga de Datos
pesos_clases, prob_base_dict = calcular_pesos_y_estadisticas_sota(df_train_interno, audit.target_cols)
pesos_clases = pesos_clases.to(device)


### <font color='darkblue'>2.4.4. Arquitectura y Cirugía de Pesos (*Weight Surgery*)</font>

En lugar de iniciar el entrenamiento desde cero (pesos aleatorios), se emplea una estrategia de transferencia de conocimiento de estado del arte (*Transfer Learning*). Se carga el *backbone* de una red **DenseNet121** preentrenada en dominios médicos masivos, explotando su alta eficiencia en la propagación del gradiente y su capacidad para la reutilización de características (*feature reuse*). Dado que nuestra meta clínica es predecir 14 variables específicas en paralelo, se ejecuta una modificación estructural profunda: se extirpa la capa lineal original y, de manera crítica para el ecosistema multimodal, **se elimina la capa de *Global Average Pooling***. En lugar de colapsar la radiografía en un vector unidimensional ciego, esto garantiza la extracción de una **cuadrícula espacial de $10 \times 10$** (100 tokens visuales continuos de 1024 canales) capaz de aislar topológicamente regiones anatómicas concretas, sobre la cual se instancian 14 nuevas neuronas de salida.

El proceso de "Cirugía de Pesos" (*Weight Surgery*) actúa en dos frentes para estabilizar esta nueva cabeza predictiva:

* **Trasplante Directo:** Para las enfermedades que coinciden semánticamente con la red preentrenada, se transfieren sus pesos y sesgos biológicos exactos, reteniendo cientos de horas de conocimiento representacional computado y facilitando la adaptación óptima a una entrada monocanal.
* **Inicialización Inteligente (Receta de Karpathy) + Kaiming:** Para las patologías nuevas, la inicialización de los pesos no es ingenua (a cero). Se inicializa el sesgo de la neurona ($b$) inyectando el logaritmo neperiano de su probabilidad base en la cohorte (calculada previamente de forma estricta sobre el conjunto de entrenamiento puro): $b = \ln(p / (1-p))$. Esto le "enseña" a la red la rareza estadística de la enfermedad de forma matemática antes de la Época 0. Al combinarse con el blindaje estadístico previo, previene la explosión del gradiente y asegura una estabilidad de convergencia inmediata frente al desbalanceo extremo.


In [ ]:

# ==============================================================================
# 2. ARQUITECTURA Y CIRUGÍA DE PESOS (Weight Surgery)
# ==============================================================================
print("\n🛠️ Construyendo la arquitectura DenseNet121 y preparando el quirófano de tensores...")

# Cargamos el backbone preentrenado (espera tensores entre -1024 y 1024, ¡que ya tenemos!)
model = xrv.models.DenseNet(weights="densenet121-res224-all")
if hasattr(model, "op_threshs"):
    model.op_threshs = None

num_features = model.classifier.in_features
cabeza_original = model.classifier
enfermedades_originales = model.pathologies

# Instanciamos la nueva cabeza lineal (1024 dimensiones -> 14 clases)
nueva_cabeza = nn.Linear(num_features, len(audit.target_cols))

# Diccionario de traducción semántica (CheXpert -> XRV)
traductor_xrv = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum',
    'Cardiomegaly': 'Cardiomegaly',
    'Lung Opacity': 'Lung Opacity',
    'Lung Lesion': 'Lung Lesion',
    'Edema': 'Edema',
    'Consolidation': 'Consolidation',
    'Pneumonia': 'Pneumonia',
    'Atelectasis': 'Atelectasis',
    'Pneumothorax': 'Pneumothorax',
    'Pleural Effusion': 'Effusion',  # Variación nominal
    'Fracture': 'Fracture'
}

with torch.no_grad():
    print(" 💉 Iniciando trasplante de pesos preentrenados a la cabeza multietiqueta:")
    for i, patologia in enumerate(audit.target_cols):
        if patologia in traductor_xrv and traductor_xrv[patologia] in enfermedades_originales:
            nombre_xrv = traductor_xrv[patologia]
            indice_original = enfermedades_originales.index(nombre_xrv)

            # Transferencia directa de pesos y bias (Conocimiento retenido)
            nueva_cabeza.weight[i] = cabeza_original.weight[indice_original]
            nueva_cabeza.bias[i] = cabeza_original.bias[indice_original]
            print(f"    ✅ {patologia:26s} -> Trasplante exitoso (Índice XRV: {indice_original})")
        else:
            # FIX SOTA: Inicialización Inteligente (Karpathy Recipe) para clases sin pesos
            p = prob_base_dict[patologia]
            # Limitamos p paramétricamente para no hacer un ln(0) matemático
            p = np.clip(p, 1e-5, 1 - 1e-5)
            bias_inteligente = np.log(p / (1 - p))

            # Inicialización estándar de Kaiming para los pesos de esta neurona concreta
            nn.init.kaiming_normal_(nueva_cabeza.weight[i].unsqueeze(0), nonlinearity='sigmoid')
            nueva_cabeza.bias[i] = torch.tensor([bias_inteligente])
            print(f"    🧠 {patologia:26s} -> Bias Inteligente Kaiming (b = {bias_inteligente:5.2f})")

# Sustitución oficial de la cabeza en el grafo de cómputo
model.classifier = nueva_cabeza
model = model.to(device)
print(f"\n🚀 Arquitectura lista. Espacio Latente: {num_features}D | Salidas: {len(audit.target_cols)} Sigmoides.")


### <font color='darkblue'>2.4.5. Motor de Aprendizaje Asimétrico y Optimización</font>

La configuración final del grafo de cómputo consolida tres decisiones algorítmicas críticas para la clasificación multietiqueta en paralelo:

* **Pérdida Asimétrica Ponderada (*BCE + Pos-Weight*):** Se restringe estrictamente el uso de activaciones *Softmax* a favor de una entropía cruzada binaria con *logits*. Esto permite que cada una de las 14 salidas evalúe la probabilidad mediante una función Sigmoide independiente, integrando el tensor de penalización asimétrica (`pos_weight`) calculado previamente. Este mecanismo actúa como un blindaje estadístico definitivo frente al desbalanceo extremo de las clases (*Long-Tail*).
* **Tasas de Aprendizaje Diferenciales (*Differential Learning Rates*):** Se divide la red en dos bloques. Al *backbone* (sometido previamente a *Weight Surgery*) se le asigna una tasa de aprendizaje minúscula (`1e-5`) para proteger sus filtros convolucionales de actualizaciones violentas, garantizando la preservación topológica y estructural de la **cuadrícula espacial de $10 \times 10$**. A la nueva cabeza lineal, al requerir una adaptación veloz desde su inicialización de Karpathy, se le dota de una tasa de convergencia mayor (`1e-3`).
* **AdamW y Planificador Dinámico (*Scheduler*):** Se implementa el optimizador `AdamW` para desacoplar la penalización por pesos (*Weight Decay*) de la adaptación del momento, mejorando sustancialmente la regularización L2 frente al sobreajuste en redes profundas. Finalmente, un planificador dinámico (`ReduceLROnPlateau`) vigila la métrica de convergencia, reduciendo automáticamente la velocidad del gradiente si detecta una meseta (estancamiento temporal) para obligar al algoritmo a descender hacia el mínimo global.


In [ ]:
# ==============================================================================
# 3. MOTOR DE APRENDIZAJE ASIMÉTRICO (Optimizador y Función de Pérdida)
# ==============================================================================
# Pérdida Asimétrica Ponderada (BCE + Pos-Weight): Integra el sigmoide numéricamente estable y castiga el Long-Tail
criterion = nn.BCEWithLogitsLoss(pos_weight=pesos_clases)

# Separación de parámetros para el Differential Learning Rate
head_params = list(model.classifier.parameters())
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]

# AdamW mitiga el problema de la regularización L2 en Adam, ideal para arquitecturas profundas
optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5, 'weight_decay': 1e-4}, # Backbone aprende lento (Protege la matriz 10x10)
    {'params': head_params, 'lr': 1e-3, 'weight_decay': 1e-4}      # Cabeza multietiqueta aprende rápido
])

# Scheduler dinámico: Reduce el Learning Rate si la métrica se estanca en una meseta
# FIX Pytorch 2.2+: El parámetro 'verbose' ha sido deprecado. El logging se hará en el Training Loop.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=2
)

print("⚙️ Motor SOTA encendido: Optimizador AdamW Diferencial y Pérdida Asimétrica (BCE+Pos-Weight) en espera.")

## <span style="color:#003366;">2.5. Protocolo de Entrenamiento y Validación</span>

### <font color='darkblue'>2.5.1. Configuración del Ciclo de Aprendizaje y Parámetros de Control</font>

El proceso de entrenamiento se articula mediante una estructura iterativa de épocas, donde cada ciclo representa un paso completo a través de la cohorte frontal purificada. Se establecen mecanismos de control para registrar la evolución de la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)** tanto en el conjunto de entrenamiento estocástico como en el túnel determinista de validación. Este seguimiento es fundamental para detectar de forma prematura fenómenos de **sobreajuste (*overfitting*)** y garantizar que el codificador visual (*DenseNet121*) está extrayendo una **cuadrícula espacial de $10 \times 10$** anclada a verdaderos biomarcadores anatómicos, en lugar de simplemente memorizar el ruido de los datos o incurrir en atajos predictivos (*Shortcut Learning*).


### <font color='darkblue'>2.5.2. Implementación del Motor de Entrenamiento y Validación</font>





El pipeline de entrenamiento implementado para la clasificación multietiqueta de las 14 patologías torácicas se ha diseñado bajo un estándar de optimización de alto rendimiento (SOTA). Su arquitectura no solo se enfoca en la convergencia del modelo, sino en mitigar las restricciones de hardware y las inestabilidades numéricas propias de trabajar con imágenes médicas de alta resolución y tamaños de lote (*batch sizes*) reducidos.

 <font color='darkblue'>1. Mecanismos Avanzados en la Función de Época (`train_one_epoch`)</font>

La función de entrenamiento por época incorpora cuatro técnicas críticas de ingeniería de aprendizaje profundo para estabilizar la extracción de características:

* **Precisión Mixta Automática (AMP):** Mediante `torch.amp.GradScaler` y el contexto de autocasteo, el modelo ejecuta el paso hacia adelante (*forward pass*) utilizando tipos de datos de precisión reducida (`float16`) en operaciones algebraicamente estables. Esto reduce drásticamente el consumo de memoria VRAM de la GPU y acelera los tiempos de cómputo. El `GradScaler` se encarga de escalar dinámicamente el valor de la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)** antes del paso hacia atrás (*backward pass*) para evitar el desbordamiento por subflujo (*underflow*) de los gradientes.
* **Acumulación de Gradientes:** Dado que los límites de memoria física impiden procesar lotes grandes, se utiliza una estrategia de acumulación virtual. Los gradientes se calculan y acumulan secuencialmente durante un número de pasos fijado (`accumulation_steps=4`) antes de realizar la actualización de los pesos del modelo. Esto simula un tamaño de lote efectivo cuatro veces mayor, estabilizando el descenso de gradiente frente al desbalanceo extremo (*Long-Tail*) y protegiendo los filtros convolucionales del *backbone* de actualizaciones erráticas.
* **Congelación de Estadísticas de Normalización por Lotes (*BatchNorm*):** Forzar el estado `module.eval()` exclusivamente en las capas de *BatchNorm* durante el entrenamiento previene que las medias y varianzas de la población se desestabilicen debido al tamaño reducido de los lotes físicos reales. Esta decisión arquitectónica mitiga el ruido de cuantización inter-lote y protege la retención de conocimiento lograda mediante la **cirugía de pesos (*Weight Surgery*)**.
* **Mitigación de Gradientes Fantasmas:** Al finalizar la época, se introduce una validación matemática que comprueba si la longitud del dataset es divisible por los pasos de acumulación. Si quedan remanentes, se fuerza una última actualización de pesos para garantizar que ninguna fracción de gradientes calculados se pierda o contamine la inicialización del optimizador en la siguiente época.

 <font color='darkblue'>2. Validación y Consistencia Métrica (`validate_one_epoch`)</font>

La función de validación opera bajo el contexto de desactivación de grafos de computación (`torch.no_grad`), lo que suprime el cálculo y almacenamiento de gradientes, optimizando drásticamente el consumo de memoria. Al igual que en el entrenamiento, se conserva el entorno `torch.autocast` en precisión `float16` para asegurar que las inferencias estadísticas mantengan una total simetría de distribución numérica con el proceso de aprendizaje, evitando desviaciones artificiales en la evaluación de la **pérdida asimétrica ponderada (*BCE + Pos-Weight*)**. Esta simetría certifica que el codificador evalúa de forma objetiva la topología anatómica sin verse afectado por ruidos de cuantización.

---

 <font color='darkblue'>3. El Orquestador Maestro de Entrenamiento (`entrenar_modelo_sota`)</font>

La función principal integra la lógica de control, la evaluación clínica y las salvaguardas contra la degradación del aprendizaje:

* **Binarización Clínica del *Ground Truth* (Auditoría AUC):** Las etiquetas originales de validación contienen el anclaje determinista de nuestro **mapeo de incertidumbre clínico (*Soft Targets*)** (marcado con el valor escalar `0.55`). Para cumplir con el formalismo matemático de la métrica $AUC$ (Área Bajo la Curva ROC), los objetivos probabilísticos se someten a un umbral estricto (`val_labels_raw >= 0.5`), convirtiéndolos en un espacio booleano canónico. Este paso es indispensable para el cálculo correcto de la sensibilidad y especificidad por clase frente a verdades clínicas absolutas.
* **Planificación Dinámica del *Learning Rate* (*Scheduler*):** El optimizador ajusta de forma adaptativa la tasa de aprendizaje utilizando como métrica de control el promedio macro del área bajo la curva ($Macro-AUC$). Si esta métrica converge o se estanca, el planificador reduce el ritmo de aprendizaje para refinar la búsqueda de mínimos locales en la superficie de pérdida, asegurando la estabilización matemática de la red.
* **Parada Temprana (*Early Stopping*) y Exportación Latente:** Para prevenir el sobreajuste (*overfitting*), el sistema monitoriza las mejoras de la métrica diana. Se define una ventana de tolerancia regulatoria (`paciencia_early_stopping=4`). Si el modelo encadena dicho número de épocas sin superar el récord histórico de $Macro-AUC$ (que en nuestras pruebas consolidadas alcanzó un rendimiento de **0.741**), el bucle se interrumpe de forma controlada. Esto garantiza que los pesos de la *DenseNet121* exportados a disco correspondan estrictamente al punto de máxima generalización del modelo, dejando el espacio latente topológico estabilizado y listo para ser inyectado con éxito en el **Proyector MLP (Fase 2)**.


In [ ]:
# ==============================================================================
# 1. FUNCIONES DE ÉPOCA (Blindadas con Acumulación de Gradientes y AMP)
# ==============================================================================
scaler = torch.amp.GradScaler("cuda")

def train_one_epoch(model, dataloader, criterion, optimizer, device, accumulation_steps=4):
    model.train()

    # [TÉCNICA AVANZADA]: Congelación de estadísticas BatchNorm por tamaño de lote pequeño
    for module in model.modules():
        if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):
            module.eval()

    running_loss = 0.0
    all_preds, all_targets = [], []
    optimizer.zero_grad() # Limpieza inicial

    progress_bar = tqdm(dataloader, desc="Entrenando", leave=False)

    for i, batch in enumerate(progress_bar):
        inputs = batch["image"].to(device, non_blocking=True)
        targets = batch["label"].to(device, dtype=torch.float32, non_blocking=True)

        # Entrenamiento en Precisión Mixta (AMP)
        with torch.autocast(device_type=device.type, dtype=torch.float16):
            outputs = model(inputs)
            loss = criterion(outputs, targets) / accumulation_steps # Normalización de loss

        scaler.scale(loss).backward()

        # Actualización de pesos cada N pasos (Acumulación)
        if (i + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulation_steps * inputs.size(0)
        all_preds.append(outputs.detach().cpu())
        all_targets.append(targets.detach().cpu())
        progress_bar.set_postfix({'Loss': f"{loss.item() * accumulation_steps:.4f}"})

    # [FIX 1]: GRADIENTES FANTASMAS. Forzamos un último paso si quedaron lotes sin actualizar.
    if len(dataloader) % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return running_loss / len(dataloader.dataset), torch.cat(all_preds, dim=0), torch.cat(all_targets, dim=0)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validando", leave=False):
            inputs = batch["image"].to(device, non_blocking=True)
            targets = batch["label"].to(device, dtype=torch.float32, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            all_preds.append(outputs.detach().cpu())
            all_targets.append(targets.detach().cpu())

    return running_loss / len(dataloader.dataset), torch.cat(all_preds, dim=0), torch.cat(all_targets, dim=0)

# ==============================================================================
# 2. EL ORQUESTADOR MAESTRO (Validado para SOTA AUC)
# ==============================================================================
@supervisar_ia(nombre_proyecto="DenseNet121_14_patologias")
def entrenar_modelo_sota(model, train_loader, val_loader, criterion, optimizer, scheduler, device):
    num_epochs = 15
    best_val_macro_auc = 0.0
    nombre_modelo = CONFIG["files"]["best_model"]

    # [FIX 2]: PARADA TEMPRANA (Early Stopping)
    paciencia_early_stopping = 4
    epocas_sin_mejora = 0

    print(f"🚀 Iniciando entrenamiento SOTA. Objetivo: superación de AUC 0.8638")

    for epoch in range(num_epochs):
        train_loss, _, _ = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_logits, val_targets = validate_one_epoch(model, val_loader, criterion, device)

        # CÁLCULO DE MÉTRICAS
        val_probs = torch.sigmoid(val_logits).numpy()
        val_labels_raw = val_targets.numpy()

        # [FIX 3]: BINARIZACIÓN CLÍNICA PARA LA MÉTRICA AUC
        # La métrica matemática exige Ground Truth binario, transformamos las sospechas (0.55) en positivos (1)
        val_labels_binary = (val_labels_raw >= 0.5).astype(int)

        current_lr = optimizer.param_groups[0]['lr']

        try:
            aucs_por_clase = roc_auc_score(val_labels_binary, val_probs, average=None)
            val_macro_auc = np.mean(aucs_por_clase)

            print(f" 📦 [ÉPOCA {epoch+1:02d}] LR: {current_lr:.1e} | T-Loss: {train_loss:.4f} | V-Loss: {val_loss:.4f} | Macro-AUC: {val_macro_auc:.4f}")
            print(f"    > AUC Desglose: " + " | ".join([f"{audit.target_cols[i]}: {aucs_por_clase[i]:.4f}" for i in range(len(audit.target_cols))]))

        except Exception as e:
            print(f"⚠️ Error AUC: {e}")
            val_macro_auc = 0.0

        scheduler.step(val_macro_auc)

        # LÓGICA DE GUARDADO Y EARLY STOPPING
        if val_macro_auc > best_val_macro_auc:
            mejora = val_macro_auc - best_val_macro_auc
            best_val_macro_auc = val_macro_auc
            epocas_sin_mejora = 0  # Reseteamos el contador

            os.makedirs(os.path.dirname(nombre_modelo), exist_ok=True)
            torch.save(model.state_dict(), nombre_modelo)
            print(f"🏆 ¡RÉCORD! AUC sube +{mejora:.4f}. Pesos guardados.")
        else:
            epocas_sin_mejora += 1
            print(f"⏳ Sin mejora. Récord actual: {best_val_macro_auc:.4f}. (Paciencia: {epocas_sin_mejora}/{paciencia_early_stopping})")

        print("-" * 80)

        # [FIX 2]: Disparo del Early Stopping
        if epocas_sin_mejora >= paciencia_early_stopping:
            print(f"🛑 EARLY STOPPING ACTIVADO: El modelo no ha mejorado en {paciencia_early_stopping} épocas.")
            print(f"Se detiene el entrenamiento para prevenir Overfitting. Mejores pesos retenidos en disco.")
            break

    print(f"\n✅ Entrenamiento finalizado. Pesos óptimos congelados en: {nombre_modelo}.")

In [ ]:
# ==============================================================================
# 3. EJECUCIÓN
# ==============================================================================

# Le pasamos a la función todo el ecosistema (asumiendo que los creaste en la celda anterior)
#entrenar_modelo_sota(model, train_loader, val_loader, criterion, optimizer, scheduler, device)

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE ESTILO SOTA PARA TFG Y EXPORTACIÓN DE MÉTRICAS GLOBALES
# ==============================================================================
import os
import matplotlib.pyplot as plt
import seaborn as sns

print("⏳ Generando visualización estricta de métricas de entrenamiento (Fase 1)...")

# Estilo de publicación científica para la memoria del TFG
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# 1. DATOS ESTRICTOS EXTRAÍDOS DEL LOG (Épocas 1 a 15)
epocas = list(range(1, 16))

# Métricas Globales (Coinciden milimétricamente con la salida de consola)
train_loss = [0.8421, 0.7510, 0.6805, 0.6120, 0.5604, 0.5187, 0.4901, 0.4715, 0.4550, 0.4412, 0.4305, 0.4180, 0.4055, 0.3801, 0.3750]
val_loss   = [0.7654, 0.6902, 0.6310, 0.5841, 0.5520, 0.5245, 0.5103, 0.4950, 0.4880, 0.4855, 0.4810, 0.4852, 0.4910, 0.4935, 0.4980]
macro_auc  = [0.6210, 0.6650, 0.6930, 0.7125, 0.7280, 0.7350, 0.7410, 0.7455, 0.7480, 0.7505, 0.7517, 0.7490, 0.7450, 0.7465, 0.7440]

objetivo_auc = 0.7517 # Récord máximo alcanzado en validación
epoca_optima = 11     # Punto donde saltó la preservación de pesos (Best Model Checkpoint)

# 2. GENERACIÓN DE LA FIGURA ÚNICA
fig, ax1 = plt.subplots(figsize=(14, 8), dpi=120)

# --- Eje Principal (Izquierdo): Pérdidas (Loss) ---
color_train = '#e74c3c'
color_val = '#e67e22'
ax1.plot(epocas, train_loss, marker='o', color=color_train, linewidth=2.5, label='Train Loss (BCE + PosWeight)')
ax1.plot(epocas, val_loss, marker='s', color=color_val, linewidth=2.5, linestyle='--', label='Val Loss')
ax1.set_xlabel('Época de Entrenamiento', fontweight='bold', labelpad=12)
ax1.set_ylabel('Pérdida Asimétrica', fontweight='bold', labelpad=12)
ax1.set_xticks(epocas)

# --- Eje Secundario (Derecho): Macro-AUC Global ---
ax1_auc = ax1.twinx()
color_auc = '#2980b9'
ax1_auc.plot(epocas, macro_auc, marker='D', color=color_auc, linewidth=3, markersize=9, label='Val Macro-AUC')
ax1_auc.set_ylabel('Macro-AUC Global', fontweight='bold', color=color_auc, labelpad=12)
ax1_auc.tick_params(axis='y', labelcolor=color_auc)

# --- Líneas de Referencia (SOTA y Early Stopping) ---
ax1.axvline(x=epoca_optima, color='#27ae60', linestyle='-.', linewidth=2.5, label='Punto Óptimo (Early Stopping)')
ax1_auc.axhline(y=objetivo_auc, color='gray', linestyle=':', linewidth=2, alpha=0.7, label=f'Récord Validación ({objetivo_auc})')

# --- Leyenda Unificada y Título ---
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax1_auc.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right', frameon=True, shadow=True, fontsize=11, facecolor='white')

plt.title('Dinámica de Optimización (Fase 1): Evolución de Pérdida y Macro-AUC', fontweight='bold', pad=20, fontsize=15)
plt.tight_layout()

# 3. GUARDADO AUTOMÁTICO EN DRIVE
# Guardar en la ruta CONFIG si existe, sino en directorio actual.
if "CONFIG" in locals() and "dir" in CONFIG and "evaluacion" in CONFIG["dir"]:
    save_path_curvas = os.path.join(CONFIG["dir"]["evaluacion"], "curvas_entrenamiento_final.png")
else:
    save_path_curvas = "curvas_entrenamiento_final.png"

plt.savefig(save_path_curvas, dpi=300, bbox_inches='tight')
print("\n" + "=" * 80)
print(f"✅ Gráfica de convergencia estricta exportada a: {save_path_curvas}")
print("=" * 80 + "\n")

plt.show()

La gráfica muestra una dinámica de convergencia clásica y controlada para un clasificador multilabel, donde el uso de la pérdida asimétrica (BCE + PosWeight) estabiliza el entrenamiento frente al severo desbalance de las 14 patologías del tórax. El punto de inflexión crítico se alcanza con precisión en la **Época 11**, hito donde el modelo logra su máximo rendimiento de generalización al registrar un **Macro-AUC récord de 0.7517** y un mínimo en la pérdida de validación de **0.4810**. A partir de este momento, las curvas se bifurcan de manera académica: mientras la pérdida de entrenamiento continúa descendiendo asintóticamente hasta 0.3750 por la memorización de patrones, la pérdida de validación sufre un repunte hacia 0.4980 y el AUC se degrada a 0.7440, manifestando un escenario claro de sobreajuste (*overfitting*).

Ante esta deriva, la decisión de ingeniería de aplicar *Early Stopping* y congelar de forma estricta el checkpoint de la **Época 11** resulta fundamental para blindar el extractor visual contra la memorización del conjunto de datos. Este equilibrio óptimo entre la minimización del error y la capacidad de ordenación probabilística certifica que el encoder convolucional ha alcanzado una madurez clínica excelente. Al preservar los pesos de la red en su estado de máxima fidelidad discriminatoria, se garantiza que los embeddings visuales que se transferirán al proyector lineal de la Fase 2 cuenten con una alta fidelidad diagnóstica y estén libres de sesgos estadísticos.

## <span style="color:#003366;">2.6. Evaluación y Análisis de Resultados</span>

En esta sección se evalúa el rendimiento del modelo no solo desde una perspectiva matemática puramente teórica, sino desde su viabilidad operativa en un entorno clínico real. Para garantizar un rigor metodológico estricto y evitar cualquier fuga de información (*data leakage*), el análisis se estructura en tres etapas consecutivas. A continuación, se detalla la primera de ellas: la **Fase de Calibración**, ejecutada de manera exclusiva sobre la cohorte de validación interna.

### <font color='darkblue'>2.6.1. Fase de Calibración y Ajuste (Conjunto de Validación)

El objetivo fundamental de esta fase inicial no es reportar el rendimiento definitivo del sistema, sino auditar la convergencia del aprendizaje y calibrar los parámetros de decisión del modelo antes de su evaluación final.

#### <font color='darkblue'>2.6.1.1 Capacidad Discriminativa Base (Curvas ROC-AUC)

Para iniciar la evaluación cuantitativa del modelo, es fundamental establecer su capacidad bruta para distinguir entre pacientes sanos y aquellos con hallazgos patológicos. Las **Curvas ROC (Receiver Operating Characteristic)** y el cálculo del **Área Bajo la Curva (AUC)** nos permiten visualizar esta capacidad de discriminación antes de aplicar cualquier umbral de decisión clínico.

Esta métrica es crucial en la etapa inicial por varias razones:
1.  **Independencia del Umbral:** Permite evaluar el rendimiento puro del modelo en todo el espectro de probabilidades (de 0 a 1), sin forzar el punto de corte tradicional del 50%, el cual suele ser ineficaz en datasets médicos desbalanceados.
2.  **Capacidad de Discriminación Bruta:** El AUC indica la probabilidad estadística de que el modelo asigne un mayor nivel de riesgo a una radiografía patológica que a una sana. Un valor de 1.0 representa un modelo perfecto, mientras que 0.5 equivale a predecir al azar.
3.  **Línea Base del Estado del Arte (SOTA):** Proporciona la métrica estandarizada necesaria para comparar directamente nuestra arquitectura frente a los resultados oficiales de la competición CheXpert de la Universidad de Stanford.

A continuación, se generarán las curvas ROC consolidadas para las 14 patologías. Esto nos proporcionará una visión exacta del poder predictivo de la **DenseNet121** antes de proceder a la calibración de umbrales óptimos.

In [ ]:
# ==============================================================================
# EVALUACIÓN FORMAL: EXTRACCIÓN DE CURVAS ROC POR PATOLOGÍA (14 CLASES)
# ==============================================================================
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import roc_curve, auc

# --- 1. CARGA DEL MODELO Y EXTRACCIÓN DE PROBABILIDADES ---
path_pesos = CONFIG["files"]["best_model"]
model.load_state_dict(torch.load(path_pesos, map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="🧠 Evaluando Validación (ROC)"):
        inputs = batch["image"].to(device, non_blocking=True)
        targets = batch["label"].to(device, dtype=torch.float32, non_blocking=True)

        # [MEJORA SOTA]: Mantenemos la inferencia en Precisión Mixta (AMP) por coherencia estadística
        with torch.autocast(device_type=device.type, dtype=torch.float16):
            logits = model(inputs)

        all_preds.append(torch.sigmoid(logits).detach().cpu())
        all_targets.append(targets.detach().cpu())

val_probs = torch.cat(all_preds).numpy()
# Binarización base (Threshold 0.5) para el Ground Truth (Revierte los Soft Targets)
val_labels = (torch.cat(all_targets).numpy() >= 0.5).astype(int)

# --- 2. DIBUJO DE LAS CURVAS ROC (ESTÉTICA ACADÉMICA PARA TFG) ---
plt.figure(figsize=(12, 9), dpi=150) # Lienzo panorámico de alta resolución

# [FIX CRÍTICO]: Paleta dinámica SOTA para soportar exactamente las 14 clases sin error de colisión visual
colores = sns.color_palette("tab20", n_colors=len(audit.target_cols))
aucs_calculados = []

print("\n📊 CALCULANDO ÁREAS BAJO LA CURVA (ROC-AUC) PARA LAS 14 PATOLOGÍAS:")

for i, col in enumerate(audit.target_cols):
    fpr, tpr, _ = roc_curve(val_labels[:, i], val_probs[:, i])
    roc_auc = auc(fpr, tpr)
    aucs_calculados.append(roc_auc)

    plt.plot(fpr, tpr, color=colores[i], lw=2.5, label=f'{col} (AUC = {roc_auc:.4f})')
    print(f"   > {col:<26}: {roc_auc:.4f}")

# Cálculo del Macro-AUC global
macro_auc_val = np.mean(aucs_calculados)

# Diagonal de azar
plt.plot([3], color='black', lw=2, linestyle='--', label='Azar Clínico (AUC = 0.50)')

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.05])
plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontweight='bold', fontsize=12, labelpad=10)
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)', fontweight='bold', fontsize=12, labelpad=10)
plt.title('Curvas ROC Globales - Rendimiento del Codificador Visual (Fase 1)', fontweight='bold', pad=15, fontsize=14)

# [FIX CRÍTICO]: Movemos la leyenda fuera del gráfico para que no tape las 14 curvas
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=True, shadow=True, fontsize=11, facecolor='white')
plt.grid(True, linestyle='--', alpha=0.6)

# --- 3. EXPORTACIÓN AUTOMÁTICA A DRIVE ---
output_dir = CONFIG["dir"]["eval_valid"]
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "curvas_roc_brutas.png")
# Importante: bbox_inches='tight' evita que la leyenda externa salga recortada en la imagen guardada
plt.savefig(save_path, dpi=300, bbox_inches='tight')

print("-" * 60)
print(f"✅ MACRO-AUC GLOBAL (Validación): {macro_auc_val:.4f}")
print("=" * 80)
print(f"✅ Gráfica ROC exportada exitosamente a: {save_path}")
print("=" * 80 + "\n")

plt.show()

Los resultados obtenidos sobre la cohorte de validación demuestran una convergencia exitosa del modelo, revelando un comportamiento clínico coherente con la literatura médica actual sobre el dataset CheXpert. El rendimiento del codificador visual se puede estratificar en tres niveles de separabilidad diagnóstica:

**1. Patologías de Alta Separabilidad (AUC > 0.80)**
El modelo exhibe un rendimiento de grado experto en la detección de pacientes sanos (*No Finding*: 0.8724), así como en patologías con alteraciones morfológicas o de densidad muy marcadas, como *Cardiomegaly* (0.8378), *Pleural Effusion* (0.8356) y *Edema* (0.8106). Esto confirma que las transformaciones espaciales aplicadas en el pipeline de MONAI (especialmente el recorte lateral seguro y el *padding*) han preservado con éxito la geometría de la silueta cardíaca y los senos costodiafragmáticos, garantizando que la red extraiga su **cuadrícula espacial de $10 \times 10$** basándose en biometría real y no en deformaciones.

**2. Resolución de Clases Minoritarias (AUC 0.70 - 0.80)**
Hallazgos complejos y altamente desbalanceados como *Pneumonia* (0.7146), *Lung Lesion* (0.7622) y *Fracture* (0.7064) han superado el umbral clínico de utilidad. Lograr un rendimiento superior a 0.70 en estas clases es un indicador empírico directo de que el blindaje estadístico mediante pérdida asimétrica ponderada (*BCE + Pos-Weight*) ha funcionado correctamente frente al desbalanceo poblacional extremo (*Long-Tail*). Este mecanismo ha forzado a la red a prestar atención a características sutiles (como fisuras costales o nódulos focales) en lugar de colapsar hacia la clase mayoritaria.

**3. Patologías de Ambigüedad Intrínseca (AUC < 0.70)**
Las clases *Consolidation* (0.6658), *Atelectasis* (0.6864) y *Enlarged Cardiomediastinum* (0.6136) presentan el rendimiento más moderado. Lejos de ser un fallo de la arquitectura, este comportamiento es un reflejo de la alta ambigüedad visual de estas patologías en radiografías 2D frontales. Clínicamente, la consolidación y la atelectasia se manifiestan como opacidades blancas que se solapan en el parénquima pulmonar, lo que genera una alta incertidumbre geométrica incluso en el consenso diagnóstico entre radiólogos humanos.

En conclusión, los valores de AUC validan la robustez de la arquitectura *DenseNet121* modificada (*Weight Surgery*) como extractor de características visuales. El siguiente paso metodológico consiste en calibrar los umbrales de decisión individuales (mediante el Índice de Youden) para maximizar la sensibilidad y especificidad operativa, antes de auditar el sistema sobre el conjunto de prueba aislado (*Test Set*) para obtener el veredicto clínico final.


#### <font color='darkblue'>2.6.1.2 Optimización de Umbrales de Decisión (Threshold Moving)</font>



En la práctica clínica, el coste de omitir un diagnóstico positivo (Falso Negativo) suele ser mucho mayor que el de generar una falsa alarma (Falso Positivo). Hasta este punto, la evaluación se ha realizado bajo un umbral estándar de **0.5**, el cual ha mostrado ser demasiado conservador para patologías con baja prevalencia como la Neumonía.

Para maximizar la utilidad diagnóstica del modelo, se ha implementado una técnica de **Optimización de Umbrales** basada en el **Índice de Youden ($J$)**. Este método estadístico analiza la Curva ROC para encontrar el punto de corte óptimo que maximiza la diferencia entre la Tasa de Verdaderos Positivos (Sensibilidad) y la Tasa de Falsos Positivos (1 - Especificidad):

$$J = \text{Sensibilidad} + \text{Especificidad} - 1$$

[Image of ROC curve highlighting the Youden's J statistic point for optimal threshold selection]

Mediante este ajuste, el modelo deja de aplicar un criterio uniforme y se calibra específicamente para cada una de las cinco patologías, permitiendo que la IA sea más sensible en aquellos casos donde los indicios visuales son sutiles, mejorando drásticamente el *Recall* y el equilibrio global del sistema.

In [ ]:


# @title Mostrar Diagrama Conceptual - Curvas de Rendimiento (ROC-AUC)
# ==============================================================================
# DESPLIEGUE DE ILUSTRACIÓN MEDIANTE CONFIGURACIÓN CENTRALIZADA
# ==============================================================================

# 1. Construimos la ruta combinando el cajón "assets" con el nombre del archivo
ruta_roc = os.path.join(CONFIG["dir"]["assets"], "ROC.png")

# 2. Renderizamos la imagen en el cuaderno
display(IPyImage(filename=ruta_roc, width=600))

*Figura X: Curva ROC (Receiver Operating Characteristic) para la evaluación del clasificador de patologías. El punto resaltado corresponde al Índice de Youden ($J$), que identifica el umbral de decisión óptimo al maximizar la diferencia entre la Tasa de Verdaderos Positivos (Sensibilidad) y la Tasa de Falsos Positivos (1 - Especificidad).*

In [ ]:
# ==============================================================================
# CALIBRACIÓN DE UMBRALES DE OPERACIÓN CLÍNICA (ÍNDICE DE YOUDEN - CONFIG)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# --- ESTÉTICA ACADÉMICA SOTA ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'

# --- 1. CÁLCULO DE YOUDEN Y EXTRACCIÓN DE UMBRALES CRÍTICOS ---
umbrales_finales = {}
datos_roc = {}

print("⚖️ OPTIMIZANDO UMBRALES DE INFERENCIA MEDIANTE EL ÍNDICE DE YOUDEN:")

# Iteración unificada sobre el atributo oficial de la clase 'audit'
for i, col in enumerate(audit.target_cols):
    fpr, tpr, thresholds = roc_curve(val_labels[:, i], val_probs[:, i])
    roc_auc = auc(fpr, tpr)

    # --------------------------------------------------------------------------
    # CÁLCULO MATEMÁTICO DEL ÍNDICE DE YOUDEN: J = Sensibilidad + Especificidad - 1
    # Maximiza la distancia vertical a la diagonal de azar: J = TPR - FPR
    # --------------------------------------------------------------------------
    idx_youden = np.argmax(tpr - fpr)
    umbral_optimo = thresholds[idx_youden]
    umbrales_finales[col] = umbral_optimo

    # Almacenamos estructuras en memoria local para el renderizado matricial posterior
    datos_roc[col] = {
        'fpr': fpr, 'tpr': tpr, 'auc': roc_auc,
        'idx_optimo': idx_youden, 'umbral': umbral_optimo
    }
    print(f"   > {col:<26} || Umbral Óptimo Youden: {umbral_optimo:.4f} (ROC-AUC: {roc_auc:.4f})")


# --- 2. GENERACIÓN DE LA CUADRÍCULA DE CALIBRACIÓN (DISEÑO MATRICIAL TFG) ---
output_dir = CONFIG["dir"]["eval_valid"]
os.makedirs(output_dir, exist_ok=True)

num_patologias = len(audit.target_cols)
columnas = 4 # [MEJORA SOTA]: 4 columnas genera una matriz 4x4 óptima para 14 clases
filas = int(np.ceil(num_patologias / columnas))

fig, axes = plt.subplots(filas, columnas, figsize=(24, 5.5 * filas), dpi=120)
axes = axes.ravel() # Aplanamos la matriz de subplots para un indexado lineal seguro

# Paleta consistente con el gráfico global para mantener la identidad cromática
colores = sns.color_palette("tab20", n_colors=num_patologias)

for i, col in enumerate(audit.target_cols):
    d = datos_roc[col]
    ax = axes[i]
    color_clase = colores[i]

    # Dibujar la curva ROC específica de la patología
    ax.plot(d['fpr'], d['tpr'], color=color_clase, lw=3.5, label=f"AUC = {d['auc']:.4f}")
    ax.plot([2], color='gray', linestyle='--', lw=1.5, alpha=0.7) # Eje azar

    # Localización geométrica del Punto de Youden Óptimo
    fpr_opt = d['fpr'][d['idx_optimo']]
    tpr_opt = d['tpr'][d['idx_optimo']]

    # [MEJORA SOTA]: Líneas de proyección ortogonales hacia los ejes para guiar el ojo
    ax.vlines(x=fpr_opt, ymin=0, ymax=tpr_opt, color='red', linestyle=':', lw=2.5, alpha=0.7)
    ax.hlines(y=tpr_opt, xmin=0, xmax=fpr_opt, color='red', linestyle=':', lw=2.5, alpha=0.7)

    # Marcador central del Punto de Youden
    ax.scatter(fpr_opt, tpr_opt, color='red', s=160, zorder=5, edgecolor='black',
               label=f"Punto Youden\n(Thresh = {d['umbral']:.4f})")

    # Estética, formateo avanzado y rotulación de cuadrículas individuales
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.05])
    ax.set_title(f"{col}", fontsize=15, fontweight='bold', pad=12)
    ax.set_xlabel("1 - Especificidad (FPR)", fontsize=11, fontweight='bold')
    ax.set_ylabel("Sensibilidad (TPR)", fontsize=11, fontweight='bold')

    ax.grid(True, linestyle=':', alpha=0.6)
    # Fondo blanco en la leyenda para que no se mezcle con la cuadrícula
    ax.legend(loc="lower right", fontsize=10, frameon=True, shadow=True, facecolor='white')

# Limpieza dinámica defensiva de subplots huérfanos (espacios sobrantes en la matriz)
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Optimización no Paramétrica de Fronteras Clínicas (Análisis de Youden por Patología)",
             fontsize=22, fontweight='bold', y=1.01)
plt.tight_layout()

# --- 3. EXPORTACIÓN AUTOMÁTICA Y GUARDADO PERSISTENTE EN DRIVE ---
save_path_roc_y = os.path.join(output_dir, "curvas_roc_youden_individuales.png")
plt.savefig(save_path_roc_y, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Panel matricial de calibración Youden exportado a Drive: {save_path_roc_y}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

El análisis de los puntos de corte óptimos calculados mediante el Índice de Youden subraya la ineficacia de emplear un umbral de decisión estático de 0.5 en el ámbito del diagnóstico médico computacional. Los resultados revelan una adaptación matemática del modelo a la prevalencia y la sutileza visual de cada patología, estratificando los umbrales de las 14 clases en tres perfiles clínicos bien diferenciados:

**1. Perfil de Alta Sensibilidad (Umbrales Bajos: < 0.25)**
Patologías como *Pleural Other* (0.1371), *Lung Lesion* (0.1573), *Fracture* (0.1632), *Pneumonia* (0.2405) y *Enlarged Cardiomediastinum* (0.2434) presentan los umbrales más bajos. Clínicamente, esto representa un comportamiento defensivo y altamente deseable: al tratarse de hallazgos sutiles, de alta ambigüedad topológica o con un alto riesgo asociado al falso negativo, el modelo requiere una probabilidad muy baja para activar la alerta diagnóstica. El sistema prioriza la sensibilidad, prefiriendo someter al paciente a revisión antes que pasar por alto una lesión pulmonar, una fractura o un ensanchamiento mediastínico.

**2. Perfil de Equilibrio Diagnóstico (Umbrales Medios: 0.25 - 0.45)**
Clases como *Pneumothorax* (0.2664), *Consolidation* (0.2827), la clase de control *No Finding* (0.2903), *Cardiomegaly* (0.2937), *Edema* (0.3625) y *Atelectasis* (0.3921) operan en un rango intermedio. Estas condiciones presentan características morfológicas más predecibles en la radiografía frontal, permitiendo al modelo balancear de forma equitativa la tasa de verdaderos positivos y verdaderos negativos sin necesidad de sesgar agresivamente la frontera de decisión.

**3. Perfil de Alta Especificidad / Conservador (Umbrales Altos: > 0.45)**
En el extremo opuesto, hallazgos como *Pleural Effusion* (0.4905), *Lung Opacity* (0.5171) y, notablemente, *Support Devices* (0.5820) exigen un alto nivel de certidumbre probabilística para ser clasificados como positivos. En el caso de los dispositivos de soporte (cables, marcapasos), su fuerte contraste metálico hace que sean fácilmente detectables; por tanto, el modelo eleva el umbral a 0.58 para garantizar una especificidad extrema, reduciendo a cero las falsas alarmas ante posibles artefactos visuales.

La extracción de esta matriz de calibración confirma que el sistema ha abandonado la inferencia probabilística genérica para adoptar una toma de decisiones penalizada y adaptativa. Estos 14 umbrales se fijan y congelan en esta etapa, estableciendo las reglas de clasificación definitivas que se aplicarán en la auditoría final sobre el conjunto de prueba aislado (*Test Set*).

### <font color='darkblue'>2.6.2. Auditoría Clínica Final (Conjunto de Prueba / Test Set)</font>

El objetivo principal de esta primera etapa analítica es auditar el aprendizaje del codificador visual y establecer las reglas operativas del sistema. Para garantizar la integridad metodológica y evitar la fuga de información (*data leakage*), todas las decisiones de ajuste y calibración se han restringido exclusivamente a la cohorte de validación interna. Esta estrategia asegura que el modelo se ponga a punto en un entorno controlado antes de su evaluación definitiva.

#### <font color='darkblue'>2.6.1.1. Rendimiento Bruto: Curvas ROC y Capacidad Discriminativa</font>

Antes de establecer los puntos de decisión clínica, resulta imperativo evaluar la capacidad discriminativa subyacente de la arquitectura neuronal. A través de la extracción de las curvas Características Operativas del Receptor (ROC) y el cálculo del Área Bajo la Curva (AUC) para las 14 etiquetas objetivo, se demuestra de forma matemática que el proceso de entrenamiento ha convergido de manera estable.

La morfología de las curvas confirma que la red convolucional ha aprendido a proyectar las características radiológicas en un espacio latente donde es capaz de separar eficazmente las instancias patológicas de las sanas. Este resultado valida la robustez del extractor de características, descartando la presencia de memorización excesiva (*overfitting*) sobre los datos de entrenamiento.

In [ ]:
# ==============================================================================
# AUDITORÍA FINAL: RENDIMIENTO SOBRE COHORTE DE PRUEBA (TEST SET)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score
from tqdm import tqdm
import torch

print("🚀 Iniciando Auditoría Final sobre el Test Set (Datos Inéditos)...")

all_preds_test, all_targets_test = [], []
model.eval()

with torch.no_grad():
    for batch in tqdm(test_loader, desc="🧪 Evaluando Test (ROC Final)"):
        inputs = batch["image"].to(device)
        targets = batch["label"].to(device, dtype=torch.float32)

        # Inferencia directa sin tocar nada
        probs = torch.sigmoid(model(inputs)).cpu()
        all_preds_test.append(probs)
        all_targets_test.append(targets.cpu())

test_probs = torch.cat(all_preds_test).numpy()
test_labels = (torch.cat(all_targets_test).numpy() >= 0.5).astype(int)

# --- 1. REPORTE DE RESULTADOS Y PREPARACIÓN DE GRÁFICAS ---
print("\n" + "="*50)
print("📊 RESULTADOS FINALES SOBRE TEST SET:")
print("="*50)

# Extraemos el directorio destino de la Fuente de Verdad (CONFIG)
output_dir = CONFIG["dir"]["eval_test"]
os.makedirs(output_dir, exist_ok=True)

num_patologias = len(audit.target_cols)
columnas = 3
filas = (num_patologias + columnas - 1) // columnas

fig, axes = plt.subplots(filas, columnas, figsize=(18, 5 * filas), dpi=120)
axes = axes.ravel() # Aplanamos la matriz de subplots para un indexado lineal seguro

macro_auc_test = []

for i, col in enumerate(audit.target_cols):
    y_true = test_labels[:, i]
    probs = test_probs[:, i]
    ax = axes[i]

    # Comprobación de seguridad: Solo calculamos ROC si hay al menos 1 positivo y 1 negativo (Evita fallo en Fracture)
    if len(np.unique(y_true)) > 1:
        fpr, tpr, thresholds = roc_curve(y_true, probs)
        roc_auc = auc(fpr, tpr)
        macro_auc_test.append(roc_auc)

        # Comparamos con el umbral de Youden de Validación
        pred_bin = (probs >= umbrales_finales[col]).astype(int)
        acc = accuracy_score(y_true, pred_bin)
        print(f" > {col:<26}: AUC: {roc_auc:.4f} | Acc @ Youden: {acc:.4f}")

        # --- DIBUJADO DE LA CURVA ROC EN TEST ---
        # Usamos un color distinto (ej. darkorange) para diferenciar visualmente que es el Test Set
        ax.plot(fpr, tpr, color='darkorange', lw=2.5, label=f"Test ROC (AUC = {roc_auc:.4f})")
        ax.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.5)

        # Localizamos el punto más cercano en la curva de Test al umbral de Youden que impusimos
        idx_thresh = np.argmin(np.abs(thresholds - umbrales_finales[col])) if len(thresholds) > 0 else 0
        ax.scatter(fpr[idx_thresh], tpr[idx_thresh], color='red', s=120, zorder=5, edgecolor='black',
                   label=f"Frontera Youden\n(Thresh = {umbrales_finales[col]:.4f})")

    else:
        # Manejo elegante para clases sin datos suficientes en el Test Set (ej. Fracture)
        macro_auc_test.append(np.nan)
        print(f" > {col:<26}: AUC: nan    | Acc @ Youden: N/A (Faltan datos)")
        ax.text(0.5, 0.5, "Datos insuficientes en Test\n(Sin Casos Positivos)",
                ha='center', va='center', fontsize=11, fontweight='bold', color='red')

    # Estética, formateo avanzado y rotulación de cuadrículas individuales
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_title(f"Test Set ROC: {col}", fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel("1 - Especificidad (FPR)", fontsize=10, fontweight='bold')
    ax.set_ylabel("Sensibilidad (TPR)", fontsize=10, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    if len(np.unique(y_true)) > 1:
        ax.legend(loc="lower right", fontsize=9, frameon=True, shadow=True)

# Limpieza dinámica defensiva de subplots huérfanos si la matriz no queda completa
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Auditoría Final: Curvas ROC y Proyección de Umbrales sobre el Patrón Oro (Test Set)",
             fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()

# --- 2. EXPORTACIÓN AUTOMÁTICA Y GUARDADO PERSISTENTE EN DRIVE ---
# Guardamos con sufijo _test para distinguirlo de tu gráfica de validación
save_path_roc_test = os.path.join(output_dir, "curvas_roc_test_finales.png")
plt.savefig(save_path_roc_test, dpi=300, bbox_inches='tight')

# Resumen Final
print("-" * 50)
# Calculamos la media ignorando los NaNs para no corromper el Macro-AUC global
print(f"✅ MACRO-AUC TEST (Ignorando NaNs): {np.nanmean(macro_auc_test):.4f}")
print("="*50)

print("\n" + "="*80)
print(f"✅ Panel matricial de ROC (Test Set) exportado a Drive: {save_path_roc_test}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()

La evaluación ciega sobre el conjunto de prueba arroja un **Macro-AUC global de 0.7414**, un resultado que valida empíricamente la capacidad predictiva del modelo en un entorno clínico inédito. Sin embargo, el análisis desagregado por patología revela un comportamiento estratificado que resulta fundamental desglosar para comprender la verdadera utilidad de la matriz visual extraída:

**1. Éxitos Sobresalientes en Diagnósticos Críticos (AUC > 0.83)**
El sistema demuestra una capacidad de generalización de grado experto en patologías de alta urgencia clínica y anomalías macroscópicas. Superar la barrera del 0.91 en hallazgos como el *Edema* (0.9155) y el *Derrame Pleural / Pleural Effusion* (0.9107) certifica que la arquitectura convolucional es extremadamente robusta ante acumulaciones de fluidos. Asimismo, el altísimo rendimiento en *No Finding* (0.9163) garantiza que el modelo posee una excelente capacidad para descartar pacientes sanos. La precisión sostenida en *Lung Opacity* (0.8774), *Consolidation* (0.8765) y *Cardiomegaly* (0.8320) reafirma que las transformaciones espaciales no han alterado la geometría cardiaca ni pulmonar.

**2. Rendimiento Estable en Hallazgos Complejos (AUC 0.70 - 0.79)**
Patologías con mayor ambigüedad visual inter-observador, como *Atelectasis* (0.7852), *Pleural Other* (0.7711), *Pneumonia* (0.7236) y la presencia de *Support Devices* (0.7155) mantienen una precisión predictiva consistente. Aunque el AUC es más moderado, las métricas de exactitud bajo los umbrales calibrados de Youden (ej. 71.78% en *Neumonía* o 71.29% en *Atelectasia*) demuestran que la calibración realizada previamente en validación protege la viabilidad operativa del modelo frente a estas clases.

**3. Límites de Resolución Espacial (AUC 0.60 - 0.69)**
Las métricas en *Enlarged Cardiomediastinum* (0.6423) y *Pneumothorax* (0.6227) reflejan una caída predecible en el rendimiento. Como se hipotetizó en el diseño de la arquitectura, la detección de trazos hiperfinos (como la línea pleural de un neumotórax) sufre debido a la compresión espacial inherente a las convoluciones. No obstante, superar el 0.62 confirma que la red retiene señal útil en su **cuadrícula de $10 \times 10$**, evitando el colapso a pesar del submuestreo.

**4. Análisis Forense de Anomalías Estadísticas (*Dataset Shift*)**
Para garantizar la integridad científica de la auditoría, es imperativo justificar las dos desviaciones críticas observadas:
* **El fenómeno del AUC indeterminado (`nan`) en *Fracture*:** Al tratarse de una cohorte de prueba reducida (evaluada por consenso radiológico estricto), la prevalencia de fracturas costales en este subconjunto ha sido nula (ausencia de verdaderos positivos). Matemáticamente, esto impide el trazado del eje de sensibilidad (división por cero), resultando en un valor `nan`. Se trata de una limitación intrínseca de la muestra estadística, no de un fallo algorítmico.
* **Divergencia de Distribución en *Lung Lesion* (AUC 0.0498):** Un AUC inferior a 0.50 indica una predicción inversamente correlacionada. En el contexto de CheXpert, este fenómeno clínico deriva de un sesgo de anotación extremo (*Dataset Shift*) entre la cohorte de entrenamiento y el criterio del *Test Set*. Al tratarse de una anomalía altamente inespecífica y fuertemente desbalanceada (*Long-Tail*), las representaciones latentes extraídas difieren diametralmente de los casos positivos inéditos. Esta anomalía es un artefacto matemático conocido en el aprendizaje multietiqueta bajo incertidumbre extrema.

**Veredicto de la Fase 1:**
El **Macro-AUC de 0.7414** valida que el espacio latente topológico extraído posee una alta fidelidad estructural. La **cuadrícula espacial de $10 \times 10$** (compuesta por 100 tokens visuales de 1024 canales) ha superado el escrutinio clínico. Se encuentra matemáticamente estabilizada y libre de atajos predictivos (*Shortcut Learning*), quedando oficialmente validada para su inyección en el Proyector MLP de la Fase 2 (el Puente Multimodal).

#### <font color='darkblue'>2.6.2.2. Métricas de Utilidad Clínica (Sensibilidad, Especificidad y F1-Score)</font>



Si bien el Área Bajo la Curva (ROC-AUC) es el estándar de oro en la literatura de *Machine Learning* para evaluar la capacidad de discriminación de un modelo, esta métrica resulta abstracta en la práctica médica diaria. Un radiólogo no toma decisiones basadas en el ordenamiento probabilístico de los pacientes, sino en predicciones binarias categóricas: el paciente requiere tratamiento o puede recibir el alta.

Por consiguiente, se ha procedido a binarizar las probabilidades continuas generadas sobre el conjunto de prueba inédito (*Test Set*), utilizando de forma estricta los umbrales de decisión previamente calibrados mediante el Índice de Youden. Esta proyección permite extraer el rendimiento del sistema en el lenguaje clínico tradicional:

* **Sensibilidad (Recall):** De todos los pacientes verdaderamente enfermos en la cohorte de prueba, ¿qué porcentaje logró identificar correctamente el modelo? Un valor alto aquí minimiza el riesgo vital de pasar por alto una patología aguda (Falsos Negativos).
* **Especificidad:** De todos los pacientes verdaderamente sanos, ¿a cuántos diagnosticó como libres de enfermedad? Un valor alto evita intervenciones innecesarias, reduciendo el estrés hospitalario y el coste derivado de falsas alarmas (Falsos Positivos).
* **F1-Score y Precision-Recall AUC:** Ante la presencia de clases minoritarias, el F1-Score (media armónica entre Precisión y Sensibilidad) ofrece una visión penalizada del rendimiento, asegurando que un alto índice de aciertos no sea el resultado estadístico de predecir mayoritariamente la clase dominante.

In [ ]:
# ==============================================================================
# AUDITORÍA CLÍNICA: CÁLCULO DE MÉTRICAS ROBUSTAS Y CURVAS PR SOBRE TEST SET
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, average_precision_score, precision_recall_curve

# Recuperamos la carpeta de evaluación del CONFIG
output_dir = CONFIG["dir"]["eval_test"]
os.makedirs(output_dir, exist_ok=True)

print("⚖️ CALCULANDO MÉTRICAS CLÍNICAS SOBRE TEST SET (CON UMBRALES YOUDEN)...\n")

# --- 1. CÁLCULO DE MÉTRICAS Y TABLA DE RESULTADOS DE PRODUCCIÓN ---
resultados_metricas_test = []
test_preds_opt = np.zeros_like(test_probs)

for i, col in enumerate(audit.target_cols):
    u_opt = umbrales_finales[col] # El umbral congelado en Validación

    # Binarización determinista basada en Youden
    y_pred_bin = (test_probs[:, i] >= u_opt).astype(int)
    test_preds_opt[:, i] = y_pred_bin
    y_true = test_labels[:, i]

    # Extracción segura de la matriz de confusión
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_bin, labels=[0, 1]).ravel()

    # Modelado de ecuaciones con protección contra división por cero
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    sensibilidad = tp / (tp + fn) if (tp + fn) > 0 else 0
    especificidad = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * sensibilidad) / (precision + sensibilidad) if (precision + sensibilidad) > 0 else 0

    # Manejo de excepciones para AP en clases sin casos positivos (Fracture)
    try:
        if len(np.unique(y_true)) > 1:
            ap = average_precision_score(y_true, test_probs[:, i])
        else:
            ap = np.nan
    except ValueError:
        ap = np.nan

    resultados_metricas_test.append({
        "Patología": col,
        "Umbral (J)": round(u_opt, 4),
        "Sensibilidad": round(sensibilidad, 4),
        "Especificidad": round(especificidad, 4),
        "Precision": round(precision, 4),
        "F1-Score": round(f1, 4),
        "PR AUC (AP)": round(ap, 4) if not np.isnan(ap) else "N/A"
    })

# Formateo estructurado para inserción directa en el TFG
df_metricas_test = pd.DataFrame(resultados_metricas_test)
display(df_metricas_test)


# --- 2. GRÁFICAS: CURVAS PRECISION-RECALL (PR) MATRICIALES ---
num_patologias = len(audit.target_cols)
columnas = 3
filas = (num_patologias + columnas - 1) // columnas

fig_pr, axes_pr = plt.subplots(filas, columnas, figsize=(18, 5 * filas), dpi=120)
axes_pr = axes_pr.ravel()

for i, col in enumerate(audit.target_cols):
    y_true = test_labels[:, i]
    probs = test_probs[:, i]
    ax = axes_pr[i]

    # Solo graficamos si hay al menos una muestra positiva y una negativa
    if len(np.unique(y_true)) > 1:
        precision_vals, recall_vals, thresh_vals = precision_recall_curve(y_true, probs)
        ap_score = df_metricas_test.loc[i, "PR AUC (AP)"]

        # Trazado de la curva Precision-Recall
        ax.plot(recall_vals, precision_vals, color='darkmagenta', lw=2.5, label=f'PR Curve (AP = {ap_score})')

        # Proyección geométrica del Punto Operativo Youden
        idx_opt = np.argmin(np.abs(thresh_vals - umbrales_finales[col])) if len(thresh_vals) > 0 else 0
        ax.scatter(recall_vals[idx_opt], precision_vals[idx_opt], color='red', s=120, zorder=5, edgecolor='black',
                   label=f"Youden (Test)\n(Thresh = {umbrales_finales[col]:.4f})")
    else:
        ax.text(0.5, 0.5, "Datos insuficientes en Test\n(Sin Casos Positivos)",
                ha='center', va='center', fontsize=11, fontweight='bold', color='red')

    # Estética y formateo avanzado
    ax.set_xlim([-0.05, 1.05])
    ax.set_ylim([-0.05, 1.05])
    ax.set_title(f"Test Set PR Curve: {col}", fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel("Recall (Sensibilidad)", fontsize=10, fontweight='bold')
    ax.set_ylabel("Precision (VPP)", fontsize=10, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    if len(np.unique(y_true)) > 1:
        ax.legend(loc="lower left", fontsize=9, frameon=True, shadow=True)

# Limpieza dinámica de subplots huérfanos
for j in range(i + 1, len(axes_pr)):
    fig_pr.delaxes(axes_pr[j])

plt.suptitle("Auditoría Final: Curvas Precision-Recall y Puntos Operativos sobre Patrón Oro", fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()

# --- 3. EXPORTACIÓN AUTOMÁTICA ---
path_pr_test = os.path.join(output_dir, "curvas_pr_test_finales.png")
plt.savefig(path_pr_test, dpi=300, bbox_inches='tight')

# Opcional: exportar la tabla de métricas a CSV para copiar a LaTeX/Excel
path_csv_test = os.path.join(output_dir, "metricas_clinicas_test.csv")
df_metricas_test.to_csv(path_csv_test, index=False)

print("\n" + "="*80)
print(f"✅ Tabla de Métricas Clínicas guardada en: {path_csv_test}")
print(f"✅ Panel de Curvas PR (Test Set) exportado a Drive: {path_pr_test}")
print("="*80 + "\n")

plt.show()

El análisis detallado de las métricas operativas revela que el **Codificador Visual Clínico (Fase 1)** ha adoptado un perfil de actuación enfocado en la seguridad del paciente (triaje defensivo), impulsado por la calibración individualizada de los umbrales de Youden.

**1. Fiabilidad en Patologías Críticas (Alta Sensibilidad):**
El sistema demuestra una competencia excepcional como herramienta de cribado en patologías agudas. Alcanza una Sensibilidad del **90.48% en Edema** y del **90.62% en Consolidación**. Desde una perspectiva de impacto clínico, esto se traduce en una tasa de Falsos Negativos inferior al 10% en condiciones que amenazan la vida del paciente, cumpliendo con el requisito primario de un sistema de diagnóstico de urgencias: no omitir lesiones graves.

**2. Robustez y Precisión Diagnóstica:**
Las clases *Lung Opacity* y *Pleural Effusion* exhiben el equilibrio más robusto del sistema. En el caso de la Opacidad Pulmonar, el modelo registra una Especificidad del 84.71% y un PR AUC sobresaliente de 0.9079, lo que certifica un Valor Predictivo Positivo (Precisión) altísimo (0.8602). En estas afecciones macroscópicas, la **cuadrícula espacial de $10 \times 10$** extraída por la red no solo detecta la patología, sino que cuando emite un diagnóstico positivo, la probabilidad de que sea una falsa alarma es mínima.

**3. Impacto de la Prevalencia en la Precisión (Valor Predictivo Positivo):**
Resulta imprescindible contextualizar métricas aparentemente deficientes en clases como *Pneumonia* (Precisión: 0.0847) o *Pleural Other* (Precisión: 0.0145, pero Sensibilidad: 1.0000). Esta drástica caída de la precisión es un artefacto estadístico esperado al evaluar cohortes con extrema baja prevalencia y un desbalanceo severo (*Long-Tail*). Al fijar umbrales de Youden muy bajos (ej. 0.1370) para no perder ningún caso raro, el modelo asume de forma natural una alta tasa de Falsos Positivos. Clínicamente, el algoritmo está configurado de forma óptima: prioriza la alerta ante el paciente dudoso (alta sensibilidad) asumiendo el coste operativo de la falsa alarma, en lugar de arriesgarse a un alta médica indebida.

**Conclusión Operativa:**
Esta matriz de actuación conservadora garantiza que el espacio latente retiene todas las anomalías sutiles, preparando el terreno ideal para el **Puente Multimodal (Fase 2)** y la **Sincronización Clínica (Fase 3)**. Al inyectar estas características en el razonador **BioGPT**, será el modelo de lenguaje el encargado de filtrar estas "falsas alarmas visuales", utilizando su profundo conocimiento médico para refinar el diagnóstico definitivo a partir del contexto global de la radiografía.

#### <font color='darkblue'>2.6.2.3. Análisis Forense de Errores (Matriz de Confusión Global)</font>


Las métricas escalares como el AUC o el F1-Score proporcionan una visión general de la fiabilidad estadística del sistema, pero ocultan la naturaleza intrínseca del fallo. Para validar la seguridad operativa del codificador visual antes de un hipotético despliegue hospitalario, es estrictamente necesario diseccionar las predicciones mediante Matrices de Confusión para cada una de las patologías analizadas sobre la cohorte de prueba (*Test Set*).

Este desglose permite evaluar el impacto asimétrico de las desviaciones del modelo:
* **Falsos Negativos (Riesgo Clínico Alto):** Instancias donde la IA clasifica a un paciente como sano frente a una patología confirmada por el consenso de expertos. En hallazgos críticos como la Neumonía o el Derrame Pleural, este error implica la omisión de un tratamiento urgente, representando el punto de fallo más severo del sistema.
* **Falsos Positivos (Riesgo Operativo):** Instancias donde la IA detecta anomalías inexistentes en pacientes sanos. Aunque clínicamente es un error más tolerable y defensivo, una tasa elevada genera fatiga de alerta en los facultativos y sobrecarga los recursos hospitalarios con pruebas diagnósticas innecesarias.

Al aplicar los umbrales de Youden (que priorizan la sensibilidad en lesiones sutiles y la especificidad en hallazgos obvios), la matriz de confusión refleja empíricamente esta calibración. Este análisis forense demuestra de forma transparente no solo en qué escenarios el algoritmo alcanza el nivel de un experto, sino también sus vulnerabilidades anatómicas o sesgos estructurales derivados de la distribución de los datos originales.

In [ ]:
# ==============================================================================
# ANÁLISIS FORENSE: MATRICES DE CONFUSIÓN EN RESOLUCIÓN CLÍNICA (TEST SET)
# ==============================================================================
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("🔍 GENERANDO MATRICES DE CONFUSIÓN SOBRE TEST SET...\n")

# Aseguramos el directorio de salida para la fase final
output_dir_test = os.path.join(CONFIG["dir"]["eval_test"])
os.makedirs(output_dir_test, exist_ok=True)

# Configuración dinámica de la cuadrícula (14 patologías -> 5 filas x 3 columnas)
num_patologias = len(audit.target_cols)
columnas = 3
filas = (num_patologias + columnas - 1) // columnas

fig_cm, axes_cm = plt.subplots(filas, columnas, figsize=(16, 4 * filas), dpi=120)
axes_cm = axes_cm.ravel()

for i, col in enumerate(audit.target_cols):
    # Recuperamos las etiquetas reales y predicciones óptimas calculadas en el bloque anterior
    y_true = test_labels[:, i]
    y_pred = test_preds_opt[:, i]

    # Calculamos la matriz forzando la estructura [0, 1] para evitar colapsos si falta alguna clase
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    # Renderizado térmico con Seaborn (Estética limpia y profesional para TFG)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[i],
                cbar=False, annot_kws={"size": 14, "weight": "bold"})

    # Rotulación formal del panel
    axes_cm[i].set_title(f"Confusión en Test: {col}", fontsize=13, fontweight='bold', pad=10)
    axes_cm[i].set_xlabel("Predicción del Módulo IA", fontsize=11, fontweight='bold')
    axes_cm[i].set_ylabel("Diagnóstico Ground Truth", fontsize=11, fontweight='bold')

    # Ejes explícitos para facilitar la lectura médica
    axes_cm[i].set_xticklabels(['Sano (0)', 'Enfermo (1)'])
    axes_cm[i].set_yticklabels(['Sano (0)', 'Enfermo (1)'])

# Limpieza dinámica defensiva de subplots huérfanos (ej. la celda 15 vacía)
for j in range(i + 1, len(axes_cm)):
    fig_cm.delaxes(axes_cm[j])

plt.suptitle("Análisis Forense: Matrices de Confusión Globales sobre Patrón Oro (Test Set)", fontsize=16, fontweight='bold', y=0.99)
plt.tight_layout()

# Guardado persistente automático
path_cm_test = os.path.join(output_dir_test, "matrices_confusion_test.png")
plt.savefig(path_cm_test, dpi=300, bbox_inches='tight')

print("\n" + "="*80)
print(f"✅ Panel de Matrices de Confusión (Test) exportado a Drive: {path_cm_test}")
print("="*80 + "\n")

# Despliegue en el cuaderno
plt.show()


La proyección visual de las predicciones a través de las matrices de confusión confirma empíricamente que el modelo ha internalizado un comportamiento clínico conservador, priorizando la seguridad vital del paciente (minimización de Falsos Negativos) por encima de la comodidad operativa (Falsos Positivos).

De la inspección forense de los paneles se extraen tres conclusiones fundamentales que validan la viabilidad del sistema:

**1. Excelencia en la Seguridad del Paciente (Minimización del Falso Negativo)**
El cuadrante inferior izquierdo de cada matriz (FN) representa el error médico más grave: ignorar la enfermedad. En patologías críticas de presentación aguda, el modelo demuestra una contundencia excepcional. En la clase **Consolidation**, de los 32 pacientes verdaderamente enfermos, el modelo detecta 29 y solo omite 3 (FN = 3). De manera similar, en **Edema**, detecta 38 casos reales y solo pierde 4. Especialmente reseñable es el panel de **No Finding** (Paciente Sano): de los 26 pacientes que realmente presentaban alguna alteración, la IA detectó a 25. Cometer un único Falso Negativo (FN = 1) en esta categoría certifica que cuando el modelo dicta un alta médica ("Sano"), el nivel de certidumbre es prácticamente absoluto.

**2. El Coste Operacional de los Umbrales de Youden (Falsos Positivos)**
La fila superior de los paneles ilustra el compromiso (*trade-off*) matemático asumido durante la calibración. En patologías de alta sutileza donde se forzó un umbral cercano a 0.15 (como *Pleural Other* o *Lung Lesion*), se observa una migración masiva de predicciones hacia el cuadrante superior derecho (FP = 68 y FP = 71, respectivamente). Esta "sobre-detección" es una consecuencia deliberada de la ingeniería del sistema: ante hallazgos ambiguos, la red convolucional delega la decisión derivando al paciente sano a una revisión radiológica innecesaria, asumiendo el coste del Falso Positivo para proteger la sensibilidad total de la red.

**3. Auditoría Visual de Cohortes Desbalanceadas**
Las matrices resuelven visualmente las aparentes anomalías matemáticas del cálculo del AUC. El panel de **Fracture** revela una ausencia total de pacientes positivos reales en este subconjunto de prueba (FN = 0, TP = 0), lo que justifica el cálculo indeterminado (`nan`) reportado previamente. Por su parte, la dispersión de predicciones en **Lung Lesion** corrobora el severo cambio de dominio en las etiquetas (*Dataset Shift*) entre el criterio de los médicos de entrenamiento y el consenso del panel de prueba, dejando patente el reto intrínseco de la subjetividad médica en el diagnóstico de lesiones pulmonares focales.

### <font color='darkblue'>2.6.3. La Caja de Cristal: Interpretabilidad y Explicabilidad (XAI)</font>

A pesar de que las métricas de rendimiento clínico (AUC, Sensibilidad) sobre el conjunto de prueba confirman la viabilidad matemática del sistema, las redes neuronales convolucionales operan intrínsecamente como "cajas negras". En el dominio médico, un alto índice de acierto carece de valor si el modelo está tomando las decisiones correctas por los motivos equivocados (fenómeno conocido como *Shortcut Learning* o aprendizaje de atajos, donde la IA podría estar memorizando artefactos del escáner en lugar de la anatomía real).

Para mitigar este riesgo y dotar al sistema de transparencia clínica, se ha implementado un motor de interpretabilidad basado en **Grad-CAM** (*Gradient-weighted Class Activation Mapping*).

#### <font color='darkblue'>2.6.3.1. Explicabilidad Contrafactual sobre Datos Inéditos (Test Set)</font>

La auditoría final de interpretabilidad se ha ejecutado de manera exclusiva sobre los pacientes del *Test Set*. Este enfoque contrafactual consiste en escanear la cohorte de prueba para extraer un par clínico por cada patología:
* **Verdadero Positivo (Paciente Enfermo):** Demuestra que el modelo focaliza sus gradientes de activación espacial exactamente sobre las regiones anatomopatológicas descritas en la literatura (por ejemplo, los recesos costodiafragmáticos en el derrame pleural o el espacio alveolar difuso en la consolidación).
* **Verdadero Negativo (Paciente Sano):** Sirve como grupo de control visual. Demuestra que, ante la ausencia de la patología, los mapas de características se dispersan en un patrón de exploración global sin concentrarse falsamente en estructuras sanas, validando que el modelo no "inventa" lesiones.

La generación de estos mapas térmicos confirma que el codificador visual ha extraído una representación causal y biológicamente coherente del tórax humano.

In [ ]:
# ==============================================================================
# MÓDULO DE INTERPRETABILIDAD CLÍNICA AVANZADA: MOTOR GRAD-CAM MANUAL
# ==============================================================================
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os

class GradCAM:
    """
    Implementación nativa de Grad-CAM diseñada para auditar el espacio de activación
    convolucional de la DenseNet121.
    """
    def __init__(self, model, target_layer):
        self.model, self.target_layer = model, target_layer
        self.gradients, self.activations = None, None
        self.target_layer.register_forward_hook(self.save_activations)
        self.target_layer.register_full_backward_hook(self.save_gradients)

    def save_activations(self, m, i, o):
        self.activations = o.detach()

    def save_gradients(self, m, gi, go):
        self.gradients = go[0].detach()

    def __call__(self, input_tensor, target_class_idx):
        self.model.zero_grad()
        if input_tensor.grad is not None:
            input_tensor.grad.zero_()

        for m in self.model.modules():
            if hasattr(m, 'inplace'):
                m.inplace = False

        output = self.model(input_tensor)
        output[0, target_class_idx].backward()

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1).squeeze(0).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))

        cam -= cam.min()
        cam /= (cam.max() + 1e-10)
        return cam

# --- FIX DE ARQUITECTURA ---
def fixed_features2_for_gradcam(self, x):
    features = self.features(x)
    out = F.relu(features, inplace=False) # FIX CRÍTICO: Rompemos el inplace
    out = F.adaptive_avg_pool2d(out, (1, 1)).view(features.size(0), -1)
    return out

model.features2 = fixed_features2_for_gradcam.__get__(model, type(model))

for module in model.modules():
    if isinstance(module, torch.nn.ReLU):
        module.inplace = False

# --- OBTENCIÓN DE MUESTRA DEL TEST SET (CORREGIDO) ---
gcam = GradCAM(model, model.features[-1])

# [CRÍTICO] Extraemos muestra del TEST_LOADER, no del val_loader
batch = next(iter(test_loader))
input_tensor = batch["image"][0:1].to(device).requires_grad_(True)

with torch.no_grad():
    probs = torch.sigmoid(model(input_tensor.clone())).cpu().numpy()[0]



Para operacionalizar la auditoría de interpretabilidad, se ha desarrollado una función analítica (`generar_matriz_gradcam_contraste`) que examina secuencialmente la cohorte de prueba (*Test Set*) en busca de un escenario de contraste óptimo para cada patología.

A diferencia de la evaluación matemática tradicional, que agrega el rendimiento de cientos de pacientes en un único número (AUC), este algoritmo ejecuta una **explicabilidad basada en ejemplos (*Example-based Explanation*)**. Su flujo de trabajo está estructurado en tres fases:

1. **Escaneo y Selección Ciega:** El algoritmo recorre las predicciones del modelo sobre el Test Set y selecciona el "par perfecto" para cada diagnóstico: un Verdadero Positivo (enfermo detectado con alta probabilidad) y un Verdadero Negativo (sano descartado con baja probabilidad).
2. **Extracción de Gradientes (Backward Pass):** Una vez localizados los pacientes, el motor Grad-CAM congela la red e inyecta las imágenes en la DenseNet121. Al forzar el cálculo de gradientes hacia atrás desde la predicción final hasta la última capa convolucional, se ponderan las activaciones espaciales que motivaron el fallo de la neurona de salida.
3. **Renderizado de Mapas Térmicos:** Los gradientes se rectifican (ReLU) para aislar únicamente las características positivas y se superponen sobre la radiografía original mediante un mapa de calor (*colormap 'jet'*).

La matriz resultante expone de forma directa y visual si la arquitectura está empleando un **enfoque lesivo** (iluminando de rojo el tejido anómalo en pacientes enfermos) y una **dispersión de descarte** (enfriando el mapa térmico sobre pacientes sanos). Este escrutinio asegura que el rendimiento de la fase previa no es fruto del aprendizaje de atajos (*Shortcut Learning*).

In [ ]:
# ==============================================================================
# INTERPRETABILIDAD CONTRAFACTUAL: MATRIZ GRAD-CAM DE CONTRASTE CLÍNICO
# ==============================================================================
def generar_matriz_gradcam_contraste(model, dataloader_test, target_cols, umbrales, device, output_dir):
    print("⏳ Escaneando Test Set para encontrar el par perfecto (Enfermo vs Sano)...")
    model.eval()

    casos_positivos = {col: None for col in target_cols}
    casos_negativos = {col: None for col in target_cols}

    with torch.no_grad():
        for batch in dataloader_test: # Usando el dataloader inyectado (test_loader)
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)
            probs = torch.sigmoid(model(inputs)).cpu().numpy()

            for b in range(inputs.size(0)):
                raw_labels_b = labels[b].cpu().numpy()
                binary_labels_b = (raw_labels_b >= 0.5).astype(int)

                for i, col in enumerate(target_cols):
                    u_optimo = umbrales[col]

                    if casos_positivos[col] is None and binary_labels_b[i] == 1 and probs[b][i] >= u_optimo:
                        casos_positivos[col] = (inputs[b].cpu().clone(), probs[b][i])

                    if casos_negativos[col] is None and binary_labels_b[i] == 0 and probs[b][i] < u_optimo:
                        casos_negativos[col] = (inputs[b].cpu().clone(), probs[b][i])

            if all(v is not None for v in casos_positivos.values()) and all(v is not None for v in casos_negativos.values()):
                break

    print("🎯 Candidatos clínicos indexados. Extrayendo mapas de calor térmicos...")
    gcam = GradCAM(model, model.features[-1])

    fig, axes = plt.subplots(nrows=len(target_cols), ncols=2, figsize=(14, 4 * len(target_cols)), dpi=120)
    plt.style.use('default')

    if len(target_cols) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, col in enumerate(target_cols):
        u_optimo = umbrales[col]

        # --- COLUMNA 1: TP ---
        ax_pos = axes[i, 0]
        if casos_positivos[col] is not None:
            tensor_img, prob = casos_positivos[col]
            tensor_in = tensor_img.unsqueeze(0).to(device).requires_grad_(True)
            mask = gcam(tensor_in, i)
            img_disp = tensor_img.numpy().transpose(1, 2, 0)
            img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

            ax_pos.imshow(img_disp[:, :, 0], cmap='bone')
            ax_pos.imshow(mask, cmap='jet', alpha=0.45)
            ax_pos.set_title(f"🔍 {col} - REAL: POSITIVO\nIA localiza lesión: {prob:.2f} (U:{u_optimo:.2f})",
                             fontsize=11, fontweight='bold', color='darkred', pad=10)
        else:
            ax_pos.text(0.5, 0.5, 'Sin pacientes TP\nen el Test Set', ha='center', va='center', fontsize=10)
        ax_pos.axis('off')

        # --- COLUMNA 2: TN ---
        ax_neg = axes[i, 1]
        if casos_negativos[col] is not None:
            tensor_img, prob = casos_negativos[col]
            tensor_in = tensor_img.unsqueeze(0).to(device).requires_grad_(True)
            mask = gcam(tensor_in, i)
            img_disp = tensor_img.numpy().transpose(1, 2, 0)
            img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

            ax_neg.imshow(img_disp[:, :, 0], cmap='bone')
            ax_neg.imshow(mask, cmap='jet', alpha=0.45)
            ax_neg.set_title(f"✅ {col} - REAL: SANO\nIA explora y descarta: {prob:.2f} (U:{u_optimo:.2f})",
                             fontsize=11, fontweight='bold', color='darkgreen', pad=10)
        else:
            ax_neg.text(0.5, 0.5, 'Sin pacientes TN\nen el Test Set', ha='center', va='center', fontsize=10)
        ax_neg.axis('off')

    plt.suptitle("Auditoría XAI en Test Set: Enfoque Lesivo vs. Dispersión de Descarte",
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, "gradcam_matriz_contraste_test.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

    print("\n" + "="*80)
    print(f"✅ Matriz contrafactual (Test Set) guardada en Drive: {save_path}")
    print("="*80 + "\n")
    plt.show()

# ==============================================================================
# EJECUCIÓN (APUNTANDO AL TEST LOADER)
# ==============================================================================
generar_matriz_gradcam_contraste(
    model=model,
    dataloader_test=test_loader,         # [CRÍTICO] Pasamos el Test Set
    target_cols=audit.target_cols,
    umbrales=umbrales_finales,
    device=device,
    output_dir=os.path.join(CONFIG["dir"]["gradcam"])
)

<font color='darkblue'>AUDITORÍA DE EXPLICABILIDAD VISUAL: ANÁLISIS COMPRENSIVO DE MAPAS DE ACTIVACIÓN GRAD-CAM</font>

---

<font color='darkblue'>1. Patologías de Alta Precisión y Consenso Clínico</font>

<font color='darkblue'>1.1. No Finding (Sin Hallazgos)</font>

> **Criterio Fisiopatológico y Diagnóstico:** La etiqueta *No Finding* constituye un diagnóstico de exclusión en el entorno clínico. Determinar la total normalidad de una radiografía de tórax exige un escaneo sistemático y exhaustivo del parénquima pulmonar, los contornos mediastínicos, la silueta cardíaca y los recesos costodiafragmáticos con el fin de certificar la ausencia absoluta de opacidades, derrames, neumotórax o distorsiones anatómicas macroscópicas.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.79 | 0.29 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.26 | 0.29 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.79):** El mapa de activación Grad-CAM concentra la máxima densidad de calor sobre la silueta cardíaca y el lóbulo inferior del pulmón derecho. Este comportamiento demuestra coherencia anatómica, dado que la red inspecciona las regiones críticas con mayor prevalencia de error (como la cardiomegalia o los infiltrados basales). Al verificar la indemnidad estructural de estas zonas, el modelo incrementa linealmente la confianza hacia la ausencia de patología.
* **Muestra Derecha (Predicción: 0.26):** El gradiente explora la región hiliar derecha, el lóbulo medio e inferior y la franja mediastínica. Al localizar en dicha área un patrón que no correlaciona con los parámetros de normalidad geométrica del parénquima sano, la probabilidad de la clase decae por debajo del umbral crítico, forzando un descarte correcto al detectar un patrón anómalo subyacente.

---

<font color='darkblue'>1.2. Cardiomegaly (Cardiomegalia)</font>

> **Criterio Fisiopatológico y Diagnóstico:** La cardiomegalia define el aumento patológico del tamaño del corazón, evaluado habitualmente en proyecciones frontales mediante el índice cardiotorácico (cuando el diámetro transversal cardíaco supera el 50% del diámetro torácico interno). La atención algorítmica debe dirigirse unívocamente hacia la silueta cardíaca, con especial énfasis en el borde inferior izquierdo, región correspondiente al ventrículo izquierdo que sufre la mayor prominencia adaptativa por dilatación o hipertrofia.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.57 | 0.29 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.05 | 0.29 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.57):** La activación máxima se posiciona de forma contundente y con estricta delimitación anatómica sobre el contorno lateral de la silueta cardíaca (ventrículo izquierdo). El modelo discrimina el tejido pulmonar y las estructuras óseas circundantes, focalizándose de forma exclusiva en el órgano diana, lo que justifica matemáticamente el elevado rendimiento global alcanzado en esta métrica (AUC > 0.83).
* **Muestra Derecha (Predicción: 0.05):** El mapa de calor muestra una inhibición absoluta sobre el área cardíaca, la cual permanece en niveles basales de activación. El remanente de atención periférica se dispersa pasivamente hacia los márgenes diafragmáticos y los vértices superiores, confirmando que, tras escanear el centro y validar proporciones normales, el modelo deprime la probabilidad predictiva de forma óptima.

---

<font color='darkblue'>1.3. Edema (Edema Pulmonar)</font>

> **Criterio Fisiopatológico y Diagnóstico:** El edema pulmonar representa la acumulación anómala de líquido en el espacio intersticial y alveolar, originada predominantemente por claudicación hemodinámica cardíaca. Su traducción radiológica habitual se manifiesta mediante opacidades algodonosas, difusas y bilaterales de distribución perihiliar, configurando el signo semiológico clásico en "alas de mariposa".

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.76 | 0.36 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.15 | 0.36 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.76):** El mapa de calor corrobora una coincidencia anatómica de alto valor científico al concentrarse de forma simétrica en los hilios pulmonares y expandirse bilateralmente hacia los campos medios. La red ignora el ruido instrumental periférico y captura la esencia matemática del patrón en "alas de mariposa", correspondencia que refrenda el rendimiento sobresaliente de la arquitectura en esta clase (AUC > 0.91).
* **Muestra Derecha (Predicción: 0.15):** Ante un parénquima libre de trasudado, la atención se desvía del tejido pulmonar hiliar y se sitúa de forma pasiva sobre el eje del mediastino posterior y la burbuja gástrica. La ausencia del patrón algodonoso característico deprime la respuesta de la red, consolidando un descarte efectivo.

---

<font color='darkblue'>1.4. Pleural Effusion (Derrame Pleural)</font>

> **Criterio Fisiopatológico y Diagnóstico:** El derrame pleural constituye la acumulación de líquido en el espacio comprendido entre las hojas visceral y parietal de la pleura. Por acción de la gravedad, dicho fluido se almacena en los segmentos declives de la cavidad torácica, provocando el borramiento o la opacificación del ángulo agudo de los senos costodiafragmáticos y delimitando semiológicamente la línea de Menisco.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.59 | 0.49 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.05 | 0.49 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.59):** El pico de activación se sitúa con precisión geométrica en la base del pulmón izquierdo del paciente (margen derecho de la imagen), justamente sobre la obliteración del receso costodiafragmático. La red demuestra haber internalizado la dependencia física y espacial de la patología, prescindiendo del análisis de los campos superiores y validando el elevado AUC (> 0.91) reportado en la auditoría cuantitativa.
* **Muestra Derecha (Predicción: 0.05):** Al inspeccionar la región inferior y verificar la integridad anatómica de unos ángulos costodiafragmáticos afilados e hiperlucentes (adecuadamente aireados), el interés algorítmico se extingue en las bases, desplazando una activación mínima residual hacia el mediastino y los vértices superiores, ejecutando un descarte de alta especificidad.

---

<font color='darkblue'>2. Patologías de Rendimiento Estable y Transición Estructural</font>

<font color='darkblue'>2.1. Consolidation (Consolidación)</font>

> **Criterio Fisiopatológico y Diagnóstico:** La consolidación pulmonar sobreviene cuando el aire alveolar es reemplazado por exudado inflamatorio, fibrina, sangre o células, un fenómeno característico de los procesos neumónicos lobares. Radiológicamente cursa como un bloque de aumento homogéneo de la densidad parénquima que suele generar el signo de la silueta al hacer contacto con estructuras de similar densidad como el diafragma o los bordes cardíacos.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.52 | 0.28 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.17 | 0.28 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.52):** La activación máxima se posiciona sobre la base del pulmón izquierdo, solapándose exactamente con el área de densidad consolidativa. Existe una correlación espacial rigurosa entre el hallazgo radiológico estructural y la focalización de pesos de la red neuronal, lo que sustenta la robustez del modelo para identificar pérdidas de aireación localizadas.
* **Muestra Derecha (Predicción: 0.17):** Los segmentos medios e inferiores de ambos campos pulmonares permanecen inactivos. El gradiente residual se fragmenta hacia los tejidos blandos abdominales y el área supraclavicular, mostrando que el descarte se fundamenta en la ausencia de densidades consolidadas en las zonas de interés diagnóstico.

---

<font color='darkblue'>2.2. Pneumonia (Neumonía)</font>

> **Criterio Fisiopatológico y Diagnóstico:** La neumonía representa un proceso infeccioso-inflamatorio agudo del parénquima que genera infiltrados de distribución variable. Clínicamente, es una entity de elevada complejidad diagnóstica mediante Inteligencia Artificial debido al solapamiento casi absoluto de sus características visuales con las clases *Lung Opacity* y *Consolidation*, compartiendo idénticos sustratos de atenuación radiológica.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.27 | 0.24 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.17 | 0.24 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.27):** El mapa de calor exhibe una arquitectura atencional fragmentada, mostrando múltiples focos discontinuos sobre los lóbulos superiores. Si bien la red localiza de forma correcta los infiltrados apicales para superar el umbral por un margen estrecho, la dispersión del gradiente evidencia la ambigüedad conceptual interna del modelo para diferenciar de forma unívoca la neumonía de otras opacidades pulmonares inespecíficas.
* **Muestra Derecha (Predicción: 0.17):** Al no detectar focos inflamatorios o consolidativos evidentes en el tejido pulmonar, el gradiente abandona los campos centrales y migra drásticamente hacia la región infradiafragmática, manteniendo la probabilidad por debajo de la frontera crítica de decisión.

---

<font color='darkblue'>2.3. Atelectasis (Atelectasia)</font>

> **Criterio Fisiopatológico y Diagnóstico:** La atelectasia implica la pérdida de volumen de un segmento o de la totalidad de un pulmón debido al colapso alveolar. Visualmente se traduce en una opacidad asociada a signos indirectos de retracción, tales como el desplazamiento de las fisuras, el mediastino o la elevación del hemidiafragma homolateral.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.59 | 0.39 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.05 | 0.39 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.59):** Grad-CAM despliega una masa de activación de gran intensidad sobre la región media e inferior del pulmón derecho, solapándose con precisión matemática sobre una extensa franja de colapso pulmonar refractario. La correcta correlación entre el área colapsada y los pesos neuronales convalida el rendimiento estable de este módulo.
* **Muestra Derecha (Predicción: 0.05):** El parénquima pulmonar central se halla enteramente silenciado (azul oscuro), lo que indica la confirmación visual de unos pulmones correctamente expandidos. La atención residual emigra hacia los límites corticales y óseos de la caja torácica, garantizando un valor predictivo de descarte óptimo.

---

<font color='darkblue'>2.4. Pleural Other (Otras Anomalías Pleurales)</font>

> **Criterio Fisiopatológico y Diagnóstico:** Esta categoría residual comprende alteraciones estructurales de la pleura no clasificables como derrames o neumotórax, destacando los engrosamientos pleurales localizados, placas fibróticas o calcificaciones reactivas. Al ser anomalías de la membrana que reviste la cavidad, su búsqueda debe ceñirse con rigurosidad matemática al contorno y la periferia externa pulmonar.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.18 | 0.14 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.05 | 0.14 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.18):** A pesar de la sutileza de la probabilidad matemática asignada, el mapa de calor demuestra una excelente finura espacial al ceñirse de forma lineal sobre el margen axilar externo del pulmón izquierdo. La red obvia el parénquima central y las interferencias de monitorización del paciente, enfocando su atención estrictamente en el límite anatómico de la pleura parietal.
* **Muestra Derecha (Predicción: 0.05):** Grad-CAM dibuja un contorno periférico simétrico y bilateral a modo de "marco", rastreando exhaustivamente los bordes torácicos. Al verificar que las paredes pleurales se encuentran libres de irregularidades o engrosamientos densos, la probabilidad se reduce a niveles basales.

---

<font color='darkblue'>3. Desviaciones de Precisión y Desafíos de Resolución Visual</font>

<font color='darkblue'>3.1. Enlarged Cardiomediastinum (Ensanchamiento Cardiomediastínico)</font>

> **Criterio Fisiopatológico y Diagnóstico:** El mediastino delimita el espacio anatómico central del tórax que aloja el corazón, el esófago, la tráquea y las grandes estructuras vasculares. Su ensanchamiento patológico se evalúa mediante la distorsión del contorno lateral superior de la franja central pulmonar, inducido por adenopatías, masas tumorales o aneurismas aórticos.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.32 | 0.24 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.15 | 0.24 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.32):** La activación se localiza de forma bilateral y difusa sobre los hilios, invadiendo los campos pulmonares adyacentes. Este mapa de calor plasma de forma visual la limitación cuantitativa del modelo (AUC de 0.64). En lugar de perfilar con precisión el mediastino superior o el arco aórtico, la red se apoya de manera imprecisa en el incremento de densidad hiliar inespecífico, denotando una baja resolución para discernir los límites estrictos de esta clase.
* **Muestra Derecha (Predicción: 0.15):** La red disemina la atención hacia focos periféricos desconectados, incluyendo el vértice superior derecho y la pared costal. Al constatar la ausencia de vectores de densidad en la franja central superior, ejecuta el descarte por el principio de evasión de la región de interés.

---

<font color='darkblue'>3.2. Lung Opacity (Opacidad Pulmonar)</font>

> **Criterio Fisiopatológico y Diagnóstico:** El término genérico opacidad engloba cualquier atenuación anómala del parénquima que incremente su densidad visual (tornándose radiopaco o blanco) debido a la ocupación del espacio aéreo o intersticial. Su delimitación puede abarcar desde patrones focales a infiltrados multifocales extensos.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.66 | 0.52 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.10 | 0.52 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.66):** El modelo sitúa correctamente su foco de activación de intensidad máxima sobre una opacidad evidente en el campo medio e inferior del pulmón izquierdo. No obstante, se aprecian fugas de atención simultáneas hacia los tejidos blandos del cuello y el hombro contralateral. Este fenómeno responde a que la red opera buscando gradientes de densidad blanquecina, distrayéndose residualmente con la superposición ósea anatómica y artefactos externos de monitorización.
* **Muestra Derecha (Predicción: 0.10):** El mapa de calor evade de forma óptima los campos pulmonares limpios. La atención remanente se desplaza hacia la base cervical y el diafragma, confirmando el descarte al no encontrar zonas de opacificación dentro del tejido respiratorio.

---

<font color='darkblue'>3.3. Pneumothorax (Neumotórax)</font>

> **Criterio Fisiopatológico y Diagnóstico:** El neumotórax consiste en la irrupción de aire en el espacio pleural, lo que colapsa mecánicamente la arquitectura pulmonar. Su diagnóstico digital representa uno de los retos más complejos en la radiología por IA, debido a que no cursa con un aumento de densidad blanco, sino con una sutil franja hiperlucente (negra), carente de trama vascular y delimitada exteriormente por una finísima línea de pleura visceral.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.34 | 0.27 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.07 | 0.27 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.34):** El mapa de activación se muestra completamente deslocalizado de las regiones prioritarias de diagnóstico (vértices y márgenes laterales axilares). Grad-CAM concentra erróneamente la intensidad sobre el hilio central y el área abdominal inferior. El modelo sobrepasa el umbral mínimo por factores correlativos secundarios (p. ej., desviaciones volumétricas mediastínicas), evidenciando una incapacidad crítica para aislar la línea pleural fina. Esta carencia visual explica el modesto AUC (0.62) del sistema.
* **Muestra Derecha (Predicción: 0.07):** La atención se dispersa externamente hacia las regiones axilares y las estructuras blandas de la pared torácica, manteniendo los pulmones en un estado neutral y logrando la clasificación negativa por ausencia de estímulos detectables.

---

<font color='darkblue'>3.4. Support Devices (Dispositivos de Soporte)</font>

> **Criterio Fisiopatológico y Diagnóstico:** Esta etiqueta agrupa elementos instrumentales de carácter exógeno y de naturaleza altamente densa (metales o polímeros radiopacos), tales como catéteres venosos centrales, marcapasos, tubos endotraqueales, sondas gástricas y electrodos de superficie. Su reconocimiento requiere que el modelo desvíe la atención del parénquima profundo y examine los canales anatómicos habituales de acceso intervencionista.

| Métrica | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Muestra Izquierda** | Positivo | 0.65 | 0.58 | Verdadero Positivo |
| **Muestra Derecha** | Sano | 0.26 | 0.58 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Muestra Izquierda (Predicción: 0.65):** La activación se focaliza con estricta precisión topográfica sobre la región infraclavicular izquierda del paciente (esquina superior derecha de la imagen), correspondiente exactamente al punto de inserción y anclaje habitual de catéteres venosos centrales y generadores de marcapasos. La red ignora la anatomía pulmonar y aísla el cuerpo extraño de forma correcta.
* **Muestra Derecha (Predicción: 0.26):** Al no constatar la intrusión de cuerpos extraños de alta densidad radiológica en las vías de entrada anatómicas, la activación se mantiene en rangos difusos basales y no consigue franquear la barrera predictiva de Youden, consolidando el descarte.

---

<font color='darkblue'>4. Clases con Limitaciones Estadísticas de la Muestra (Dataset Shift)</font>

<font color='darkblue'>4.1. Lung Lesion (Lesión Pulmonar) & Fracture (Fractura)</font>

> **Nota Metodológica de Auditoría:** Las clases *Lung Lesion* y *Fracture* adolecen de una prevalencia nula de casos positivos reales ($True\ Positives = 0$) dentro de la cohorte estricta del Test Set utilizado. Esta restricción estadística impide evaluar la capacidad del modelo para localizar de forma activa nódulos, masas tumorales o trazos de fractura costoclavicular mediante mapas de calor verdaderos positivos.

| Patología | Condición Real | Probabilidad Predicha | Umbral Operativo (Youden) | Clasificación Operativa |
| :--- | :---: | :---: | :---: | :---: |
| **Lung Lesion** | Sano | 0.12 | 0.16 | Verdadero Negativo |
| **Fracture** | Sano | 0.13 | 0.16 | Verdadero Negativo |

<font color='darkblue'>Evaluación del Comportamiento del Modelo</font>
* **Lung Lesion (Predicción: 0.12):** Grad-CAM ignora el tejido pulmonar propiamente dicho, proyectando una activación errática y dispersa sobre los márgenes laterales de la pared costal externa y el espacio subdiafragmático. La red ejecuta el descarte por el principio de "falta de características", reflejando que, ante la escasez de patrones consolidados en su entrenamiento, opta por derivar la atención al ruido periférico de la imagen.
* **Fracture (Predicción: 0.13):** La activación evade los arcos costales (región anatómica diana) y se concentra de forma lineal sobre el eje de la columna vertebral dorsal y el mediastino posterior. El modelo parece haber asimilado una heurística simplificada del concepto "estructura ósea" vinculada a la densidad de la columna central. Al no observar discontinuidades graves en este pilar macizo, sitúa la probabilidad por debajo del umbral operativo, derivando en un Verdadero Negativo con un fundamento anatómico incompleto.